In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import os

# Point to your VSC_DATA library folder
lib_dir = os.path.join(os.environ.get('VSC_DATA'), 'thesis_libs')

# Force install NumPy 1.x to fix the compatibility crash
!pip install --no-cache-dir "numpy<2" --target={lib_dir} --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 305.3 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
import os

# 1. Point Python to your VSC_DATA library folder
lib_dir = os.path.join(os.environ.get('VSC_DATA'), 'thesis_libs')
if lib_dir not in sys.path:
    sys.path.insert(0, lib_dir)

# 2. Run your specific imports
import cv2
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
from scipy.ndimage import distance_transform_edt

from skimage.morphology import medial_axis, remove_small_objects, binary_closing, disk
from skimage.measure import label, regionprops

import glob
import pandas as pd
from tqdm import tqdm

print("✓ All morphology and processing dependencies loaded successfully!")

✓ All morphology and processing dependencies loaded successfully!


In [4]:
import os

# --- 1. DATASET FOLDERS ---
# Using the exact absolute path from the VSC storage
DRIVE_ROOT = "/vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis/predicted_masks"

# os.path.join safely builds the paths
CATHETER_FOLDER = os.path.join(DRIVE_ROOT, "catheter", "train")
ATRIUM_FOLDER   = os.path.join(DRIVE_ROOT, "atrium", "train")

# where to save outputs
OUTPUT_FOLDER = DRIVE_ROOT
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

CSV_OUTPUT_PATH = os.path.join(OUTPUT_FOLDER, "catheter_tip_features.csv")


# --- 2. PARAMETERS ---
TIP_LENGTH_PX = 200.0
TIP_AVG_WIN = 25
SMOOTH_LINE = True
SMOOTH_WIN = 31
SMOOTH_POLY = 3
TIP_THICKNESS_THRESHOLD = 28.0
SPUR_MAX_LEN = 60
SPUR_DT_RATIO = 0.70

# overlap fallback trigger
ENDPOINT_TO_ATRIUM_MAX_DIST = 60.0
JUNCTION_SEARCH_RADIUS = 5

# Tunable constant
OVERLAP_OVERRIDE_THRESHOLD = 15   # pixels

In [5]:
# LOADERS

def load_grayscale_any(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Cannot load: {path}")
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = img.astype(np.float32)
    m, M = float(img.min()), float(img.max())
    if M > m:
        img = (img - m) / (M - m) * 255.0
    return img.astype(np.uint8)


def load_binary_mask(path: str, reference_shape=None) -> np.ndarray:
    """
    Load binary mask and optionally resize to match reference shape.
    """

    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    if img is None:
        raise FileNotFoundError(f"Cannot load: {path}")

    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # convert to binary
    mask = ((img > 0).astype(np.uint8) * 255)

    # resize if reference shape provided
    if reference_shape is not None:

        H, W = reference_shape

        mask = cv2.resize(
            mask,
            (W, H),
            interpolation=cv2.INTER_NEAREST
        )

    return mask

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# BINARIZATION  (v2 — boundary smoothing before skeletonization)
# ─────────────────────────────────────────────────────────────────────────────
# Fix: Added morphological opening with disk(2) after thresholding and before
# binary closing. Opening removes small boundary protrusions and thin bridges
# that create spurious junctions during skeletonization. This reduces false
# loop detection and prevents the loop-recovery pipeline from cutting near
# real endpoints.
#
# Opening order: threshold → opening → closing → largest component
#   - opening first: removes noise without expanding the mask
#   - closing after: fills small holes while keeping the smoothed boundary

import cv2
import numpy as np
from skimage.morphology import binary_closing, opening, disk, remove_small_objects
from skimage.measure import label, regionprops


def largest_component(bin_bool: np.ndarray) -> np.ndarray:
    if bin_bool.sum() == 0:
        return bin_bool
    lab = label(bin_bool, connectivity=2)
    props = regionprops(lab)
    if not props:
        return bin_bool
    biggest = max(props, key=lambda r: r.area).label
    return lab == biggest


def binarize_catheter(gray8: np.ndarray) -> np.ndarray:
    blur = cv2.GaussianBlur(gray8, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    if np.sum(th == 255) > np.sum(th == 0):
        th = 255 - th

    bin_bool = th.astype(bool)

    # NEW v2: Smooth boundary to remove small protrusions that create
    # spurious junctions during skeletonization. disk(2) removes noise
    # up to ~2 pixels wide without significantly altering catheter shape.
    bin_bool = opening(bin_bool, disk(2))

    bin_bool = binary_closing(bin_bool, disk(4))
    bin_bool = remove_small_objects(bin_bool, 50)
    bin_bool = largest_component(bin_bool)

    return (bin_bool.astype(np.uint8) * 255)

In [7]:
# ── Mask quality feature ──────────────────────────────────────────────
from skimage.measure import regionprops, label as sk_label

def compute_mask_solidity(bin_mask):
    """
    Solidity of the largest connected component in the catheter mask.
    1.0 = perfectly convex (clean mask).
    < 0.85 = fragmented or noisy predicted mask.
    Used to give the classifier context about segmentation quality.
    """
    lab = sk_label(bin_mask.astype(bool))
    props = regionprops(lab)
    if not props:
        return 0.0
    largest = max(props, key=lambda r: r.area)
    return float(largest.solidity)

In [8]:
# SKELETON HELPERS
# Converts the catheter mask into a single-pixel-wide centerline (skeleton)
# and analyses its structure. Each skeleton pixel is classified by how many
# neighbours it has: 1 neighbour = endpoint (tip or connector end),
# 2 neighbours = normal midpoint, 3+ neighbours = junction (overlap/crossing).
# A clean catheter should have exactly 2 endpoints and zero junctions.
# If junctions are found, the catheter has overlapped itself and the
# overlap recovery pipeline is triggered before tip selection can proceed.

def build_skeleton_and_dist(mask_bool):
    skel_bool, dist = medial_axis(mask_bool, return_distance=True)
    return skel_bool, dist


def get_neighbors_from_pts(point, pts_set):
    y, x = point
    neighbors = []
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            if dy == 0 and dx == 0:
                continue
            q = (y + dy, x + dx)
            if q in pts_set:
                neighbors.append(q)
    return neighbors


def detect_all_endpoints(skel_bool):
    ys, xs = np.nonzero(skel_bool)
    pts = set(zip(ys.tolist(), xs.tolist()))
    endpoints = []

    for p in pts:
        neighbors = get_neighbors_from_pts(p, pts)
        if len(neighbors) == 1:
            endpoints.append((int(p[0]), int(p[1])))

    return endpoints, pts


def detect_junction_points(skel_bool):
    ys, xs = np.nonzero(skel_bool)
    pts = set(zip(ys.tolist(), xs.tolist()))
    junctions = []

    for p in pts:
        neighbors = get_neighbors_from_pts(p, pts)
        if len(neighbors) >= 3:
            junctions.append((int(p[0]), int(p[1])))

    return junctions, pts

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# LOOP DETECTION v4 — RATIO-BASED THRESHOLD
# ─────────────────────────────────────────────────────────────────────────────
# Issue with v3: absolute threshold MIN_CYCLE_PIXELS = 100 was too low
# for this dataset. Large proximal cuffs were producing graph cycles
# of 100-300 pixels, just over threshold, which mislabelled clean
# curves with prominent cuffs (IMG-0023, IMG-0043, IMG-0069, IMG-0094)
# as success_overlap_mode.
#
# Fix: switch from absolute cycle pixel count to RATIO of cycle pixels
# to total skeleton pixels.
#
# Empirically observed in this dataset:
#   - Real loops:    ratio = 0.70 to 0.95   (most of the catheter
#                                            participates in the loop)
#   - Cuff artifacts: ratio = 0.01 to 0.10  (cuff is small relative to
#                                            full catheter length)
#   - Clean curves:   ratio = 0.00          (no cycles at all)
#
# Threshold of 0.30 sits comfortably between the two regimes — well
# above any plausible cuff artifact, well below any real loop.

import numpy as np
from scipy import ndimage as ndi
from skimage.morphology import label as cc_label


# Minimum fraction of skeleton pixels that must participate in cycles
# for the case to count as a real catheter loop.
MIN_CYCLE_RATIO    = 0.10   # loop must be ≥15% of skeleton
MIN_CYCLE_PIXELS   = 80     # loop must have ≥80 pixels (kills tiny cuff rings)


def _skeleton_degree(skel_bool):
    """Per-pixel skeleton degree (8-neighbour count, excluding self)."""
    kernel = np.ones((3, 3), dtype=np.uint8)
    nb = ndi.convolve(
        skel_bool.astype(np.uint8), kernel, mode="constant", cval=0
    )
    return (nb - 1) * skel_bool.astype(np.uint8)


def count_graph_cycles(skel_bool):
    """Total cycles in skeleton graph via Euler's formula."""
    if not skel_bool.any():
        return 0
    deg = _skeleton_degree(skel_bool)
    V = int(skel_bool.sum())
    E = int(deg.sum() // 2)
    components = cc_label(skel_bool, connectivity=2)
    C = int(components.max())
    return E - V + C


def cycle_pixel_count(skel_bool):
    """Number of skeleton pixels participating in any cycle (the 2-core)."""
    if not skel_bool.any():
        return 0

    # Crop to bounding box for speed
    rows = np.any(skel_bool, axis=1)
    cols = np.any(skel_bool, axis=0)
    rmin, rmax = np.argmax(rows), len(rows) - 1 - np.argmax(rows[::-1])
    cmin, cmax = np.argmax(cols), len(cols) - 1 - np.argmax(cols[::-1])
    rmin_p, rmax_p = max(rmin - 1, 0), min(rmax + 2, skel_bool.shape[0])
    cmin_p, cmax_p = max(cmin - 1, 0), min(cmax + 2, skel_bool.shape[1])

    current = skel_bool[rmin_p:rmax_p, cmin_p:cmax_p].copy()
    while True:
        deg = _skeleton_degree(current)
        leaves = current & (deg <= 1)
        if not leaves.any():
            break
        current = current & ~leaves
    return int(current.sum())


def has_real_loop(skel_bool, min_ratio=MIN_CYCLE_RATIO):
    if count_graph_cycles(skel_bool) < 1:
        return False
    total = int(skel_bool.sum())
    if total == 0:
        return False
    cyc = cycle_pixel_count(skel_bool)
    # BOTH conditions must hold: large enough ratio AND large enough absolute size
    return (cyc / total) >= min_ratio and cyc >= MIN_CYCLE_PIXELS


# ─────────────────────────────────────────────────────────────────────────────
# Diagnostic — now reports the cycle-to-skeleton ratio
# ─────────────────────────────────────────────────────────────────────────────

def inspect_skeleton_cycles(gray8, image_name=""):
    bin_img   = binarize_catheter(gray8)
    mask_bool = bin_img.astype(bool)
    skel_bool, dist = build_skeleton_and_dist(mask_bool)
    skel_bool = prune_short_spurs(
        skel_bool, dist, max_len=SPUR_MAX_LEN, dt_ratio=SPUR_DT_RATIO
    )

    endpoints, _ = detect_all_endpoints(skel_bool)
    junctions, _ = detect_junction_points(skel_bool)
    n_cycles = count_graph_cycles(skel_bool)
    total_px = int(skel_bool.sum())
    cycle_px = cycle_pixel_count(skel_bool) if n_cycles >= 1 else 0
    ratio    = (cycle_px / total_px) if total_px > 0 else 0.0

    print(f"\n=== {image_name or 'image'} ===")
    print(f"  Skeleton pixels:     {total_px}")
    print(f"  Endpoints:           {len(endpoints)}")
    print(f"  Junctions:           {len(junctions)}")
    print(f"  Graph cycles (Euler):{n_cycles}")
    print(f"  Pixels in cycles:    {cycle_px}")
    print(f"  Cycle/skeleton ratio:{ratio:.3f}  "
          f"(threshold = {MIN_CYCLE_RATIO})")
    if n_cycles >= 1 and ratio >= MIN_CYCLE_RATIO:
        print(f"  → REAL LOOP")
    elif n_cycles >= 1:
        print(f"  → no loop (cycle is only {ratio*100:.1f}% of skeleton "
              f"— likely a cuff artifact)")
    else:
        print(f"  → no loop (no graph cycles)")

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# LOOP RECOVERY v2 — 3-strategy hierarchy
# Insert this cell BEFORE compute_centerline_fixed (Cell 13)
# ─────────────────────────────────────────────────────────────────────────────

def _skeleton_degree_v2(skel_bool):
    kernel = np.ones((3, 3), dtype=np.uint8)
    nb = ndi.convolve(skel_bool.astype(np.uint8), kernel, mode="constant", cval=0)
    return (nb - 1) * skel_bool.astype(np.uint8)


def _compute_2core(skel_bool):
    current = skel_bool.copy()
    while True:
        deg = _skeleton_degree_v2(current)
        leaves = current & (deg <= 1)
        if not leaves.any():
            break
        current = current & ~leaves
    return current


def _2core_boundary_pixels(skel_bool):
    core = _compute_2core(skel_bool)
    if not core.any():
        return []
    ys, xs = np.nonzero(core)
    pts_set = set(zip(ys.tolist(), xs.tolist()))
    skel_pts = set(zip(*np.nonzero(skel_bool)))
    boundary = []
    for p in pts_set:
        y, x = p
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                if dy == 0 and dx == 0:
                    continue
                nb = (y + dy, x + dx)
                if nb in skel_pts and nb not in pts_set:
                    boundary.append(p)
                    break
    return boundary


def _erase_disk(skel, centre, radius=4):
    sk = skel.copy()
    H, W = sk.shape
    cy, cx = centre
    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            if dy * dy + dx * dx <= radius * radius:
                ny, nx = cy + dy, cx + dx
                if 0 <= ny < H and 0 <= nx < W:
                    sk[ny, nx] = False
    return sk


def _local_curvature_along_path(path, half_win=8):
    n = len(path)
    curv = np.full(n, np.nan)
    pts = np.array([(float(y), float(x)) for y, x in path])
    for i in range(half_win, n - half_win):
        v1 = pts[i] - pts[i - half_win]
        v2 = pts[i + half_win] - pts[i]
        l1 = np.linalg.norm(v1)
        l2 = np.linalg.norm(v2)
        if l1 < 1e-6 or l2 < 1e-6:
            continue
        cos_a = np.clip(np.dot(v1, v2) / (l1 * l2), -1.0, 1.0)
        curv[i] = np.arccos(cos_a)
    return curv


def _trace_longest_path(skel_bool):
    from collections import deque
    ys, xs = np.nonzero(skel_bool)
    if len(ys) == 0:
        return []
    pts_set = set(zip(ys.tolist(), xs.tolist()))

    def bfs_farthest(start):
        visited = {start: None}
        q = deque([start])
        last = start
        while q:
            node = q.popleft()
            last = node
            for dy in (-1, 0, 1):
                for dx in (-1, 0, 1):
                    if dy == 0 and dx == 0:
                        continue
                    nb = (node[0] + dy, node[1] + dx)
                    if nb in pts_set and nb not in visited:
                        visited[nb] = node
                        q.append(nb)
        path = []
        cur = last
        while cur is not None:
            path.append(cur)
            cur = visited[cur]
        return last, path[::-1]

    start = (int(ys[0]), int(xs[0]))
    far1, _ = bfs_farthest(start)
    _, path = bfs_farthest(far1)
    return path

def _strategy1_junction_cut(skel_bool, dist, original_endpoints):
    # FIX: Must call detect_junction_points first to get 'junctions'
    junctions, pts_set = detect_junction_points(skel_bool)

    if not junctions:
        return None, None, None

    if original_endpoints:
        thick_end = max(original_endpoints,
                        key=lambda ep: float(dist[int(ep[0]), int(ep[1])]))
    else:
        ys, xs = np.nonzero(skel_bool)
        idx = int(np.argmax(dist[ys, xs]))
        thick_end = (int(ys[idx]), int(xs[idx]))

    ty, tx = thick_end

    # NEW: Filter out junctions that are too close to original endpoints
    PROTECT_RADIUS = 20  # pixels - don't cut near real endpoints
    safe_junctions = []
    for j in junctions:
        jy, jx = j
        # Check distance to ALL original endpoints
        min_dist_to_endpoint = min(
            np.hypot(jy - ep[0], jx - ep[1]) for ep in original_endpoints
        ) if original_endpoints else float('inf')

        if min_dist_to_endpoint > PROTECT_RADIUS:
            safe_junctions.append(j)

    # If no safe junctions, fall through to next strategy
    if not safe_junctions:
        print(f"[strategy1] All junctions too close to endpoints, skipping")
        return None, None, None

    # Use only safe junctions for selection
    junction = min(safe_junctions,
                   key=lambda j: (j[0]-ty)**2 + (j[1]-tx)**2)

    print(f"[strategy1] Protected {len(original_endpoints)} endpoints, "
          f"using safe junction at {junction}")

    sk_cut = _erase_disk(skel_bool, junction, radius=4)
    new_eps, new_pts_set = detect_all_endpoints(sk_cut)
    if len(new_eps) >= 2:
        new_eps = _filter_stub_endpoints(list(new_eps), sk_cut)
        if len(new_eps) >= 2:
            return sk_cut, list(new_eps), new_pts_set
    return None, None, None


def _strategy2_2core_boundary_cut(skel_bool, dist, original_endpoints):
    boundary = _2core_boundary_pixels(skel_bool)
    if not boundary:
        return None, None, None

    cut_pt = max(boundary, key=lambda p: float(dist[p[0], p[1]]))

    for radius in (4, 7):
        sk_cut = _erase_disk(skel_bool, cut_pt, radius=radius)
        new_eps, new_pts_set = detect_all_endpoints(sk_cut)
        if len(new_eps) >= 2:
            new_eps = _filter_stub_endpoints(list(new_eps), sk_cut)  # ← new line
        if len(new_eps) >= 2:
            return sk_cut, list(new_eps), new_pts_set

    return None, None, None


def _strategy3_curvature_cut(skel_bool, dist, original_endpoints):
    path = _trace_longest_path(skel_bool)
    if len(path) < 30:
        return None, None, None

    curv = _local_curvature_along_path(path, half_win=8)
    margin = max(10, len(path) // 5)
    curv[:margin] = np.nan
    curv[-margin:] = np.nan

    pts = np.array([(float(y), float(x)) for y, x in path])
    for ep in (original_endpoints or []):
        ey, ex = float(ep[0]), float(ep[1])
        dists_to_ep = np.sqrt(np.sum((pts - np.array([ey, ex])) ** 2, axis=1))
        curv[dists_to_ep < 30] = np.nan

    if np.all(np.isnan(curv)):
        return None, None, None

    cut_idx = int(np.nanargmax(curv))
    cut_pt = (int(round(path[cut_idx][0])), int(round(path[cut_idx][1])))

    sk_cut = _erase_disk(skel_bool, cut_pt, radius=4)
    new_eps, new_pts_set = detect_all_endpoints(sk_cut)

    if len(new_eps) >= 2:
        new_eps = _filter_stub_endpoints(list(new_eps), sk_cut)  # ← new line

    if len(new_eps) >= 2:
        return sk_cut, list(new_eps), new_pts_set
    return None, None, None


def recover_loop_endpoints(skel_bool, dist, original_endpoints,
                           detect_all_endpoints_fn=None,
                           detect_junction_points_fn=None,
                           get_neighbors_from_pts_fn=None):
    """
    Returns: (skel_bool, endpoints, pts_set, confident)
    confident = False if geometric fallback was used
    """
    strategies = [
        ("junction_cut",       lambda: _strategy1_junction_cut(skel_bool, dist, original_endpoints)),
        ("2core_boundary_cut", lambda: _strategy2_2core_boundary_cut(skel_bool, dist, original_endpoints)),
        ("curvature_cut",      lambda: _strategy3_curvature_cut(skel_bool, dist, original_endpoints)),
    ]

    for name, strategy_fn in strategies:
        sk_cut, new_eps, new_pts_set = strategy_fn()
        if sk_cut is not None and len(new_eps) >= 2:
            print(f"[loop_recovery_v3] Strategy '{name}' succeeded → {len(new_eps)} endpoints")
            merged = list(new_eps)
            for ep in original_endpoints:
                if not any(abs(int(ep[0])-int(m[0])) <= 2 and
                           abs(int(ep[1])-int(m[1])) <= 2
                           for m in merged):
                    merged.append(ep)
            return sk_cut, merged, new_pts_set, True  # Confident!

    # All strategies failed — geometric fallback
    print("[loop_recovery_v3] All strategies failed — using geometric fallback")
    coords = np.argwhere(skel_bool)
    if len(coords) >= 2:
        # v3: Use distance-transform-aware fallback
        radii = dist[skel_bool]
        min_radius_idx = int(np.argmin(radii))
        thinnest = tuple(int(v) for v in coords[min_radius_idx])

        # Find farthest point from thinnest along skeleton path
        from collections import deque
        pts_set = set(map(tuple, coords))
        visited = {thinnest: 0}
        queue = deque([thinnest])
        farthest = thinnest
        max_dist = 0

        while queue:
            current = queue.popleft()
            for dy, dx in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
                nb = (current[0]+dy, current[1]+dx)
                if nb in pts_set and nb not in visited:
                    visited[nb] = visited[current] + np.hypot(dy, dx)
                    queue.append(nb)
                    if visited[nb] > max_dist:
                        max_dist, farthest = visited[nb], nb

        fallback_eps = [thinnest, farthest]
    else:
        fallback_eps = list(original_endpoints)

    _, fallback_pts_set = detect_all_endpoints_fn(skel_bool) if detect_all_endpoints_fn else detect_all_endpoints(skel_bool)

    return skel_bool, fallback_eps, fallback_pts_set, False  # NOT confident

In [11]:
MIN_ARM_LENGTH_PX = 80

def _get_8neighbors(point, pts_set):
    y, x = point
    return [(y + dy, x + dx)
            for dy in (-1, 0, 1) for dx in (-1, 0, 1)
            if (dy != 0 or dx != 0) and (y + dy, x + dx) in pts_set]


def _arm_length_from_endpoint(endpoint, skel_bool):
    ys, xs = np.nonzero(skel_bool)
    pts_set = set(zip(ys.tolist(), xs.tolist()))
    current = endpoint
    prev = None
    length = 0.0
    for _ in range(2000):
        neighbors = [n for n in _get_8neighbors(current, pts_set) if n != prev]
        if len(neighbors) == 0:
            break
        if len(neighbors) > 1:
            break
        nxt = neighbors[0]
        length += np.hypot(nxt[0] - current[0], nxt[1] - current[1])
        prev = current
        current = nxt
    return length


def _dedup_endpoints(endpoints, tolerance=10):
    """
    Remove near-duplicate endpoints within `tolerance` pixels of each other.
    When two endpoints are too close, keep the one with the longer arm_len
    (already stored as the second element of the (ep, arm_len) tuples).
    Call this on the measured list before filtering.
    """
    # endpoints here is list of (coord, arm_len)
    kept = []
    for ep, arm_len in endpoints:
        too_close = False
        for i, (kep, karm) in enumerate(kept):
            dist = np.hypot(ep[0] - kep[0], ep[1] - kep[1])
            if dist <= tolerance:
                # keep the one with the longer arm
                if arm_len > karm:
                    kept[i] = (ep, arm_len)
                too_close = True
                break
        if not too_close:
            kept.append((ep, arm_len))
    return kept


def _filter_stub_endpoints(endpoints, skel_bool, min_arm_px=MIN_ARM_LENGTH_PX):
    measured = []
    for ep in endpoints:
        arm_len = _arm_length_from_endpoint(ep, skel_bool)
        measured.append((ep, arm_len))
        print(f"  [stub_filter] ep={ep}  arm_len={arm_len:.1f}px")

    # Step 1 — deduplicate
    measured = _dedup_endpoints(measured, tolerance=10)
    print(f"  [stub_filter] after dedup: {len(measured)} endpoints")

    # Step 2 — keep arms above threshold
    kept = [(ep, arm_len) for ep, arm_len in measured if arm_len >= min_arm_px]

    # Step 3 — fallback: if only 1 real arm survived, find the skeleton
    # opposite end geometrically instead of picking another stub
    if len(kept) == 0:
        measured_sorted = sorted(measured, key=lambda x: x[1], reverse=True)
        kept = measured_sorted[:1]
        print(f"  [stub_filter] 0 passed — keeping only the longest arm")

        if len(kept) == 1:
            # NEW: BFS along skeleton to find farthest point
            real_ep = kept[0][0]
            pts_set = set(zip(*np.nonzero(skel_bool)))

            from collections import deque
            visited = {real_ep: 0}
            queue = deque([real_ep])
            farthest, max_dist = real_ep, 0

            while queue:
                current = queue.popleft()
                for dy, dx in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
                    nb = (current[0]+dy, current[1]+dx)
                    if nb in pts_set and nb not in visited:
                        visited[nb] = visited[current] + np.hypot(dy, dx)
                        queue.append(nb)
                        if visited[nb] > max_dist:
                            max_dist, farthest = visited[nb], nb

            kept.append((farthest, float(max_dist)))
            print(f"  [stub_filter] only 1 real arm — "
                  f"added geometric opposite {farthest}")

    result = [ep for ep, _ in kept]
    print(f"  [stub_filter] final: {len(result)} endpoints")
    return result

In [12]:
# ── SPUR PRUNING / OVERLAP SUPPORT ───────────────────────────────────────────
# Cleans the skeleton and handles problematic catheter shapes.
# prune_short_spurs() removes small noise branches (spurs) that appear on the
# skeleton after skeletonisation — a branch is removed if it is shorter than
# 40px AND its endpoint is noticeably thinner than the rest of the branch.
# find_extreme_points() locates the topmost, bottommost, and narrowest points
# on the skeleton — the narrowest point is where two overlapping catheter
# segments are closest together and is used as the cut location.
# break_skeleton_near_narrowest() surgically erases a small circular patch of
# skeleton pixels at the narrowest point to break a loop into an open curve,
# creating two new endpoints so normal processing can resume.
# handle_insufficient_endpoints() is the full recovery pipeline called when
# the skeleton has fewer than 2 endpoints — it breaks the loop, re-detects
# endpoints, and falls back to geometric top/bottom points if needed.
# get_endpoint_thickness() measures average catheter diameter over 20 pixels
# inward from an endpoint — the thinner end is the free catheter tip.

def prune_short_spurs(skel_bool, dist, max_len=40, dt_ratio=0.65):
    sk = skel_bool.copy()

    changed = True
    while changed:
        changed = False
        endpoints, pts_set = detect_all_endpoints(sk)

        for ep in endpoints:
            path = [ep]
            current = ep
            prev = None

            while True:
                neighbors = get_neighbors_from_pts(current, pts_set)
                if prev is not None:
                    neighbors = [n for n in neighbors if n != prev]

                if len(neighbors) == 0:
                    break
                if len(neighbors) > 1:
                    break

                nxt = neighbors[0]
                path.append(nxt)
                prev = current
                current = nxt

                deg = len(get_neighbors_from_pts(current, pts_set))
                if deg != 2:
                    break

            if len(path) <= max_len:
                path_radii = [float(dist[y, x]) for (y, x) in path]
                endpoint_r = path_radii[0]
                mean_r = np.mean(path_radii)

                if endpoint_r <= dt_ratio * max(mean_r, 1e-6) or len(path) <= max_len // 2:
                    for y, x in path[:-1]:
                        sk[y, x] = False
                    changed = True
                    break

    return sk


def find_extreme_points(skel_bool, dist):
    coords = np.argwhere(skel_bool)
    if len(coords) == 0:
        return None, None, None

    lowest = tuple(int(v) for v in max(coords, key=lambda x: x[0]))
    highest = tuple(int(v) for v in min(coords, key=lambda x: x[0]))

    radii = dist[skel_bool]
    min_radius_idx = int(np.argmin(radii))
    narrowest = tuple(int(v) for v in coords[min_radius_idx])

    return lowest, highest, narrowest


def break_skeleton_near_narrowest(skel_bool, dist, radius=3):
    _, _, narrowest = find_extreme_points(skel_bool, dist)

    if narrowest is None:
        return skel_bool.copy(), []

    sk = skel_bool.copy()
    H, W = sk.shape

    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            ny, nx = narrowest[0] + dy, narrowest[1] + dx
            if 0 <= ny < H and 0 <= nx < W and dy * dy + dx * dx <= radius * radius:
                sk[ny, nx] = False

    return sk, [narrowest]


def handle_insufficient_endpoints_keep_existing_tip(skel_bool, dist, current_endpoints):
    broken_skel, candidate_points = break_skeleton_near_narrowest(skel_bool, dist)
    new_endpoints, pts_set = detect_all_endpoints(broken_skel)

    merged = []
    for p in current_endpoints + new_endpoints:
        if p not in merged:
            merged.append(p)

    if len(merged) < 2:
        for p in candidate_points:
            if p is not None and p not in merged:
                merged.append(tuple(int(v) for v in p))

    if len(merged) < 2:
        coords = np.argwhere(broken_skel)
        if len(coords) >= 2:
            lowest = tuple(int(v) for v in max(coords, key=lambda x: x[0]))
            highest = tuple(int(v) for v in min(coords, key=lambda x: x[0]))
            merged = [lowest, highest]

    return broken_skel, merged, pts_set


def get_endpoint_thickness(endpoint, dist, pts_set, window=20):
    y, x = endpoint
    thicknesses = [2.0 * float(dist[y, x])]

    current = (y, x)
    prev = None

    for _ in range(window):
        neighbors = get_neighbors_from_pts(current, pts_set)
        if prev is not None:
            neighbors = [n for n in neighbors if n != prev]

        if len(neighbors) == 0:
            break
        if len(neighbors) > 1:
            break

        prev = current
        current = neighbors[0]
        thicknesses.append(2.0 * float(dist[current[0], current[1]]))

    return float(np.mean(thicknesses)) if thicknesses else np.inf

In [13]:
# GRAPH / PATH
# Traces the full centerline path from the selected tip to the opposite end.
# build_graph_from_skeleton() converts the skeleton into a weighted graph where
# each skeleton pixel is a node and each connection to a neighbour is an edge.
# Edge weights are designed so that paths through THICKER parts of the catheter
# are cheaper to travel — this biases the path toward the true centerline rather
# than cutting across thin regions or noise.
# smooth_path() applies a Savitzky-Golay filter to the raw pixel-by-pixel path
# to produce a smooth curve — removing the staircase jaggedness that comes from
# following a pixel grid.
# build_path_from_tip() runs Dijkstra's shortest path algorithm from the
# selected tip endpoint through the weighted graph to find the optimal route
# to the farthest reachable point (the opposite end of the catheter). The result
# is an ordered list of (y, x) coordinates representing the full centerline
# from tip to connector, which is then smoothed and used for all downstream
# feature extraction and tip segment cutting.

def build_graph_from_skeleton(skel_bool, dist, eps=1e-3):
    ys, xs = np.nonzero(skel_bool)
    coords = np.stack([ys, xs], axis=1)
    n = len(coords)
    if n == 0:
        return None, None, None

    idx_map = -np.ones(skel_bool.shape, dtype=np.int32)
    idx_map[ys, xs] = np.arange(n, dtype=np.int32)

    rows, cols, weights = [], [], []

    for k in range(n):
        y, x = coords[k]
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                if dy == 0 and dx == 0:
                    continue
                ny, nx = y + dy, x + dx
                if 0 <= ny < skel_bool.shape[0] and 0 <= nx < skel_bool.shape[1] and skel_bool[ny, nx]:
                    j = idx_map[ny, nx]
                    if j < 0 or j == k:
                        continue
                    step = np.hypot(dy, dx)
                    w = step / ((float(dist[y, x]) + float(dist[ny, nx])) / 2.0 + eps)
                    rows.append(k)
                    cols.append(j)
                    weights.append(w)

    G = csr_matrix((weights, (rows, cols)), shape=(n, n))
    return G, coords, idx_map


def smooth_path(points, win=31, poly=3):
    if not points or len(points) < 5:
        return points

    win = min(win, len(points))
    if win % 2 == 0:
        win -= 1
    if win < 5:
        return points

    poly = min(poly, win - 2)
    if poly < 1:
        return points

    ys = np.array([p[0] for p in points], float)
    xs = np.array([p[1] for p in points], float)

    ys_s = savgol_filter(ys, win, poly)
    xs_s = savgol_filter(xs, win, poly)

    return list(zip(ys_s, xs_s))


def build_path_from_tip(skel_bool, dist, tip_coord):
    G, coords, idx_map = build_graph_from_skeleton(skel_bool, dist)
    if G is None:
        return None

    tip_idx = idx_map[tip_coord]
    if tip_idx < 0:
        return None

    dist_arr, predecessors = dijkstra(G, directed=False, indices=tip_idx, return_predecessors=True)
    valid_dist = np.where(np.isinf(dist_arr), -1, dist_arr)
    farthest_idx = int(np.argmax(valid_dist))

    if valid_dist[farthest_idx] < 0:
        return None

    idx_path = []
    current = farthest_idx
    while current != tip_idx:
        idx_path.append(current)
        if predecessors[current] == -9999:
            return None
        current = predecessors[current]
    idx_path.append(tip_idx)
    idx_path.reverse()

    full_path = [(float(coords[i][0]), float(coords[i][1])) for i in idx_path]

    if SMOOTH_LINE:
        full_path = smooth_path(full_path, win=SMOOTH_WIN, poly=SMOOTH_POLY)

    return full_path

In [14]:
# PATH PROCESSING
# Handles precise measurement and cutting of the centerline path.
# cumulative_arclength() measures the true pixel distance along the path from
# the tip to every point — unlike straight-line distance, this follows the
# curve of the catheter so the 200px cut is always 200px of actual catheter
# length regardless of how curved or straight the catheter is.
# extract_tip_segment() cuts exactly 200px of arc-length from the tip end of
# the full centerline path. It uses sub-pixel interpolation at the cut point
# so the segment is always precisely 200px — not 198px or 203px depending on
# where pixels happen to fall. If extracting from the far end instead of the
# tip, it reverses the path, extracts, then reverses back so the result always
# runs in the same direction as the original path.
# mean_radius_on_path() measures the average catheter radius over the first
# or last 30 points of a path using the distance transform values — multiplying
# by 2 gives diameter. This is used as a sanity check: if the selected tip end
# is actually thicker than the proximal end, the tip selection was likely wrong
# and the status is flagged as ambiguous_tip.

def cumulative_arclength(path):
    if not path:
        return np.array([])
    pts = np.array([(float(y), float(x)) for (y, x) in path])
    if len(pts) < 2:
        return np.zeros(len(pts))
    d = np.sqrt(np.sum(np.diff(pts, axis=0) ** 2, axis=1))
    return np.concatenate([[0.0], np.cumsum(d)])

def extract_tip_segment(full_path, target_len, from_start=True):
    """
    Extract exactly target_len pixels of arc-length from full_path.

    Parameters
    ----------
    full_path   : list of (y, x) tuples — the full skeleton path
    target_len  : float — desired arc-length in pixels
    from_start  : bool
                  True  → extract from full_path[0]  (default)
                  False → extract from full_path[-1]
                          (reverses path, extracts, then reverses back
                           so the returned segment always runs in the
                           same direction as full_path)

    Returns
    -------
    list of (y, x) tuples representing the tip segment
    """
    if not full_path or target_len <= 0:
        return []

    # If extracting from end, reverse → extract from start → reverse back
    if not from_start:
        reversed_path    = full_path[::-1]
        reversed_segment = extract_tip_segment(
            reversed_path, target_len, from_start=True
        )
        return reversed_segment[::-1]

    # from_start=True: cut from path[0]
    s = cumulative_arclength(full_path)

    if s[-1] < target_len:
        print(f"[WARNING] Path length {s[-1]:.1f}px < target {target_len}px — returning full path")
        return full_path

    indices = np.where(s <= target_len)[0]
    if len(indices) == 0:
        return [full_path[0]]

    last_idx = indices[-1]

    # Sub-pixel interpolation at the cut point
    if last_idx < len(full_path) - 1:
        remaining = target_len - s[last_idx]
        seg_len   = s[last_idx + 1] - s[last_idx]
        if seg_len > 0:
            t  = remaining / seg_len
            y  = full_path[last_idx][0] + t * (full_path[last_idx+1][0] - full_path[last_idx][0])
            x  = full_path[last_idx][1] + t * (full_path[last_idx+1][1] - full_path[last_idx][1])
            return full_path[:last_idx+1] + [(y, x)]

    return full_path[:last_idx+1]


def mean_radius_on_path(path, dist, n=30, from_start=True):
    if not path:
        return np.nan
    pts = path[:n] if from_start else path[-n:]
    vals = []
    for y, x in pts:
        yy, xx = int(round(y)), int(round(x))
        if 0 <= yy < dist.shape[0] and 0 <= xx < dist.shape[1]:
            vals.append(float(dist[yy, xx]))
    return float(np.mean(vals)) if vals else np.nan

In [15]:
# RASTERIZATION
# Converts the 200px tip centerline (a list of coordinates) back into a filled
# pixel mask that can be compared against the atrium mask.
# The centerline is just a thin line of points — to measure overlap with the
# atrium, we need a filled tube shape that represents the actual physical width
# of the catheter tip at each point. The function densifies the path by
# interpolating extra points every 0.5px so there are no gaps, then at each
# point draws a filled circle whose radius is taken from the distance transform
# (the real local catheter radius at that location). The result is a accurate
# reconstruction of the catheter tip tube as a binary mask. A final binary
# closing fills any remaining small gaps between circles. This mask is then
# used to compute overlap features like tip_area_inside_ratio and
# iou_tip_atrium — how much of the catheter tip physically overlaps with
# the atrium region — which are the strongest features in the ML model.

def rasterize_segment_mask(shape, segment_points, dist_map, radius_scale=1.0, step=0.5):
    h, w = shape
    out = np.zeros((h, w), np.uint8)
    if not segment_points:
        return out

    pts = np.array(segment_points, dtype=np.float32)
    if len(pts) >= 2:
        dense = [pts[0]]
        for i in range(1, len(pts)):
            v = pts[i] - pts[i - 1]
            L = float(np.linalg.norm(v))
            if L <= 1e-6:
                continue
            n = int(np.ceil(L / max(step, 1e-6)))
            for k in range(1, n + 1):
                dense.append(pts[i - 1] + v * (k / n))
        pts = np.array(dense, dtype=np.float32)

    for (y, x) in pts:
        yy, xx = int(round(y)), int(round(x))
        if 0 <= yy < h and 0 <= xx < w:
            r = max(1.0, float(dist_map[yy, xx]) * radius_scale)
            cv2.circle(out, (int(xx), int(yy)), int(round(r)), 255, -1)

    out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
    return out

In [16]:
# HELPER — full-path minimum distance to atrium
def _min_dist_to_atrium_full_path(tip_path, dist_map, image_shape):
    """
    Return the minimum distance-to-atrium value over every point in tip_path.
    Using the full path (not just [:40]) is critical for curved catheters.
    """
    min_d = np.inf
    H, W = image_shape[:2]
    for py, px in tip_path:
        yy, xx = int(round(py)), int(round(px))
        if 0 <= yy < H and 0 <= xx < W:
            d = dist_map[yy, xx]
            if d < min_d:
                min_d = d
    return float(min_d)

In [17]:
# ENDPOINT / TIP SELECTION  (v9 — thinnest endpoint wins)
#
# Clinical ground truth from the dataset: the catheter tip — the end
# that enters the atrium — is ALWAYS the thinner endpoint. The other
# end is the connector/sheath, which is consistently thicker. This
# holds regardless of catheter shape (straight, curved, looped) and
# regardless of how the catheter projects onto the atrium ROI in 2D.
#
# Previous versions (v7 and earlier) tried to use atrium-overlap as
# the dominant signal, with a "single overlap wins" override. This
# failed on IMG-0108, IMG-0090, IMG-0179, IMG-0364, IMG-0161, IMG-0147
# because the THICK connector end can project directly on top of the
# atrium mask in 2D X-ray views, generating a large spurious overlap
# while the true thin tip overlaps less or not at all. The override
# therefore picked the connector.
#
# v9 RULE — pick the thinnest endpoint, full stop.
#   Across every failure case, the thinner endpoint was the correct
#   tip with a clear thickness margin (the smallest observed gap was
#   43.1 vs 43.4 in IMG-0147 — still resolvable).
#
#   The atrium mask is no longer used for tip selection. It remains
#   available downstream for feature extraction and as a defensive
#   tiebreaker only when two thicknesses are within 1.0 px of each
#   other (which the dataset does not currently produce, but is kept
#   to avoid undefined behaviour on future edge cases).

# If two endpoints' thicknesses differ by less than this many pixels,
# they are treated as a true tie and a defensive tiebreaker is used.
THICKNESS_TIE_TOL = 1.0


def select_best_tip_endpoint_basic(endpoints, pts_set, dist):
    if not endpoints:
        return None

    endpoint_data = []
    for ep in endpoints:
        thickness = get_endpoint_thickness(ep, dist, pts_set, window=TIP_AVG_WIN)
        endpoint_data.append({
            "coord": ep,
            "thickness": thickness
        })

    thin_ends = [e for e in endpoint_data if e["thickness"] < TIP_THICKNESS_THRESHOLD]
    if not thin_ends:
        thin_ends = endpoint_data

    best_tip = min(thin_ends, key=lambda x: x["thickness"])
    return best_tip["coord"]



def select_best_tip_endpoint_with_atrium_original_only(
    endpoints, pts_set, skel_bool, dist, atrium_mask, image_shape
):
    """
    Pick the catheter tip endpoint = the THINNEST endpoint.

    The atrium mask is computed and logged for diagnostics, but is not
    used in the selection itself unless two thicknesses are within
    THICKNESS_TIE_TOL of each other (genuine tie).
    """
    if not endpoints:
        return None, None

    atrium_bool = atrium_mask > 0
    dist_map = distance_transform_edt(~atrium_bool)

    candidates = []
    for ep in endpoints:
        thickness = get_endpoint_thickness(ep, dist, pts_set, window=TIP_AVG_WIN)
        full_path = build_path_from_tip(skel_bool, dist, ep)
        if full_path is None:
            continue

        tip_path = extract_tip_segment(full_path, TIP_LENGTH_PX)
        tip_mask = rasterize_segment_mask(image_shape, tip_path, dist)

        overlap_pixels = int(np.sum((tip_mask > 0) & atrium_bool))
        tip_pixels     = int(np.sum(tip_mask > 0))
        inside_ratio   = overlap_pixels / tip_pixels if tip_pixels > 0 else 0.0

        ey, ex = int(round(ep[0])), int(round(ep[1]))
        endpoint_dist = (
            float(dist_map[ey, ex])
            if 0 <= ey < image_shape[0] and 0 <= ex < image_shape[1]
            else np.inf
        )

        min_path_dist = _min_dist_to_atrium_full_path(tip_path, dist_map, image_shape)

        candidates.append({
            "coord":          ep,
            "thickness":      thickness,
            "full_path":      full_path,
            "tip_path":       tip_path,
            "overlap_pixels": overlap_pixels,
            "inside_ratio":   inside_ratio,
            "endpoint_dist":  endpoint_dist,
            "min_path_dist":  min_path_dist,
            "tip_y":          float(ep[0]),
        })

    if not candidates:
        return None, None

    print("\n[DEBUG] Candidate ranking:")
    for c in candidates:
        print(
            f"  coord={c['coord']} | thick={c['thickness']:.1f} | "
            f"min_path_dist={c['min_path_dist']:.1f} | "
            f"tip_y={c['tip_y']:.0f} | overlap={c['overlap_pixels']}"
        )

    # ── Pick the thinnest endpoint ─────────────────────────────────────
    by_thickness = sorted(candidates, key=lambda c: c["thickness"])
    thinnest = by_thickness[0]

    # Single candidate → trivial.
    if len(by_thickness) == 1:
        print(f"[DEBUG] Only one candidate: {thinnest['coord']}")
        return thinnest["coord"], thinnest

    # Defensive tiebreaker: thicknesses within tolerance → use atrium
    # proximity (closest approach to atrium wins). This branch is not
    # expected to fire on the current dataset; it exists to give a
    # deterministic, sensible answer on future edge cases.
    second = by_thickness[1]
    if abs(thinnest["thickness"] - second["thickness"]) < THICKNESS_TIE_TOL:
        tied = [c for c in candidates
                if abs(c["thickness"] - thinnest["thickness"]) < THICKNESS_TIE_TOL]
        tied.sort(key=lambda c: (c["min_path_dist"], c["endpoint_dist"], -c["tip_y"]))
        best = tied[0]
        print(f"[DEBUG] Thickness tie ({thinnest['thickness']:.1f} ≈ "
              f"{second['thickness']:.1f}) — atrium-proximity tiebreaker "
              f"picked {best['coord']}")
        return best["coord"], best

    print(f"[DEBUG] Thinnest endpoint wins: {thinnest['coord']} "
          f"(thick={thinnest['thickness']:.1f}, next={second['thickness']:.1f})")
    return thinnest["coord"], thinnest

In [18]:
# OVERLAP / JUNCTION MODE  (v9 — thinnest branch wins)
#
# Handles the special case where the catheter overlaps itself, creating a
# junction (branching point) in the skeleton instead of a simple open curve.
#
# Same clinical rule as the endpoint selector: the catheter tip is ALWAYS
# the thinner endpoint. In junction mode the unit of comparison is a
# BRANCH (a path traced outward from the junction), and "thickness" is
# the mean catheter thickness along the first segment of that branch.
#
# v9 RULE — pick the thinnest branch, full stop.
#   Atrium-overlap and min_path_distance are no longer used as primary
#   signals. The atrium mask is retained for diagnostics and as a
#   defensive tiebreaker when two branches' mean thicknesses are within
#   BRANCH_THICKNESS_TIE_TOL of each other.
#
# closest_skeleton_point_to_atrium / nearest_junction_to_point /
# trace_branch_from_junction are unchanged — they handle topology, not
# tip selection. Only choose_tip_branch_from_junction has new logic.


# If two branches' mean thicknesses differ by less than this many pixels,
# treat as a true tie and fall back to atrium-proximity. Loose tolerance
# is reasonable here because mean_thickness is averaged over 25 points
# of skeleton, so it's noisier than the single-endpoint thickness used
# in cell 11.
BRANCH_THICKNESS_TIE_TOL = 2.0


def closest_skeleton_point_to_atrium(skel_bool, atrium_mask):
    atrium_bool = atrium_mask > 0
    dist_map = distance_transform_edt(~atrium_bool)

    ys, xs = np.nonzero(skel_bool)
    if len(ys) == 0:
        return None

    coords = list(zip(ys, xs))
    best = min(coords, key=lambda p: dist_map[p[0], p[1]])
    return (int(best[0]), int(best[1]))


def nearest_junction_to_point(junctions, point):
    if point is None or not junctions:
        return None
    py, px = point
    return min(junctions, key=lambda j: (j[0] - py) ** 2 + (j[1] - px) ** 2)


def trace_branch_from_junction(junction, next_point, pts_set, max_steps=400):
    path = [junction, next_point]
    prev = junction
    current = next_point

    for _ in range(max_steps):
        neighbors = get_neighbors_from_pts(current, pts_set)
        neighbors = [n for n in neighbors if n != prev]

        if len(neighbors) == 0:
            break
        if len(neighbors) > 1:
            break

        nxt = neighbors[0]
        path.append(nxt)
        prev = current
        current = nxt

    return path


def choose_tip_branch_from_junction(skel_bool, dist, atrium_mask):
    """
    Junction / X-crossing recovery: pick the branch whose tip end has
    the lowest mean thickness — that is the catheter tip.

    Atrium-based metrics are computed for logging and used only as a
    defensive tiebreaker when mean thicknesses are within
    BRANCH_THICKNESS_TIE_TOL of each other.
    """
    closest_pt = closest_skeleton_point_to_atrium(skel_bool, atrium_mask)
    junctions, pts_set = detect_junction_points(skel_bool)

    if closest_pt is None or not junctions:
        return None

    junction = nearest_junction_to_point(junctions, closest_pt)
    if junction is None:
        return None

    neighbors = get_neighbors_from_pts(junction, pts_set)
    if len(neighbors) < 2:
        return None

    atrium_bool = atrium_mask > 0
    dist_map_to_atrium = distance_transform_edt(~atrium_bool)

    branch_candidates = []
    for nb in neighbors:
        branch = trace_branch_from_junction(junction, nb, pts_set, max_steps=500)
        if len(branch) < 10:
            continue

        branch_path = [(float(y), float(x)) for (y, x) in branch]
        offset = min(8, max(0, len(branch_path) - 1))
        branch_path_offset = branch_path[offset:]

        if len(branch_path_offset) < 5:
            continue

        tip_path = extract_tip_segment(branch_path_offset, TIP_LENGTH_PX)
        tip_mask = rasterize_segment_mask(skel_bool.shape, tip_path, dist)

        overlap_pixels = int(np.sum((tip_mask > 0) & atrium_bool))
        tip_pixels     = int(np.sum(tip_mask > 0))
        inside_ratio   = overlap_pixels / tip_pixels if tip_pixels > 0 else 0.0

        # Mean thickness along the first 25 px of the branch — this is
        # the primary signal in v9. Lower = the free thin tip end.
        thickness_vals = [2.0 * float(dist[y, x]) for y, x in branch[:25]]
        mean_thickness = float(np.mean(thickness_vals)) if thickness_vals else np.inf

        # Retained for diagnostics / tiebreaking.
        min_path_distance = _min_dist_to_atrium_full_path(
            tip_path, dist_map_to_atrium, skel_bool.shape
        )
        path_distances_25 = [float(dist_map_to_atrium[y, x]) for y, x in branch[:25]]
        mean_path_distance = (
            float(np.mean(path_distances_25)) if path_distances_25 else np.inf
        )

        endpoint_y = float(branch[-1][0])

        branch_candidates.append({
            "junction":           junction,
            "neighbor":           nb,
            "path":               branch_path_offset,
            "tip_path":           tip_path,
            "overlap_pixels":     overlap_pixels,
            "inside_ratio":       inside_ratio,
            "mean_thickness":     mean_thickness,
            "mean_path_distance": mean_path_distance,
            "min_path_distance":  min_path_distance,
            "endpoint_y":         endpoint_y,
        })

    if not branch_candidates:
        return None

    print("\n[DEBUG] Junction branch ranking:")
    for c in branch_candidates:
        print(
            f"  neighbor={c['neighbor']} | thick={c['mean_thickness']:.1f} | "
            f"min_path_dist={c['min_path_distance']:.1f} | "
            f"endpoint_y={c['endpoint_y']:.0f} | overlap={c['overlap_pixels']}"
        )

    # ── Pick the thinnest branch ───────────────────────────────────────
    by_thickness = sorted(branch_candidates, key=lambda c: c["mean_thickness"])
    thinnest = by_thickness[0]

    if len(by_thickness) == 1:
        print(f"[DEBUG] Only one branch candidate: neighbor={thinnest['neighbor']}")
        return thinnest

    second = by_thickness[1]
    if abs(thinnest["mean_thickness"] - second["mean_thickness"]) < BRANCH_THICKNESS_TIE_TOL:
        tied = [c for c in branch_candidates
                if abs(c["mean_thickness"] - thinnest["mean_thickness"])
                   < BRANCH_THICKNESS_TIE_TOL]
        tied.sort(key=lambda c: (
            c["min_path_distance"],
            -c["endpoint_y"],
            -c["overlap_pixels"],
        ))
        best = tied[0]
        print(f"[DEBUG] Branch thickness tie "
              f"({thinnest['mean_thickness']:.1f} ≈ {second['mean_thickness']:.1f}) — "
              f"atrium-proximity tiebreaker picked neighbor={best['neighbor']}")
        return best

    print(f"[DEBUG] Thinnest branch wins: neighbor={thinnest['neighbor']} "
          f"(thick={thinnest['mean_thickness']:.1f}, next={second['mean_thickness']:.1f})")
    return thinnest

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — REPLACE compute_centerline_fixed   (v10.6)
# ─────────────────────────────────────────────────────────────────────────────
# Fix for v10.5:
# Rule 1 split into two sub-rules:
#   (a) real free endpoint with small gap -> success (not ambiguous)
#   (b) near-original endpoint uses lower gap threshold (2.0)
# Rule 2 cut-point uses lower gap threshold (2.0) directly.
#
# Status semantics:
#   success              → no loop, OR loop but real free end with small gap.
#   success_overlap_mode → loop AND confident tip.
#   ambiguous_tip        → loop AND not confident.

import numpy as np


STATUS_CONFIDENCE_RATIO          = 0.70
ENDPOINT_PROXIMITY_TOLERANCE     = 25.0
THICKNESS_ABS_GAP_MIN            = 5.0   # for real free endpoints -> overlap_mode
THICKNESS_ABS_GAP_MIN_CUT_POINT  = 2.0   # for near-original and cut-point tips


def _coord_in_set(coord, coord_list, tolerance=2):
    if not coord_list:
        return False
    cy, cx = int(round(coord[0])), int(round(coord[1]))
    for p in coord_list:
        py, px = int(p[0]), int(p[1])
        if abs(py - cy) <= tolerance and abs(px - cx) <= tolerance:
            return True
    return False


def _tip_near_any_original(tip_coord, original_endpoint_list,
                            tolerance=ENDPOINT_PROXIMITY_TOLERANCE):
    if not original_endpoint_list:
        return False
    ty, tx = float(tip_coord[0]), float(tip_coord[1])
    for ep in original_endpoint_list:
        ey, ex = float(ep[0]), float(ep[1])
        if np.hypot(ty - ey, tx - ex) <= tolerance:
            return True
    return False


def compute_centerline_fixed(gray8: np.ndarray, atrium_mask: np.ndarray = None):
    bin_img   = binarize_catheter(gray8)
    mask_bool = bin_img.astype(bool)

    if mask_bool.sum() == 0:
        return None, None, None, None, None, "empty_mask"

    skel_bool, dist = build_skeleton_and_dist(mask_bool)
    skel_bool = prune_short_spurs(
        skel_bool, dist, max_len=SPUR_MAX_LEN, dt_ratio=SPUR_DT_RATIO
    )

    original_endpoints, pts_set = detect_all_endpoints(skel_bool)
    original_endpoint_list = list(original_endpoints)

    had_loop = has_real_loop(skel_bool)

    loop_recovery_confident = True

    if had_loop:
        print(f"  [centerline] original_endpoints before recovery: "
              f"{list(original_endpoints)}")
        skel_bool, endpoints, pts_set, loop_recovery_confident = recover_loop_endpoints(
            skel_bool, dist, original_endpoints,
            detect_all_endpoints_fn=detect_all_endpoints,
            detect_junction_points_fn=detect_junction_points,
            get_neighbors_from_pts_fn=get_neighbors_from_pts,
        )
        loop_recovery_succeeded = (len(endpoints) >= 2)
    else:
        endpoints = original_endpoints
        loop_recovery_succeeded = True

        if len(endpoints) < 2:
            skel_bool, endpoints, pts_set, loop_recovery_confident = recover_loop_endpoints(
                skel_bool, dist, original_endpoints,
                detect_all_endpoints_fn=detect_all_endpoints,
                detect_junction_points_fn=detect_junction_points,
                get_neighbors_from_pts_fn=get_neighbors_from_pts,
            )

    if len(endpoints) < 2:
        return bin_img, dist, None, None, None, "ambiguous_tip"

    if atrium_mask is not None:
        tip_coord, best_candidate = (
            select_best_tip_endpoint_with_atrium_original_only(
                endpoints, pts_set, skel_bool, dist,
                atrium_mask, gray8.shape
            )
        )
    else:
        tip_coord = select_best_tip_endpoint_basic(
            endpoints, pts_set, dist
        )
        best_candidate = None

    if tip_coord is None:
        return bin_img, dist, None, None, None, "tip_not_found"

    full_path = build_path_from_tip(skel_bool, dist, tip_coord)
    if full_path is None:
        return bin_img, dist, None, None, None, "path_not_found"

    path_len = cumulative_arclength(full_path)[-1] if len(full_path) > 1 else 0.0
    if path_len < 150.0 and len(endpoints) >= 2:
        print(f"[DEBUG] Path from tip too short ({path_len:.1f}px) — "
              f"trying other endpoints")
        best_path = full_path
        best_len  = path_len
        best_tip  = tip_coord
        for alt_ep in endpoints:
            if (int(round(alt_ep[0])) == int(round(tip_coord[0])) and
                    int(round(alt_ep[1])) == int(round(tip_coord[1]))):
                continue
            alt_path = build_path_from_tip(skel_bool, dist, alt_ep)
            if alt_path is None:
                continue
            alt_len = (cumulative_arclength(alt_path)[-1]
                       if len(alt_path) > 1 else 0.0)
            if alt_len > best_len:
                best_len  = alt_len
                best_path = alt_path
                best_tip  = alt_ep
        if best_tip != tip_coord:
            print(f"[DEBUG] Switched tip from {tip_coord} "
                  f"(path={path_len:.1f}px) to {best_tip} "
                  f"(path={best_len:.1f}px)")
        full_path = best_path
        tip_coord = best_tip

    tip_path = extract_tip_segment(full_path, TIP_LENGTH_PX)

    # ── Status assignment (v10.6) ─────────────────────────────────────────
    if not (had_loop and loop_recovery_succeeded):
        status = "success"

    else:
        # Geometric fallback → always ambiguous
        if not loop_recovery_confident:
            print(f"[DEBUG] Loop recovery used geometric fallback → ambiguous_tip")
            return bin_img, dist, tip_path, full_path, tip_coord, "ambiguous_tip"

        tip_is_real_free_end = _coord_in_set(tip_coord, original_endpoint_list)
        tip_near_original    = _tip_near_any_original(tip_coord, original_endpoint_list)

        # ── Thickness computation ────────────────────────────────────────
        chosen_thickness = (
            best_candidate.get("thickness")
            if best_candidate is not None else None
        )
        other_thicknesses = []
        for ep in endpoints:
            if (int(round(ep[0])) == int(round(tip_coord[0])) and
                    int(round(ep[1])) == int(round(tip_coord[1]))):
                continue
            t = get_endpoint_thickness(ep, dist, pts_set, window=TIP_AVG_WIN)
            other_thicknesses.append(t)

        ratio    = 1.0
        abs_gap  = 0.0
        next_thin = float('inf')

        if chosen_thickness is not None and other_thicknesses:
            next_thin = min(other_thicknesses)
            if next_thin > 0:
                ratio   = chosen_thickness / next_thin
                abs_gap = next_thin - chosen_thickness
            print(f"[DEBUG] Thickness: chosen={chosen_thickness:.1f}, "
                  f"next={next_thin:.1f}, ratio={ratio:.2f}, gap={abs_gap:.1f}")

        # ── Atrium-contact check ─────────────────────────────────────────
        tip_touches_atrium = (
            best_candidate is not None and
            (best_candidate.get("overlap_pixels", 0) > 0 or
             best_candidate.get("min_path_dist", 999) == 0.0)
        )

        # ── Status decision (v10.6) ──────────────────────────────────────
        # Rule 1a — real original free endpoint:
        #   Large gap -> success_overlap_mode (confident loop recovery)
        #   Small gap -> success (tip is genuine, gap just happens to be small
        #                on predicted masks due to uniform thickness rendering)
        if tip_is_real_free_end:
            if abs_gap >= THICKNESS_ABS_GAP_MIN:
                status = "success_overlap_mode"
                print(f"[DEBUG] Real free endpoint, gap={abs_gap:.1f} >= "
                      f"{THICKNESS_ABS_GAP_MIN} → success_overlap_mode")
            else:
                status = "success"
                print(f"[DEBUG] Real free endpoint, gap={abs_gap:.1f} < "
                      f"{THICKNESS_ABS_GAP_MIN} → success")

        # Rule 1b — near-original endpoint (within proximity tolerance):
        #   Uses lower threshold since predicted masks compress thickness range
        elif tip_near_original:
            if abs_gap >= THICKNESS_ABS_GAP_MIN_CUT_POINT:
                status = "success_overlap_mode"
                print(f"[DEBUG] Near-original endpoint, gap={abs_gap:.1f} >= "
                      f"{THICKNESS_ABS_GAP_MIN_CUT_POINT} → success_overlap_mode")
            else:
                status = "ambiguous_tip"
                print(f"[DEBUG] Near-original endpoint, gap={abs_gap:.1f} < "
                      f"{THICKNESS_ABS_GAP_MIN_CUT_POINT} → ambiguous_tip")

        # Rule 2 — synthetic cut-point: must touch atrium and meet lower gap bar
        elif tip_touches_atrium and abs_gap >= THICKNESS_ABS_GAP_MIN_CUT_POINT:
            status = "success_overlap_mode"
            print(f"[DEBUG] Cut-point tip: touches atrium and gap={abs_gap:.1f} >= "
                  f"{THICKNESS_ABS_GAP_MIN_CUT_POINT} → success_overlap_mode")

        else:
            status = "ambiguous_tip"
            print(f"[DEBUG] Cut-point tip: not confident "
                  f"(gap={abs_gap:.1f}, touches_atrium={tip_touches_atrium}) "
                  f"→ ambiguous_tip")

    return bin_img, dist, tip_path, full_path, tip_coord, status

In [20]:
# 200PX TIP REGION TO ATRIUM DISTANCE FEATURES
# Extracts spatial relationship features between the 200px tip mask and the
# atrium mask. These are the most important features for the ML model since
# they directly measure how well the catheter tip is positioned relative to
# the atrium. Three categories of features are computed:
#
# 1) BOUNDING BOX DISTANCES — compares the rectangular bounding boxes of the
#    tip and atrium. bbox_dy_top/bottom measure vertical alignment (positive =
#    tip is inside the atrium's vertical span), bbox_dx_left/right measure
#    horizontal alignment. bbox_inside_atrium is True only if the tip bounding
#    box is fully contained within the atrium bounding box in both directions.
#    These are coarse but fast positional features.
#
# 2) EDGE-TO-EDGE DISTANCES — compares the actual pixel edges of the tip and
#    atrium masks column by column (for vertical) and row by row (for
#    horizontal). More precise than bounding boxes because they follow the
#    real shape of both masks rather than just their rectangular extents.
#    Both minimum and mean distances are computed so the model can distinguish
#    between partially overlapping and fully contained cases.
#
# 3) NEAREST REGION DISTANCE — finds the single closest pair of pixels between
#    the tip mask and the atrium mask using the distance transform. This gives
#    the true minimum gap between the two regions — zero means they are
#    touching or overlapping. best_dy and best_dx record the direction of that
#    nearest pair so the model knows whether the tip is above, below, left, or
#    right of the atrium.
#
# All raw pixel distances are also normalised by the atrium height/width so
# features are comparable across images of different resolutions and atrium
# sizes — without normalisation a 50px gap in a small image and a 50px gap
# in a large image would look the same but mean very different things
# clinically.
#
# get_mask_bbox() returns the bounding box (ymin, ymax, xmin, xmax) of any
# binary mask — used for both tip and atrium bounding box computation.
# get_mask_edge_arrays() scans every column and row of a mask to find the
# topmost, bottommost, leftmost, and rightmost pixel positions — these arrays
# are used for the precise edge-to-edge distance comparisons.

def get_mask_bbox(mask):
    """
    Returns bounding box of a binary mask as:
      y_min, y_max, x_min, x_max
    If empty, returns None.
    """
    ys, xs = np.where(mask > 0)
    if len(ys) == 0 or len(xs) == 0:
        return None
    return int(ys.min()), int(ys.max()), int(xs.min()), int(xs.max())


def get_mask_edge_arrays(mask):
    """
    For a binary mask, compute:
      - top_y[x], bottom_y[x] for each column x where mask exists
      - left_x[y], right_x[y] for each row y where mask exists

    Missing positions are set to -1.
    """
    mask_bool = mask > 0
    h, w = mask_bool.shape

    top_y = np.full(w, -1, dtype=np.int32)
    bottom_y = np.full(w, -1, dtype=np.int32)

    for x in range(w):
        ys = np.where(mask_bool[:, x])[0]
        if len(ys) > 0:
            top_y[x] = int(ys.min())
            bottom_y[x] = int(ys.max())

    left_x = np.full(h, -1, dtype=np.int32)
    right_x = np.full(h, -1, dtype=np.int32)

    for y in range(h):
        xs = np.where(mask_bool[y, :])[0]
        if len(xs) > 0:
            left_x[y] = int(xs.min())
            right_x[y] = int(xs.max())

    return top_y, bottom_y, left_x, right_x


def tip_region_to_atrium_distances(tip_mask, atrium_mask):
    """
    Distance features computed from the trimmed 200 px tip region to the atrium.

    This is REGION-BASED, not point-based.

    Returns:
    ------------------------------------------------------
    1) Bounding-box distances:
       bbox_dy_top      = tip_top - atrium_top
       bbox_dy_bottom   = atrium_bottom - tip_bottom
       bbox_dx_left     = tip_left - atrium_left
       bbox_dx_right    = atrium_right - tip_right

       Positive means tip bbox lies inside that atrial span direction.
       Negative means tip bbox extends outside that border.

    2) Edge-to-edge distances over overlapping x/y ranges:
       edge_dy_top_min / mean
       edge_dy_bottom_min / mean
       edge_dx_left_min / mean
       edge_dx_right_min / mean

       These compare the actual 200 px tip mask edges against the atrium edges,
       not just bounding boxes.

    3) Nearest distances between any tip pixel and atrium:
       region_nearest_euclidean
       region_nearest_dy
       region_nearest_dx
    """
    tip_bool = tip_mask > 0
    atrium_bool = atrium_mask > 0

    if tip_bool.sum() == 0 or atrium_bool.sum() == 0:
        return None


    # 1) Bounding-box features

    tip_bbox = get_mask_bbox(tip_mask)
    atrium_bbox = get_mask_bbox(atrium_mask)

    if tip_bbox is None or atrium_bbox is None:
        return None

    tip_ymin, tip_ymax, tip_xmin, tip_xmax = tip_bbox
    atr_ymin, atr_ymax, atr_xmin, atr_xmax = atrium_bbox

    bbox_dy_top = float(tip_ymin - atr_ymin)
    bbox_dy_bottom = float(atr_ymax - tip_ymax)
    bbox_dx_left = float(tip_xmin - atr_xmin)
    bbox_dx_right = float(atr_xmax - tip_xmax)

    bbox_vertical_inside = (bbox_dy_top >= 0) and (bbox_dy_bottom >= 0)
    bbox_horizontal_inside = (bbox_dx_left >= 0) and (bbox_dx_right >= 0)
    bbox_inside_atrium = bbox_vertical_inside and bbox_horizontal_inside


    # 2) Edge-to-edge features

    tip_top_y, tip_bottom_y, tip_left_x, tip_right_x = get_mask_edge_arrays(tip_mask)
    atr_top_y, atr_bottom_y, atr_left_x, atr_right_x = get_mask_edge_arrays(atrium_mask)

    # compare over shared valid x columns
    valid_x = np.where((tip_top_y >= 0) & (atr_top_y >= 0))[0]
    if len(valid_x) > 0:
        dy_top_arr = tip_top_y[valid_x].astype(np.float32) - atr_top_y[valid_x].astype(np.float32)
        dy_bottom_arr = atr_bottom_y[valid_x].astype(np.float32) - tip_bottom_y[valid_x].astype(np.float32)

        edge_dy_top_min = float(np.min(dy_top_arr))
        edge_dy_top_mean = float(np.mean(dy_top_arr))

        edge_dy_bottom_min = float(np.min(dy_bottom_arr))
        edge_dy_bottom_mean = float(np.mean(dy_bottom_arr))
    else:
        edge_dy_top_min = np.nan
        edge_dy_top_mean = np.nan
        edge_dy_bottom_min = np.nan
        edge_dy_bottom_mean = np.nan

    # compare over shared valid y rows
    valid_y = np.where((tip_left_x >= 0) & (atr_left_x >= 0))[0]
    if len(valid_y) > 0:
        dx_left_arr = tip_left_x[valid_y].astype(np.float32) - atr_left_x[valid_y].astype(np.float32)
        dx_right_arr = atr_right_x[valid_y].astype(np.float32) - tip_right_x[valid_y].astype(np.float32)

        edge_dx_left_min = float(np.min(dx_left_arr))
        edge_dx_left_mean = float(np.mean(dx_left_arr))

        edge_dx_right_min = float(np.min(dx_right_arr))
        edge_dx_right_mean = float(np.mean(dx_right_arr))
    else:
        edge_dx_left_min = np.nan
        edge_dx_left_mean = np.nan
        edge_dx_right_min = np.nan
        edge_dx_right_mean = np.nan

    # 3) Nearest tip-region to atrium distance

    tip_ys, tip_xs = np.where(tip_bool)
    atr_ys, atr_xs = np.where(atrium_bool)

    # use distance transform for nearest Euclidean distance from tip region to atrium
    atrium_dist_map = distance_transform_edt(~atrium_bool)
    region_nearest_euclidean = float(np.min(atrium_dist_map[tip_ys, tip_xs])) if len(tip_ys) > 0 else np.nan

    # nearest dy/dx by brute-force nearest-pair search
    # acceptable here because masks are small enough for single image feature extraction
    best_d2 = np.inf
    best_dy = np.nan
    best_dx = np.nan

    for ty, tx in zip(tip_ys, tip_xs):
        dy = atr_ys.astype(np.float32) - float(ty)
        dx = atr_xs.astype(np.float32) - float(tx)
        d2 = dy * dy + dx * dx
        k = int(np.argmin(d2))
        if d2[k] < best_d2:
            best_d2 = float(d2[k])
            best_dy = float(dy[k])
            best_dx = float(dx[k])

    # normalize by atrium bbox size

    atrium_height = max(1.0, float(atr_ymax - atr_ymin))
    atrium_width = max(1.0, float(atr_xmax - atr_xmin))

    return {
        # bounding boxes
        "tip_bbox_ymin": tip_ymin,
        "tip_bbox_ymax": tip_ymax,
        "tip_bbox_xmin": tip_xmin,
        "tip_bbox_xmax": tip_xmax,
        "atrium_bbox_ymin": atr_ymin,
        "atrium_bbox_ymax": atr_ymax,
        "atrium_bbox_xmin": atr_xmin,
        "atrium_bbox_xmax": atr_xmax,

        "bbox_dy_top": bbox_dy_top,
        "bbox_dy_bottom": bbox_dy_bottom,
        "bbox_dx_left": bbox_dx_left,
        "bbox_dx_right": bbox_dx_right,

        "bbox_vertical_inside": bool(bbox_vertical_inside),
        "bbox_horizontal_inside": bool(bbox_horizontal_inside),
        "bbox_inside_atrium": bool(bbox_inside_atrium),

        # edge features
        "edge_dy_top_min": edge_dy_top_min,
        "edge_dy_top_mean": edge_dy_top_mean,
        "edge_dy_bottom_min": edge_dy_bottom_min,
        "edge_dy_bottom_mean": edge_dy_bottom_mean,

        "edge_dx_left_min": edge_dx_left_min,
        "edge_dx_left_mean": edge_dx_left_mean,
        "edge_dx_right_min": edge_dx_right_min,
        "edge_dx_right_mean": edge_dx_right_mean,

        # nearest region-to-atrium distances
        "region_nearest_euclidean": region_nearest_euclidean,
        "region_nearest_dy": best_dy,
        "region_nearest_dx": best_dx,

        # normalized versions
        "bbox_dy_top_norm": bbox_dy_top / atrium_height,
        "bbox_dy_bottom_norm": bbox_dy_bottom / atrium_height,
        "bbox_dx_left_norm": bbox_dx_left / atrium_width,
        "bbox_dx_right_norm": bbox_dx_right / atrium_width,

        "edge_dy_top_min_norm": edge_dy_top_min / atrium_height if not np.isnan(edge_dy_top_min) else np.nan,
        "edge_dy_top_mean_norm": edge_dy_top_mean / atrium_height if not np.isnan(edge_dy_top_mean) else np.nan,
        "edge_dy_bottom_min_norm": edge_dy_bottom_min / atrium_height if not np.isnan(edge_dy_bottom_min) else np.nan,
        "edge_dy_bottom_mean_norm": edge_dy_bottom_mean / atrium_height if not np.isnan(edge_dy_bottom_mean) else np.nan,

        "edge_dx_left_min_norm": edge_dx_left_min / atrium_width if not np.isnan(edge_dx_left_min) else np.nan,
        "edge_dx_left_mean_norm": edge_dx_left_mean / atrium_width if not np.isnan(edge_dx_left_mean) else np.nan,
        "edge_dx_right_min_norm": edge_dx_right_min / atrium_width if not np.isnan(edge_dx_right_min) else np.nan,
        "edge_dx_right_mean_norm": edge_dx_right_mean / atrium_width if not np.isnan(edge_dx_right_mean) else np.nan,

        "region_nearest_dist_norm_h": region_nearest_euclidean / atrium_height,
        "region_nearest_dist_norm_w": region_nearest_euclidean / atrium_width,
    }

In [21]:
# OTHER TIP + ATRIUM FEATURES
# path_inside_ratio() measures what fraction of the 200px tip centerline
# points fall directly inside the atrium mask — purely point-based unlike
# the region-based features above. It walks every (y, x) coordinate of the
# tip path, rounds to the nearest pixel, and checks whether that pixel is
# inside the atrium mask. The result is a ratio between 0.0 and 1.0 where
# 1.0 means the entire tip centerline runs through the atrium and 0.0 means
# no part of the centerline touches the atrium at all. This is used as the
# centerline_inside_ratio feature in the ML model — complementing the
# tip_area_inside_ratio which measures the filled tube mask overlap. Together
# they give the model both a centerline view and a full-width tube view of
# how much of the catheter tip is inside the atrium.

def path_inside_ratio(tip_path, atrium_mask):
    inside = 0
    total = 0
    for y, x in tip_path:
        yy, xx = int(round(y)), int(round(x))
        if 0 <= yy < atrium_mask.shape[0] and 0 <= xx < atrium_mask.shape[1]:
            total += 1
            if atrium_mask[yy, xx] > 0:
                inside += 1
    return inside / total if total > 0 else 0.0


In [22]:
# ══ REPLACE THE ENTIRE CELL CONTENT WITH THIS ════════════════════════════════

from scipy.ndimage import distance_transform_edt

# CELL 15b — UPDATED 27/05/2026
# Changes vs previous version
#   REMOVED : tip_in_upper_third, tip_in_middle_third, tip_in_lower_third
#   ADDED   : distance_from_optimal_depth  (signed, continuous)
#             tip_to_atrium_dist_norm      (0 when tip is inside atrium)

def tip_point_and_depth_features(tip_path, atrium_mask, tip_coord):
    """
    Point-level tip features. Called unchanged from process_single_case_to_row.

    Returns
    -------
    dict with keys:
        tip_depth_in_atrium_px       – arc-length depth inside atrium (px)
        tip_depth_in_atrium_norm     – depth / atrium_height
        tip_y_norm_to_atrium         – (tip_y – atrium_top) / atrium_height
        tip_x_norm_to_atrium         – (tip_x – atrium_left) / atrium_width
        distance_from_optimal_depth  – NEW: signed distance from optimal zone
                                         optimal zone = [atrium_top + H/3,
                                                         atrium_top + H/2]
                                         < 0  tip is ABOVE zone (too shallow)
                                           0  tip is INSIDE zone (correct)
                                         > 0  tip is BELOW zone (too deep)
                                         normalised by atrium_height
        tip_to_atrium_dist_norm      – NEW: Euclidean distance from tip pixel
                                         to nearest atrium pixel / atrium_height
                                         0.0 when tip is already inside
    """
    nan = np.nan
    feats = {
        "tip_depth_in_atrium_px":       nan,
        "tip_depth_in_atrium_norm":     nan,
        "tip_y_norm_to_atrium":         nan,
        "tip_x_norm_to_atrium":         nan,
        "distance_from_optimal_depth":  nan,   # NEW
        "tip_to_atrium_dist_norm":      nan,   # NEW
    }

    if (tip_path is None or len(tip_path) == 0
            or atrium_mask is None or tip_coord is None):
        return feats

    H, W = atrium_mask.shape
    atrium_bbox = get_mask_bbox(atrium_mask)
    if atrium_bbox is None:
        return feats

    a_ymin, a_ymax, a_xmin, a_xmax = atrium_bbox
    atrium_height = max(1.0, float(a_ymax - a_ymin))
    atrium_width  = max(1.0, float(a_xmax - a_xmin))

    # ── 1. Arc-length depth inside the atrium (unchanged) ───────────────────
    depth = 0.0
    prev  = None
    inside_at_start = False
    for i, (y, x) in enumerate(tip_path):
        yy, xx = int(round(y)), int(round(x))
        if not (0 <= yy < H and 0 <= xx < W):
            break
        is_inside = atrium_mask[yy, xx] > 0
        if i == 0:
            inside_at_start = is_inside
            prev = (y, x)
            continue
        if not is_inside:
            break
        depth += float(np.hypot(y - prev[0], x - prev[1]))
        prev = (y, x)
    if not inside_at_start:
        depth = 0.0

    feats["tip_depth_in_atrium_px"]   = depth
    feats["tip_depth_in_atrium_norm"] = depth / atrium_height

    # ── 2. Normalised tip position (unchanged) ───────────────────────────────
    ty, tx = float(tip_coord[0]), float(tip_coord[1])
    feats["tip_y_norm_to_atrium"] = (ty - a_ymin) / atrium_height
    feats["tip_x_norm_to_atrium"] = (tx - a_xmin) / atrium_width

    # ── 3. distance_from_optimal_depth  (NEW — replaces binary thirds) ───────
    # Clinically, the tip should sit between the upper-third line and the
    # mid-line of the right atrium.
    optimal_top    = a_ymin + atrium_height / 3.0   # upper-third line
    optimal_bottom = a_ymin + atrium_height / 2.0   # mid-line

    if ty < optimal_top:
        signed_dist = (ty - optimal_top)    / atrium_height   # negative
    elif ty > optimal_bottom:
        signed_dist = (ty - optimal_bottom) / atrium_height   # positive
    else:
        signed_dist = 0.0                                      # correct zone

    feats["distance_from_optimal_depth"] = signed_dist

    # ── 4. Distance from tip point to nearest atrium pixel  (NEW) ────────────
    yy, xx = int(round(ty)), int(round(tx))
    tip_is_inside = (0 <= yy < H and 0 <= xx < W and atrium_mask[yy, xx] > 0)

    if tip_is_inside:
        feats["tip_to_atrium_dist_norm"] = 0.0
    else:
        dist_map = distance_transform_edt(~(atrium_mask > 0))
        if 0 <= yy < H and 0 <= xx < W:
            feats["tip_to_atrium_dist_norm"] = (
                float(dist_map[yy, xx]) / atrium_height
            )

    return feats

In [23]:
# TIP DIRECTION FEATURES
# Measures the direction the catheter tip is pointing at its very end.
# Takes two points on the tip path — the starting point (index 0, the tip
# endpoint) and a point 15 pixels further along (index 15) — and computes
# the direction vector between them. Using 15 pixels of separation rather
# than adjacent pixels makes the direction estimate stable and resistant to
# the small wiggles that remain even after smoothing. The vector is normalised
# to unit length so tip_dx and tip_dy are always between -1 and +1 regardless
# of image scale. tip_angle is the arctangent of that vector in radians.
# In a correctly placed catheter the tip should be pointing roughly downward
# (tip_dy close to +1, tip_angle close to -π/2) since the catheter descends
# into the right atrium — a tip pointing sideways or upward is a strong
# signal of misplacement. These three features give the ML model information
# about catheter orientation that the purely positional bbox and overlap
# features cannot capture.

def tip_direction_features(tip_path):
    if tip_path is None or len(tip_path) < 15:
        return {"tip_dx": 0.0, "tip_dy": 0.0, "tip_angle": 0.0}

    # use longer segment → more stable
    y0, x0 = tip_path[0]
    y1, x1 = tip_path[15]

    dy = y1 - y0
    dx = x1 - x0

    norm = np.sqrt(dx**2 + dy**2) + 1e-6
    dx /= norm
    dy /= norm

    angle = np.arctan2(dy, dx)

    return {
        "tip_dx": dx,   # normalized
        "tip_dy": dy,
        "tip_angle": angle
    }

In [24]:
# OVERLAYS
# Creates colour visualisation images for inspection and debugging.
#
# make_overlay() draws the centerline and tip segment on top of the original
# grayscale catheter image using three colour layers:
#   GREEN  — the full centerline from tip to connector end (full_path)
#   RED    — the 200px tip segment (tip_path)
#   YELLOW — a filled circle marking the exact tip endpoint (tip_coord)
# This is the primary visual used to verify the pipeline is selecting the
# correct tip and tracing the centerline accurately. All the sample images
# shown throughout the thesis were produced by this function.
#
# make_tip_atrium_overlay() creates a colour map showing the spatial
# relationship between the tip mask and atrium mask using three colours:
#   GREEN  — atrium region only (no tip overlap)
#   RED    — tip region only (outside the atrium)
#   YELLOW — pixels where tip and atrium overlap simultaneously
# A correctly placed catheter should show a large yellow overlap region,
# while a misplaced catheter will show mostly red with little or no yellow.
# This overlay makes the overlap features (tip_area_inside_ratio, iou) visually
# interpretable and was used to validate the feature extraction logic.

def make_overlay(gray8, full_path, tip_path, tip_coord):
    overlay = cv2.cvtColor(gray8, cv2.COLOR_GRAY2BGR)

    if full_path:
        pts = np.array([(int(round(x)), int(round(y))) for (y, x) in full_path], np.int32)
        cv2.polylines(overlay, [pts], False, (0, 255, 0), 2)

    if tip_path:
        pts = np.array([(int(round(x)), int(round(y))) for (y, x) in tip_path], np.int32)
        cv2.polylines(overlay, [pts], False, (0, 0, 255), 3)

    if tip_coord:
        cv2.circle(overlay, (tip_coord[1], tip_coord[0]), 6, (0, 255, 255), -1)

    return overlay


def make_tip_atrium_overlay(tip_mask, atrium_mask):
    overlay = np.zeros((tip_mask.shape[0], tip_mask.shape[1], 3), dtype=np.uint8)

    tip_bool = tip_mask > 0
    atrium_bool = atrium_mask > 0

    overlay[atrium_bool] = [0, 255, 0]
    overlay[tip_bool] = [255, 0, 0]
    overlay[tip_bool & atrium_bool] = [255, 255, 0]

    return overlay

In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# TIP POINT VISUALISATION — shows only the single (y, x) tip coordinate
# No 200px tube, no full centerline — just the tip point on the image
# ─────────────────────────────────────────────────────────────────────────────

def make_tip_point_overlay(gray8, tip_coord, atrium_mask=None, sigma=15):
    """
    Draws only the tip point on the image.

    Shows:
      YELLOW dot   — the detected tip coordinate (tip_coord)
      GREEN outline — atrium boundary (optional, for clinical context)
      RED Gaussian  — soft halo showing the tip location region (sigma px)

    Parameters
    ----------
    gray8       : 2D uint8 grayscale image
    tip_coord   : (y, x) — single tip pixel from compute_centerline_fixed
    atrium_mask : optional 2D binary mask — draws atrium border in green
    sigma       : size of the Gaussian halo around the tip point
    """
    overlay = cv2.cvtColor(gray8, cv2.COLOR_GRAY2BGR)
    H, W = gray8.shape

    # ── Gaussian halo at tip point (shows the region not the 200px tube) ──
    if tip_coord is not None:
        ty, tx = float(tip_coord[0]), float(tip_coord[1])
        y_grid, x_grid = np.mgrid[0:H, 0:W].astype(np.float32)
        halo = np.exp(-((y_grid - ty)**2 + (x_grid - tx)**2)
                       / (2.0 * sigma**2))
        halo = (halo * 180).astype(np.uint8)   # max intensity 180/255

        # Add red channel halo
        overlay[:, :, 2] = np.clip(
            overlay[:, :, 2].astype(np.int32) + halo, 0, 255
        ).astype(np.uint8)

    # ── Atrium boundary in green (for clinical context) ───────────────────
    if atrium_mask is not None:
        contours, _ = cv2.findContours(
            (atrium_mask > 0).astype(np.uint8),
            cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)

    # ── Tip point as filled yellow circle ─────────────────────────────────
    if tip_coord is not None:
        cy, cx = int(round(tip_coord[0])), int(round(tip_coord[1]))
        cv2.circle(overlay, (cx, cy), 8,  (0, 255, 255), -1)  # filled
        cv2.circle(overlay, (cx, cy), 10, (0,   0,   0),  2)  # black border

    return overlay


def visualise_tip_point_vs_200px(catheter_path, atrium_path, sigma=15):
    """
    Side-by-side comparison:
      Left  — old approach: full 200px tube (red) + centerline (green)
      Right — new approach: tip point only (yellow dot + red halo)

    Use this to confirm the tip point is correct before using it as a feature.
    """
    gray8       = load_grayscale_any(catheter_path)
    atrium_mask = load_binary_mask(atrium_path, reference_shape=gray8.shape)

    bin_img, dist, tip_path, full_path, tip_coord, status = \
        compute_centerline_fixed(gray8, atrium_mask)

    print(f"Status    : {status}")
    print(f"Tip coord : {tip_coord}")

    # ── Left panel: old 200px overlay ─────────────────────────────────────
    if tip_path is not None and dist is not None:
        tip_mask = rasterize_segment_mask(gray8.shape, tip_path, dist)
        old_overlay = make_overlay(gray8, full_path, tip_path, tip_coord)
    else:
        tip_mask    = None
        old_overlay = cv2.cvtColor(gray8, cv2.COLOR_GRAY2BGR)

    # ── Right panel: new tip-point-only overlay ────────────────────────────
    new_overlay = make_tip_point_overlay(
        gray8, tip_coord, atrium_mask=atrium_mask, sigma=sigma
    )

    # ── Plot ──────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    axes[0].imshow(cv2.cvtColor(old_overlay, cv2.COLOR_BGR2RGB))
    axes[0].set_title(
        "OLD approach — 200px tube\n"
        "(green=centerline, red=200px segment, yellow=tip point)",
        fontsize=10
    )
    axes[0].axis("off")

    axes[1].imshow(cv2.cvtColor(new_overlay, cv2.COLOR_BGR2RGB))
    axes[1].set_title(
        "NEW approach — tip point only\n"
        "(green=atrium boundary, red halo=tip region, yellow=tip point)",
        fontsize=10
    )
    axes[1].axis("off")

    image_name = os.path.basename(catheter_path)
    plt.suptitle(f"{image_name}  |  status: {status}", fontsize=12)
    plt.tight_layout()
    plt.show()

    return tip_coord, status

In [26]:
# ── Run on a few sample images to visually verify the tip point ──────────────
# Change these image names to ones from your dataset

sample_images = [
    "IMG-0041-00001.png",   # a success case
    "IMG-0147-00001.png",   # an ambiguous_tip case
    "IMG-0027-00001.png",   # a loop case (success_overlap_mode)
]

for img_name in sample_images:
    cath_path  = os.path.join(CATHETER_FOLDER, img_name)
    atrium_path = os.path.join(ATRIUM_FOLDER,   img_name)

    if os.path.exists(cath_path) and os.path.exists(atrium_path):
        tip_coord, status = visualise_tip_point_vs_200px(
            cath_path, atrium_path, sigma=15
        )
    else:
        print(f"File not found: {img_name}")

/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.


[DEBUG] Candidate ranking:
  coord=(516, 226) | thick=29.7 | min_path_dist=117.2 | tip_y=516 | overlap=0
  coord=(576, 360) | thick=14.4 | min_path_dist=0.0 | tip_y=576 | overlap=933
[DEBUG] Thinnest endpoint wins: (576, 360) (thick=14.4, next=29.7)
Status    : success
Tip coord : (576, 360)
  [centerline] original_endpoints before recovery: []
  [stub_filter] ep=(322, 273)  arm_len=933.9px
  [stub_filter] ep=(313, 274)  arm_len=933.9px
  [stub_filter] after dedup: 1 endpoints
  [stub_filter] final: 1 endpoints
[loop_recovery_v3] All strategies failed — using geometric fallback


/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.


[DEBUG] Candidate ranking:
  coord=(293, 366) | thick=14.0 | min_path_dist=21.9 | tip_y=293 | overlap=0
  coord=(503, 273) | thick=15.2 | min_path_dist=81.4 | tip_y=503 | overlap=0
[DEBUG] Thinnest endpoint wins: (293, 366) (thick=14.0, next=15.2)
[DEBUG] Loop recovery used geometric fallback → ambiguous_tip
Status    : ambiguous_tip
Tip coord : (293, 366)


/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255


  [centerline] original_endpoints before recovery: [(656, 443), (647, 397)]
[strategy1] Protected 2 endpoints, using safe junction at (550, 397)
  [stub_filter] ep=(546, 395)  arm_len=66.0px
  [stub_filter] ep=(552, 401)  arm_len=121.4px
  [stub_filter] ep=(656, 443)  arm_len=121.4px
  [stub_filter] ep=(554, 394)  arm_len=100.9px
  [stub_filter] ep=(647, 397)  arm_len=100.9px
  [stub_filter] after dedup: 3 endpoints
  [stub_filter] final: 3 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 3 endpoints
[WARNING] Path length 113.4px < target 200.0px — returning full path
[WARNING] Path length 113.4px < target 200.0px — returning full path
[WARNING] Path length 94.2px < target 200.0px — returning full path

[DEBUG] Candidate ranking:
  coord=(552, 401) | thick=16.8 | min_path_dist=0.0 | tip_y=552 | overlap=3265
  coord=(656, 443) | thick=31.3 | min_path_dist=0.0 | tip_y=656 | overlap=3265
  coord=(647, 397) | thick=14.5 | min_path_dist=0.0 | tip_y=647 | overlap=1904
[DEBUG]

/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255


In [27]:
# SINGLE IMAGE TEST
# Runs the complete pipeline on one catheter/atrium image pair and produces
# a full visual and numerical report — used for development, debugging, and
# verifying the pipeline works correctly on individual cases before running
# the full batch.
#
# The function runs every stage in sequence:
#   1. Load catheter image and atrium mask, verify they are the same size
#   2. Run compute_centerline_fixed() to get the tip path, full centerline,
#      tip coordinate, and processing status
#   3. Rasterize the 200px tip segment into a filled tube mask
#   4. Compute all overlap features (overlap_pixels, inside_ratio,
#      tip_inside_atrium, centerline_inside_ratio)
#   5. Compute all spatial distance features via tip_region_to_atrium_distances
#   6. Print a full numerical summary of every extracted feature to the console
#   7. Produce a 2×4 visual panel showing all intermediate results:
#        Row 1: original image | binary mask | centerline overlay | distance view
#        Row 2: atrium mask | 200px tip mask | tip+atrium overlay | feature text
#
# The 2×4 panel is the key diagnostic tool — by looking at it you can
# immediately see whether the correct tip was selected (green=full centerline,
# red=200px tip), whether the tip mask overlaps the atrium (yellow in overlay),
# and what the extracted feature values are. This is the image shown in the
# thesis to illustrate the pipeline on individual cases.
#
# Returns a dictionary of all extracted features for that image which can be
# collected across the full dataset into the ML feature CSV.

def process_single_case(catheter_image_path, atrium_mask_path):
    print(f"Catheter image: {catheter_image_path}")
    print(f"Atrium mask:    {atrium_mask_path}")

    gray8 = load_grayscale_any(catheter_image_path)
    atrium_mask = load_binary_mask(atrium_mask_path)

    if gray8.shape != atrium_mask.shape:
        raise ValueError(f"Shape mismatch: catheter image {gray8.shape}, atrium mask {atrium_mask.shape}")

    bin_img, dist, tip_path, full_path, tip_coord, status = compute_centerline_fixed(gray8, atrium_mask)

    print(f"Status: {status}")

    if tip_path is None or dist is None or tip_coord is None:
        print("No tip path found.")
        return None

    tip_mask = rasterize_segment_mask(gray8.shape, tip_path, dist)

    overlap_pixels = int(np.sum((tip_mask > 0) & (atrium_mask > 0)))
    tip_pixels = int(np.sum(tip_mask > 0))
    inside_ratio = overlap_pixels / tip_pixels if tip_pixels > 0 else 0.0

    tip_inside_atrium = bool(atrium_mask[int(round(tip_coord[0])), int(round(tip_coord[1]))] > 0)
    centerline_inside_ratio = path_inside_ratio(tip_path, atrium_mask)

    point_feats = compute_tip_point_features(tip_coord, atrium_mask)
    row.update(point_feats)

    print(f"Tip coordinate: {tip_coord}")
    print(f"Tip mask pixels: {tip_pixels}")
    print(f"Overlap pixels: {overlap_pixels}")
    print(f"Tip area inside atrium ratio: {inside_ratio:.4f}")
    print(f"Tip point inside atrium: {tip_inside_atrium}")
    print(f"Tip centerline inside atrium ratio: {centerline_inside_ratio:.4f}")

    if region_feats is not None:
        print("\n[200 px tip-region to atrium features]")
        print(f"bbox_dy_top:         {region_feats['bbox_dy_top']:.2f} px")
        print(f"bbox_dy_bottom:      {region_feats['bbox_dy_bottom']:.2f} px")
        print(f"bbox_dx_left:        {region_feats['bbox_dx_left']:.2f} px")
        print(f"bbox_dx_right:       {region_feats['bbox_dx_right']:.2f} px")
        print(f"bbox_inside_atrium:  {region_feats['bbox_inside_atrium']}")
        print(f"edge_dy_top_mean:    {region_feats['edge_dy_top_mean']:.2f} px")
        print(f"edge_dy_bottom_mean: {region_feats['edge_dy_bottom_mean']:.2f} px")
        print(f"edge_dx_left_mean:   {region_feats['edge_dx_left_mean']:.2f} px")
        print(f"edge_dx_right_mean:  {region_feats['edge_dx_right_mean']:.2f} px")
        print(f"region_nearest_dist: {region_feats['region_nearest_euclidean']:.2f} px")

    catheter_overlay = make_overlay(gray8, full_path, tip_path, tip_coord)
    tip_atrium_overlay = make_tip_atrium_overlay(tip_mask, atrium_mask)

    # placeholder image instead of old point-distance overlay
    distance_overlay = cv2.cvtColor(gray8, cv2.COLOR_GRAY2BGR)

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    axes[0, 0].imshow(gray8, cmap='gray')
    axes[0, 0].set_title("Catheter Image")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(bin_img, cmap='gray')
    axes[0, 1].set_title("Catheter Binary Mask")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(cv2.cvtColor(catheter_overlay, cv2.COLOR_BGR2RGB))
    axes[0, 2].set_title("Centerline + 200 px Tip")
    axes[0, 2].axis("off")

    axes[0, 3].imshow(cv2.cvtColor(distance_overlay, cv2.COLOR_BGR2RGB))
    axes[0, 3].set_title("Region Distance View")
    axes[0, 3].axis("off")

    axes[1, 0].imshow(atrium_mask, cmap='gray')
    axes[1, 0].set_title("Atrium Mask")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(tip_mask, cmap='gray')
    axes[1, 1].set_title("200 px Tip Mask")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(tip_atrium_overlay)
    axes[1, 2].set_title("Tip + Atrium Overlay")
    axes[1, 2].axis("off")

    text_img = np.ones((gray8.shape[0], gray8.shape[1], 3), dtype=np.uint8) * 255

    if region_feats is not None:
        lines = [
            f"bbox_dy_top    = {region_feats['bbox_dy_top']:.2f}px",
            f"bbox_dy_bottom = {region_feats['bbox_dy_bottom']:.2f}px",
            f"bbox_dx_left   = {region_feats['bbox_dx_left']:.2f}px",
            f"bbox_dx_right  = {region_feats['bbox_dx_right']:.2f}px",
            f"edge_dy_top_mean    = {region_feats['edge_dy_top_mean']:.2f}px",
            f"edge_dy_bottom_mean = {region_feats['edge_dy_bottom_mean']:.2f}px",
            f"edge_dx_left_mean   = {region_feats['edge_dx_left_mean']:.2f}px",
            f"edge_dx_right_mean  = {region_feats['edge_dx_right_mean']:.2f}px",
            f"region_nearest_dist = {region_feats['region_nearest_euclidean']:.2f}px",
        ]

        y0 = 80
        for i, line in enumerate(lines):
            cv2.putText(
                text_img, line, (40, y0 + i * 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 2, cv2.LINE_AA
            )

    axes[1, 3].imshow(text_img)
    axes[1, 3].set_title("Extracted Region Features")
    axes[1, 3].axis("off")

    plt.tight_layout()
    plt.show()
    plt.close()

    results = {
        "status": status,
        "tip_coord": tip_coord,
        "tip_pixels": tip_pixels,
        "overlap_pixels": overlap_pixels,
        "tip_area_inside_ratio": inside_ratio,
        "tip_inside_atrium": tip_inside_atrium,
        "centerline_inside_ratio": centerline_inside_ratio,
    }

    if region_feats is not None:
        results.update(region_feats)

    return results

In [28]:
# FILE MATCHING
# Pairs each catheter image with its corresponding atrium mask by filename.
# Scans the catheter folder for all image files across five supported formats
# (tif, tiff, png, jpg, jpeg) and builds a lookup dictionary of all atrium
# mask files indexed by filename. Then for each catheter file it looks up
# whether an atrium mask with the exact same filename exists in the atrium
# folder. Matched pairs are collected into the pairs list — these are the
# valid cases that will be processed by the pipeline. Catheter images with
# no matching atrium mask are collected separately in missing_atrium so they
# can be reported without silently skipping them. This matching approach
# assumes the catheter and atrium mask files share identical filenames, which
# is the standard convention when masks are exported from annotation tools
# like ITK-SNAP or 3D Slicer. Any naming inconsistency (e.g. one folder uses
# .tif and the other uses .png for the same case) will cause a missed match
# and appear in missing_atrium — these should be investigated before running
# the full batch to ensure no cases are accidentally excluded from the dataset.

def get_image_pairs(catheter_folder, atrium_folder, exts=("*.tif", "*.tiff", "*.png", "*.jpg", "*.jpeg")):
    catheter_files = []
    for ext in exts:
        catheter_files.extend(glob.glob(os.path.join(catheter_folder, ext)))
    catheter_files = sorted(catheter_files)

    atrium_map = {}
    for ext in exts:
        for f in glob.glob(os.path.join(atrium_folder, ext)):
            atrium_map[os.path.basename(f)] = f

    pairs = []
    missing_atrium = []

    for cfile in catheter_files:
        name = os.path.basename(cfile)
        if name in atrium_map:
            pairs.append((cfile, atrium_map[name]))
        else:
            missing_atrium.append(cfile)

    return pairs, missing_atrium

In [29]:
# SAVE DEBUG VISUALS
# Saves all intermediate pipeline images to disk for every processed case,
# using a consistent numbered naming convention so files sort in processing
# order when browsed in a file explorer. For each image the following seven
# files are saved:
#   _01_gray               — original grayscale catheter image
#   _02_bin                — binary mask after Otsu thresholding and cleaning
#   _03_atrium             — atrium binary mask
#   _04_centerline_overlay — catheter image with full centerline (green),
#                            200px tip (red), and tip endpoint (yellow) drawn
#   _05_tip_mask           — filled 200px tip tube mask
#   _06_tip_atrium_overlay — colour map showing tip (red), atrium (green),
#                            and overlap (yellow)
#   _07_distance_overlay   — grayscale image used as distance view placeholder
#
# The base filename is stripped of its extension so all seven files share the
# same base name with only the numbered suffix distinguishing them. The tip
# atrium overlay is converted from RGB to BGR before saving because OpenCV's
# imwrite expects BGR channel order. These saved visuals are what the status
# plotting code later reads back to display the grid of success, ambiguous,
# and overlap mode cases — making this function the bridge between the
# processing pipeline and all downstream visual inspection and thesis figures.

def save_case_visuals(
    save_dir,
    image_name,
    gray8,
    bin_img,
    atrium_mask,
    catheter_overlay,
    tip_mask,
    tip_atrium_overlay,
    distance_overlay
):
    os.makedirs(save_dir, exist_ok=True)

    base = os.path.splitext(image_name)[0]

    cv2.imwrite(os.path.join(save_dir, f"{base}_01_gray.png"), gray8)
    cv2.imwrite(os.path.join(save_dir, f"{base}_02_bin.png"), bin_img)
    cv2.imwrite(os.path.join(save_dir, f"{base}_03_atrium.png"), atrium_mask)
    cv2.imwrite(os.path.join(save_dir, f"{base}_04_centerline_overlay.png"), catheter_overlay)
    cv2.imwrite(os.path.join(save_dir, f"{base}_05_tip_mask.png"), tip_mask)
    cv2.imwrite(os.path.join(save_dir, f"{base}_06_tip_atrium_overlay.png"), cv2.cvtColor(tip_atrium_overlay, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(save_dir, f"{base}_07_distance_overlay.png"), distance_overlay)

In [30]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 20 — REPLACE process_single_case_to_row
# ─────────────────────────────────────────────────────────────────────────────
# Two changes vs the previous version:
#
#   1. The new tip_point_and_depth_features(...) is called and merged into
#      the row, adding the 7 new point-level features described in cell 15b.
#
#   2. Nothing else changes — error handling, visual saving, and existing
#      region/direction/thickness features behave exactly as before.

def process_single_case_to_row(catheter_image_path, atrium_mask_path,
                               save_visuals=False, visuals_dir=None):
    image_name = os.path.basename(catheter_image_path)

    try:
        gray8 = load_grayscale_any(catheter_image_path)

        atrium_mask = load_binary_mask(
            atrium_mask_path,
            reference_shape=gray8.shape
        )

        print("gray8:", gray8.shape)
        print("atrium:", atrium_mask.shape)

        if gray8.shape != atrium_mask.shape:
            return {
                "image_name":    image_name,
                "catheter_path": catheter_image_path,
                "atrium_path":   atrium_mask_path,
                "status":        "shape_mismatch",
            }

        bin_img, dist, tip_path, full_path, tip_coord, status = \
            compute_centerline_fixed(gray8, atrium_mask)

        row = {
            "image_name":    image_name,
            "catheter_path": catheter_image_path,
            "atrium_path":   atrium_mask_path,
            "status":        status,
        }

        if tip_path is None or dist is None or tip_coord is None:
            return row

        tip_mask = rasterize_segment_mask(gray8.shape, tip_path, dist)

        # Existing overlap features
        overlap_pixels = int(np.sum((tip_mask > 0) & (atrium_mask > 0)))
        tip_pixels     = int(np.sum(tip_mask > 0))
        atrium_pixels  = int(np.sum(atrium_mask > 0))

        inside_ratio          = overlap_pixels / tip_pixels    if tip_pixels    > 0 else np.nan
        atrium_coverage_ratio = overlap_pixels / atrium_pixels if atrium_pixels > 0 else np.nan
        union_pixels          = np.sum((tip_mask > 0) | (atrium_mask > 0))
        iou                   = overlap_pixels / float(union_pixels) if union_pixels > 0 else np.nan

        tip_inside_atrium = bool(
            atrium_mask[int(round(tip_coord[0])), int(round(tip_coord[1]))] > 0
        )
        centerline_inside_ratio = path_inside_ratio(tip_path, atrium_mask)

        region_feats    = tip_region_to_atrium_distances(tip_mask, atrium_mask)
        direction_feats = tip_direction_features(tip_path)

        # NEW: point-level tip features (cell 15b)
        point_feats     = tip_point_and_depth_features(
            tip_path, atrium_mask, tip_coord
        )

        tip_length_measured = (
            cumulative_arclength(tip_path)[-1] if len(tip_path) > 1 else 0.0
        )
        full_tip_r  = mean_radius_on_path(full_path, dist, n=30, from_start=True)
        full_prox_r = mean_radius_on_path(full_path, dist, n=30, from_start=False)

        row.update({
            "tip_y":                   float(tip_coord[0]),
            "tip_x":                   float(tip_coord[1]),
            "tip_pixels":              tip_pixels,
            "atrium_pixels":           atrium_pixels,
            "overlap_pixels":          overlap_pixels,
            "tip_area_inside_ratio":   inside_ratio,
            "atrium_coverage_ratio":   atrium_coverage_ratio,
            "iou_tip_atrium":          iou,
            "tip_inside_atrium":       int(tip_inside_atrium),
            "centerline_inside_ratio": centerline_inside_ratio,
            "tip_dx":                  direction_feats["tip_dx"],
            "tip_dy":                  direction_feats["tip_dy"],
            "tip_angle":               direction_feats["tip_angle"],
            "tip_length_measured_px":  tip_length_measured,
            "mean_tip_radius":         full_tip_r,
            "mean_prox_radius":        full_prox_r,
        })

        if region_feats is not None:
            row.update(region_feats)
        # NEW: merge point-level features
        row.update(point_feats)

        if save_visuals:
            catheter_overlay   = make_overlay(gray8, full_path, tip_path, tip_coord)
            tip_atrium_overlay = make_tip_atrium_overlay(tip_mask, atrium_mask)
            distance_overlay   = cv2.cvtColor(gray8, cv2.COLOR_GRAY2BGR)

            save_case_visuals(
                save_dir=visuals_dir, image_name=image_name,
                gray8=gray8, bin_img=bin_img, atrium_mask=atrium_mask,
                catheter_overlay=catheter_overlay, tip_mask=tip_mask,
                tip_atrium_overlay=tip_atrium_overlay,
                distance_overlay=distance_overlay,
            )

        return row

    except Exception as e:
        return {
            "image_name":    image_name,
            "catheter_path": catheter_image_path,
            "atrium_path":   atrium_mask_path,
            "status":        f"error: {str(e)}",
        }

In [31]:
for img_name in ["IMG-0102", "IMG-0094", "IMG-0195"]:
    gray = load_grayscale_any(f"{CATHETER_FOLDER}/{img_name}-00001.png")
    inspect_skeleton_cycles(gray, f"{img_name} (false positive)")

for img_name in ["IMG-0027", "IMG-0047", "IMG-0055"]:
    gray = load_grayscale_any(f"{CATHETER_FOLDER}/{img_name}-00001.png")
    inspect_skeleton_cycles(gray, f"{img_name} (true loop)")

/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Para


=== IMG-0102 (false positive) ===
  Skeleton pixels:     734
  Endpoints:           0
  Junctions:           0
  Graph cycles (Euler):1
  Pixels in cycles:    734
  Cycle/skeleton ratio:1.000  (threshold = 0.1)
  → REAL LOOP

=== IMG-0094 (false positive) ===
  Skeleton pixels:     869
  Endpoints:           2
  Junctions:           0
  Graph cycles (Euler):0
  Pixels in cycles:    0
  Cycle/skeleton ratio:0.000  (threshold = 0.1)
  → no loop (no graph cycles)


/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Para


=== IMG-0195 (false positive) ===
  Skeleton pixels:     793
  Endpoints:           2
  Junctions:           0
  Graph cycles (Euler):0
  Pixels in cycles:    0
  Cycle/skeleton ratio:0.000  (threshold = 0.1)
  → no loop (no graph cycles)

=== IMG-0027 (true loop) ===
  Skeleton pixels:     886
  Endpoints:           2
  Junctions:           4
  Graph cycles (Euler):2
  Pixels in cycles:    616
  Cycle/skeleton ratio:0.695  (threshold = 0.1)
  → REAL LOOP


/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Para


=== IMG-0047 (true loop) ===
  Skeleton pixels:     700
  Endpoints:           0
  Junctions:           0
  Graph cycles (Euler):1
  Pixels in cycles:    700
  Cycle/skeleton ratio:1.000  (threshold = 0.1)
  → REAL LOOP

=== IMG-0055 (true loop) ===
  Skeleton pixels:     777
  Endpoints:           1
  Junctions:           3
  Graph cycles (Euler):2
  Pixels in cycles:    624
  Cycle/skeleton ratio:0.803  (threshold = 0.1)
  → REAL LOOP


In [32]:
def diagnose_status_v7(stem):
    cath_files = [f for f in os.listdir(CATHETER_FOLDER) if stem in f]
    atrium_files = [f for f in os.listdir(ATRIUM_FOLDER) if stem in f]
    if not cath_files or not atrium_files:
        print(f"\n=== {stem}: file not found ===")
        return

    image_path = os.path.join(CATHETER_FOLDER, cath_files[0])
    atrium_path = os.path.join(ATRIUM_FOLDER, atrium_files[0])

    # Call the actual pipeline function — this exercises whatever
    # version of compute_centerline_fixed is currently in scope
    gray8 = load_grayscale_any(image_path)
    atrium_mask = load_binary_mask(
    atrium_path,
    reference_shape=gray8.shape
    )

    bin_img, dist, tip_path, full_path, tip_coord, status = (
        compute_centerline_fixed(gray8, atrium_mask)
    )

    print(f"\n=== {stem} (via compute_centerline_fixed) ===")
    print(f"  status:    {status}")
    if tip_coord is not None:
        print(f"  tip_coord: ({int(tip_coord[0])}, {int(tip_coord[1])})")
    else:
        print(f"  tip_coord: None")

# Run it
for stem in ["IMG-0108", "IMG-0179", "IMG-0147", "IMG-0364", "IMG-0161", "IMG-0090"]:
    diagnose_status_v7(stem)

/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)


[strategy1] Protected 1 endpoints, using safe junction at (701, 433)
  [stub_filter] ep=(698, 430)  arm_len=901.9px
  [stub_filter] ep=(453, 435)  arm_len=901.9px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 2 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 2 endpoints

[DEBUG] Candidate ranking:
  coord=(698, 430) | thick=26.4 | min_path_dist=0.0 | tip_y=698 | overlap=572
  coord=(453, 435) | thick=13.1 | min_path_dist=6.7 | tip_y=453 | overlap=0
[DEBUG] Thinnest endpoint wins: (453, 435) (thick=13.1, next=26.4)

=== IMG-0108 (via compute_centerline_fixed) ===
  status:    success
  tip_coord: (453, 435)


/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the docum

  [centerline] original_endpoints before recovery: [(596, 557)]
[strategy1] Protected 1 endpoints, using safe junction at (424, 448)
  [stub_filter] ep=(420, 445)  arm_len=776.1px
  [stub_filter] ep=(422, 452)  arm_len=776.1px
  [stub_filter] ep=(596, 557)  arm_len=220.0px
  [stub_filter] ep=(429, 448)  arm_len=220.0px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 2 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 2 endpoints

[DEBUG] Candidate ranking:
  coord=(422, 452) | thick=17.1 | min_path_dist=0.0 | tip_y=422 | overlap=1146
  coord=(596, 557) | thick=29.9 | min_path_dist=0.0 | tip_y=596 | overlap=2696
[DEBUG] Thinnest endpoint wins: (422, 452) (thick=17.1, next=29.9)
[DEBUG] Thickness: chosen=17.1, next=29.9, ratio=0.57, gap=12.8
[DEBUG] Cut-point tip: touches atrium and gap=12.8 >= 2.0 → success_overlap_mode

=== IMG-0179 (via compute_centerline_fixed) ===
  status:    success_overlap_mode
  tip_coord: (422, 452)
  [centerline] original_endpoin

/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the docum

  [stub_filter] ep=(296, 276)  arm_len=938.4px
  [stub_filter] ep=(287, 277)  arm_len=938.4px
  [stub_filter] after dedup: 1 endpoints
  [stub_filter] final: 1 endpoints
[loop_recovery_v3] All strategies failed — using geometric fallback

[DEBUG] Candidate ranking:
  coord=(293, 365) | thick=14.0 | min_path_dist=21.9 | tip_y=293 | overlap=0
  coord=(503, 273) | thick=15.2 | min_path_dist=0.0 | tip_y=503 | overlap=2113
[DEBUG] Thinnest endpoint wins: (293, 365) (thick=14.0, next=15.2)
[DEBUG] Loop recovery used geometric fallback → ambiguous_tip

=== IMG-0147 (via compute_centerline_fixed) ===
  status:    ambiguous_tip
  tip_coord: (293, 365)


/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the docum

  [centerline] original_endpoints before recovery: [(491, 432), (514, 515)]
[strategy1] Protected 2 endpoints, using safe junction at (410, 430)
  [stub_filter] ep=(491, 432)  arm_len=86.5px
  [stub_filter] ep=(414, 427)  arm_len=86.5px
  [stub_filter] ep=(411, 434)  arm_len=139.1px
  [stub_filter] ep=(514, 515)  arm_len=139.1px
  [stub_filter] ep=(406, 428)  arm_len=37.6px
  [stub_filter] after dedup: 3 endpoints
  [stub_filter] final: 3 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 3 endpoints
[WARNING] Path length 77.9px < target 200.0px — returning full path
[WARNING] Path length 135.1px < target 200.0px — returning full path
[WARNING] Path length 135.1px < target 200.0px — returning full path

[DEBUG] Candidate ranking:
  coord=(491, 432) | thick=13.6 | min_path_dist=0.0 | tip_y=491 | overlap=1456
  coord=(411, 434) | thick=16.5 | min_path_dist=0.0 | tip_y=411 | overlap=1074
  coord=(514, 515) | thick=30.0 | min_path_dist=0.0 | tip_y=514 | overlap=1074
[DEBUG] T

/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, dis


[DEBUG] Candidate ranking:
  coord=(336, 413) | thick=14.2 | min_path_dist=35.1 | tip_y=336 | overlap=0
  coord=(604, 473) | thick=27.2 | min_path_dist=0.0 | tip_y=604 | overlap=3035
[DEBUG] Thinnest endpoint wins: (336, 413) (thick=14.2, next=27.2)

=== IMG-0161 (via compute_centerline_fixed) ===
  status:    success
  tip_coord: (336, 413)

[DEBUG] Candidate ranking:
  coord=(583, 416) | thick=22.7 | min_path_dist=7.1 | tip_y=583 | overlap=95
  coord=(459, 452) | thick=10.5 | min_path_dist=22.4 | tip_y=459 | overlap=0
[DEBUG] Thinnest endpoint wins: (459, 452) (thick=10.5, next=22.7)

=== IMG-0090 (via compute_centerline_fixed) ===
  status:    success
  tip_coord: (459, 452)


In [33]:
# ─────────────────────────────────────────────────────────────────────────────
# SPOT-CHECK GRID — visual regression check after tip-selection v9 patches
# ─────────────────────────────────────────────────────────────────────────────
# Purpose: before running the full pipeline, eyeball whether the v9 patches
#   (1) fixed the known failure cases, and
#   (2) did not regress previously-passing cases.

import os
import random
import matplotlib.pyplot as plt


# Ensure `pairs` exists. If you've already run the full batch earlier
# in this session it will already be defined and this is a no-op.
# Otherwise, build it now so the spot-check cell is self-contained.
try:
    pairs
    print(f"Using existing pairs: {len(pairs)} catheter/atrium pairs")
except NameError:
    pairs, missing_atrium = get_image_pairs(CATHETER_FOLDER, ATRIUM_FOLDER)
    print(f"Built pairs: {len(pairs)} matched, "
          f"{len(missing_atrium)} catheter images had no matching atrium mask")


def _find_pair_by_stem(stem, pairs):
    """Return (catheter_path, atrium_path) for the given image stem,
    or (None, None) if no match is found."""
    for cpath, apath in pairs:
        if os.path.splitext(os.path.basename(cpath))[0] == stem:
            return cpath, apath
    return None, None


def spot_check_grid(image_stems, pairs, ncols=3, figsize_per_cell=(5, 5),
                    suptitle=None, crop_to_catheter=True, crop_pad=80):
    """
    Render centerline overlays for a list of image stems in a grid.
    """
    n = len(image_stems)
    if n == 0:
        print("No image stems supplied.")
        return

    nrows = (n + ncols - 1) // ncols
    fig_w = figsize_per_cell[0] * ncols
    fig_h = figsize_per_cell[1] * nrows
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h))

    if nrows == 1 and ncols == 1:
        axes = [axes]
    else:
        axes = list(axes.flatten())

    for idx, stem in enumerate(image_stems):
        ax = axes[idx]
        cpath, apath = _find_pair_by_stem(stem, pairs)

        if cpath is None:
            ax.set_title(f"{stem}\n(file not found)", fontsize=10, color="red")
            ax.axis("off")
            continue

        try:
            gray8 = load_grayscale_any(cpath)

            atrium_mask = load_binary_mask(
                apath,
                reference_shape=gray8.shape
            )

            print("gray8:", gray8.shape)
            print("atrium:", atrium_mask.shape)

            if gray8.shape != atrium_mask.shape:
                ax.set_title(f"{stem}\n(shape mismatch)", fontsize=10, color="red")
                ax.axis("off")
                continue

            bin_img, dist, tip_path, full_path, tip_coord, status = (
                compute_centerline_fixed(gray8, atrium_mask)
            )

            if tip_coord is None:
                ax.set_title(f"{stem}\nstatus={status}\n(no tip)",
                             fontsize=9, color="red")
                ax.imshow(gray8, cmap="gray")
                ax.axis("off")
                continue

            overlay = make_overlay(gray8, full_path, tip_path, tip_coord)
            overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

            if crop_to_catheter and bin_img is not None and bin_img.any():
                ys, xs = np.where(bin_img > 0)
                y0 = max(0, ys.min() - crop_pad)
                y1 = min(bin_img.shape[0], ys.max() + crop_pad)
                x0 = max(0, xs.min() - crop_pad)
                x1 = min(bin_img.shape[1], xs.max() + crop_pad)
                overlay_rgb = overlay_rgb[y0:y1, x0:x1]

            ax.imshow(overlay_rgb)
            title = (f"{stem}\nstatus={status}\n"
                     f"tip=({tip_coord[0]}, {tip_coord[1]})")
            ax.set_title(title, fontsize=9)
            ax.axis("off")

        except Exception as e:
            ax.set_title(f"{stem}\n(error: {type(e).__name__})",
                         fontsize=9, color="red")
            ax.axis("off")

    for k in range(n, len(axes)):
        axes[k].axis("off")

    if suptitle is not None:
        fig.suptitle(suptitle, fontsize=14, y=1.00)

    plt.tight_layout()
    plt.show()
    plt.close()


# ──────────────────────────────────────────────────────────────────────
# Batch 1: the six known-failure cases.
# ──────────────────────────────────────────────────────────────────────
failure_stems = [
    "IMG-0108-00001", "IMG-0179-00001", "IMG-0147-00001",
    "IMG-0364-00001", "IMG-0161-00001", "IMG-0090-00001",
]
spot_check_grid(
    failure_stems, pairs,
    ncols=3,
    suptitle="Spot-check: known-failure cases (post v9)"
)


# ──────────────────────────────────────────────────────────────────────
# Batch 2: random sample for regression check.
# Picks 15 random stems from the full pair list. Eyeball each tip dot.
# ──────────────────────────────────────────────────────────────────────
random.seed(42)
all_stems = [os.path.splitext(os.path.basename(c))[0] for c, _ in pairs]
# Exclude the six failure cases so we get fresh samples.
candidates = [s for s in all_stems if s not in set(failure_stems)]
random_15 = random.sample(candidates, min(15, len(candidates)))

spot_check_grid(
    random_15, pairs,
    ncols=3,
    suptitle="Spot-check: 15 random images (regression check)"
)

Built pairs: 220 matched, 0 catheter images had no matching atrium mask
gray8: (960, 960)
atrium: (960, 960)


/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)


[strategy1] Protected 1 endpoints, using safe junction at (701, 433)
  [stub_filter] ep=(698, 430)  arm_len=898.7px
  [stub_filter] ep=(450, 434)  arm_len=898.7px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 2 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 2 endpoints

[DEBUG] Candidate ranking:
  coord=(698, 430) | thick=26.4 | min_path_dist=0.0 | tip_y=698 | overlap=571
  coord=(450, 434) | thick=13.5 | min_path_dist=7.6 | tip_y=450 | overlap=0
[DEBUG] Thinnest endpoint wins: (450, 434) (thick=13.5, next=26.4)
gray8: (960, 960)
atrium: (960, 960)


/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the docum

  [centerline] original_endpoints before recovery: [(596, 557)]
[strategy1] Protected 1 endpoints, using safe junction at (424, 448)
  [stub_filter] ep=(429, 447)  arm_len=219.6px
  [stub_filter] ep=(420, 445)  arm_len=774.7px
  [stub_filter] ep=(422, 452)  arm_len=774.7px
  [stub_filter] ep=(596, 557)  arm_len=219.6px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 2 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 2 endpoints

[DEBUG] Candidate ranking:
  coord=(422, 452) | thick=17.1 | min_path_dist=0.0 | tip_y=422 | overlap=1148
  coord=(596, 557) | thick=29.9 | min_path_dist=0.0 | tip_y=596 | overlap=2703
[DEBUG] Thinnest endpoint wins: (422, 452) (thick=17.1, next=29.9)
[DEBUG] Thickness: chosen=17.1, next=29.9, ratio=0.57, gap=12.8
[DEBUG] Cut-point tip: touches atrium and gap=12.8 >= 2.0 → success_overlap_mode
gray8: (960, 960)
atrium: (960, 960)


...

/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the docum


[DEBUG] Candidate ranking:
  coord=(434, 441) | thick=16.5 | min_path_dist=30.0 | tip_y=434 | overlap=0
  coord=(859, 444) | thick=32.3 | min_path_dist=0.0 | tip_y=859 | overlap=387
[DEBUG] Thinnest endpoint wins: (434, 441) (thick=16.5, next=32.3)
gray8: (960, 960)
atrium: (960, 960)

[DEBUG] Candidate ranking:
  coord=(509, 403) | thick=16.5 | min_path_dist=0.0 | tip_y=509 | overlap=789
  coord=(688, 460) | thick=29.4 | min_path_dist=0.0 | tip_y=688 | overlap=4274
[DEBUG] Thinnest endpoint wins: (509, 403) (thick=16.5, next=29.4)
gray8: (960, 960)
atrium: (960, 960)

[DEBUG] Candidate ranking:
  coord=(451, 426) | thick=13.0 | min_path_dist=0.0 | tip_y=451 | overlap=692
  coord=(405, 355) | thick=29.6 | min_path_dist=40.0 | tip_y=405 | overlap=0
[DEBUG] Thinnest endpoint wins: (451, 426) (thick=13.0, next=29.6)
gray8: (960, 960)
atrium: (960, 960)


/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.


[DEBUG] Candidate ranking:
  coord=(563, 462) | thick=27.8 | min_path_dist=0.0 | tip_y=563 | overlap=1591
  coord=(471, 460) | thick=13.9 | min_path_dist=4.0 | tip_y=471 | overlap=13
[DEBUG] Thinnest endpoint wins: (471, 460) (thick=13.9, next=27.8)
gray8: (960, 960)
atrium: (960, 960)
[strategy1] Protected 1 endpoints, using safe junction at (459, 356)
  [stub_filter] ep=(510, 478)  arm_len=817.0px
  [stub_filter] ep=(454, 356)  arm_len=817.0px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 2 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 2 endpoints

[DEBUG] Candidate ranking:
  coord=(510, 478) | thick=11.8 | min_path_dist=0.0 | tip_y=510 | overlap=1482
  coord=(454, 356) | thick=24.7 | min_path_dist=76.7 | tip_y=454 | overlap=0
[DEBUG] Thinnest endpoint wins: (510, 478) (thick=11.8, next=24.7)


/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255


In [34]:
spot_check_grid(
    ["IMG-0069-00001.dcm - PRIMARY", "IMG-0055-00001", "IMG-0102-00001"
     "IMG-0147-00001", "IMG-0179-00001", "IMG-0213-00001", "IMG-0257-00001"
     "IMG-0328-00001", "IMG-0364-00001", "IMG-0366-00001", "IMG-0448-00001"],
    pairs,
    ncols=3,
    suptitle="Spot-check: 6 known-failure cases (post v9)"
)

gray8: (960, 960)
atrium: (960, 960)

[DEBUG] Candidate ranking:
  coord=(656, 117) | thick=23.9 | min_path_dist=150.6 | tip_y=656 | overlap=0
  coord=(674, 370) | thick=12.3 | min_path_dist=0.0 | tip_y=674 | overlap=2043
[DEBUG] Thinnest endpoint wins: (674, 370) (thick=12.3, next=23.9)
gray8: (960, 960)
atrium: (960, 960)


/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.

  [centerline] original_endpoints before recovery: [(572, 379)]
[strategy1] Protected 1 endpoints, using safe junction at (419, 379)
  [stub_filter] ep=(572, 379)  arm_len=167.6px
  [stub_filter] ep=(423, 380)  arm_len=167.6px
  [stub_filter] ep=(415, 381)  arm_len=711.1px
  [stub_filter] ep=(416, 375)  arm_len=711.1px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 2 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 2 endpoints
[WARNING] Path length 156.4px < target 200.0px — returning full path

[DEBUG] Candidate ranking:
  coord=(572, 379) | thick=13.8 | min_path_dist=0.0 | tip_y=572 | overlap=2259
  coord=(415, 381) | thick=18.4 | min_path_dist=29.1 | tip_y=415 | overlap=0
[DEBUG] Thinnest endpoint wins: (572, 379) (thick=13.8, next=18.4)
[WARNING] Path length 156.4px < target 200.0px — returning full path
[DEBUG] Thickness: chosen=13.8, next=18.4, ratio=0.75, gap=4.5
[DEBUG] Real free endpoint, gap=4.5 < 5.0 → success
gray8: (960, 960)
atrium: (960, 

/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)


  [centerline] original_endpoints before recovery: [(595, 551)]
[strategy1] Protected 1 endpoints, using safe junction at (424, 448)
  [stub_filter] ep=(420, 445)  arm_len=775.6px
  [stub_filter] ep=(595, 551)  arm_len=213.6px
  [stub_filter] ep=(422, 452)  arm_len=775.6px
  [stub_filter] ep=(429, 448)  arm_len=213.6px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 2 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 2 endpoints

[DEBUG] Candidate ranking:
  coord=(422, 452) | thick=17.1 | min_path_dist=0.0 | tip_y=422 | overlap=1146
  coord=(595, 551) | thick=31.6 | min_path_dist=0.0 | tip_y=595 | overlap=2795
[DEBUG] Thinnest endpoint wins: (422, 452) (thick=17.1, next=31.6)
[DEBUG] Thickness: chosen=17.1, next=31.6, ratio=0.54, gap=14.5
[DEBUG] Cut-point tip: touches atrium and gap=14.5 >= 2.0 → success_overlap_mode
gray8: (960, 960)
atrium: (960, 960)
  [centerline] original_endpoints before recovery: []
[strategy1] Protected 0 endpoints, using safe j

...

/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, dis

  [centerline] original_endpoints before recovery: [(558, 449)]
[strategy1] Protected 1 endpoints, using safe junction at (482, 433)
  [stub_filter] ep=(558, 449)  arm_len=210.8px
  [stub_filter] ep=(479, 436)  arm_len=128.7px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 2 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 2 endpoints

[DEBUG] Candidate ranking:
  coord=(558, 449) | thick=14.6 | min_path_dist=0.0 | tip_y=558 | overlap=3144
  coord=(479, 436) | thick=26.2 | min_path_dist=0.0 | tip_y=479 | overlap=786
[DEBUG] Thinnest endpoint wins: (558, 449) (thick=14.6, next=26.2)
[DEBUG] Thickness: chosen=14.6, next=26.2, ratio=0.56, gap=11.6
[DEBUG] Real free endpoint, gap=11.6 >= 5.0 → success_overlap_mode
gray8: (960, 960)
atrium: (960, 960)


/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the docum

  [centerline] original_endpoints before recovery: [(544, 465)]
[strategy1] Protected 1 endpoints, using safe junction at (454, 432)
  [stub_filter] ep=(451, 435)  arm_len=765.5px
  [stub_filter] ep=(451, 428)  arm_len=765.5px
  [stub_filter] ep=(544, 465)  arm_len=98.8px
  [stub_filter] ep=(458, 434)  arm_len=98.8px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 2 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 2 endpoints
[WARNING] Path length 93.0px < target 200.0px — returning full path

[DEBUG] Candidate ranking:
  coord=(451, 435) | thick=16.2 | min_path_dist=0.0 | tip_y=451 | overlap=608
  coord=(544, 465) | thick=28.7 | min_path_dist=0.0 | tip_y=544 | overlap=3094
[DEBUG] Thinnest endpoint wins: (451, 435) (thick=16.2, next=28.7)
[DEBUG] Thickness: chosen=16.2, next=28.7, ratio=0.56, gap=12.5
[DEBUG] Cut-point tip: touches atrium and gap=12.5 >= 2.0 → success_overlap_mode


/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255


In [35]:
spot_check_grid(
    ["IMG-0069-00001.dcm - PRIMARY", "IMG-0055-00001",
     "IMG-0147-00001", "IMG-0179-00001", "IMG-0257-00001"
     "IMG-0328-00001", "IMG-0364-00001", "IMG-0366-00001", "IMG-0448-00001"
     "IMG-0102-00001", "IMG-0129-00001", "IMG-0213-00001"],
    pairs,
    ncols=3,
    suptitle="Spot-check: 6 known-failure cases (post v9)"
)

gray8: (960, 960)
atrium: (960, 960)


/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)



[DEBUG] Candidate ranking:
  coord=(657, 116) | thick=23.8 | min_path_dist=150.6 | tip_y=657 | overlap=0
  coord=(674, 370) | thick=12.3 | min_path_dist=0.0 | tip_y=674 | overlap=2034
[DEBUG] Thinnest endpoint wins: (674, 370) (thick=12.3, next=23.8)
gray8: (960, 960)
atrium: (960, 960)


/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the docum

  [centerline] original_endpoints before recovery: [(572, 379)]
[strategy1] Protected 1 endpoints, using safe junction at (419, 379)
  [stub_filter] ep=(572, 379)  arm_len=167.6px
  [stub_filter] ep=(423, 380)  arm_len=167.6px
  [stub_filter] ep=(415, 381)  arm_len=710.3px
  [stub_filter] ep=(416, 375)  arm_len=710.3px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 2 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 2 endpoints
[WARNING] Path length 156.3px < target 200.0px — returning full path

[DEBUG] Candidate ranking:
  coord=(572, 379) | thick=13.8 | min_path_dist=0.0 | tip_y=572 | overlap=2258
  coord=(415, 381) | thick=18.4 | min_path_dist=29.1 | tip_y=415 | overlap=0
[DEBUG] Thinnest endpoint wins: (572, 379) (thick=13.8, next=18.4)
[WARNING] Path length 156.3px < target 200.0px — returning full path
[DEBUG] Thickness: chosen=13.8, next=18.4, ratio=0.75, gap=4.5
[DEBUG] Real free endpoint, gap=4.5 < 5.0 → success
gray8: (960, 960)
atrium: (960, 

...

/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, dis

  [centerline] original_endpoints before recovery: [(558, 449), (484, 427)]
[strategy1] Protected 2 endpoints, using safe junction at (375, 392)
  [stub_filter] ep=(371, 391)  arm_len=71.6px
  [stub_filter] ep=(558, 449)  arm_len=207.2px
  [stub_filter] ep=(379, 389)  arm_len=207.2px
  [stub_filter] ep=(484, 427)  arm_len=133.5px
  [stub_filter] ep=(377, 396)  arm_len=133.5px
  [stub_filter] after dedup: 3 endpoints
  [stub_filter] final: 3 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 3 endpoints
[WARNING] Path length 191.8px < target 200.0px — returning full path
[WARNING] Path length 191.8px < target 200.0px — returning full path
[WARNING] Path length 125.9px < target 200.0px — returning full path

[DEBUG] Candidate ranking:
  coord=(379, 389) | thick=16.3 | min_path_dist=0.0 | tip_y=379 | overlap=3099
  coord=(558, 449) | thick=14.6 | min_path_dist=0.0 | tip_y=558 | overlap=3099
  coord=(484, 427) | thick=20.5 | min_path_dist=0.0 | tip_y=484 | overlap=636
[DEBUG]

/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, dis


[DEBUG] Candidate ranking:
  coord=(504, 362) | thick=25.9 | min_path_dist=30.4 | tip_y=504 | overlap=0
  coord=(510, 413) | thick=13.2 | min_path_dist=0.0 | tip_y=510 | overlap=1016
[DEBUG] Thinnest endpoint wins: (510, 413) (thick=13.2, next=25.9)
gray8: (960, 960)
atrium: (960, 960)
  [centerline] original_endpoints before recovery: [(538, 389)]
[strategy1] Protected 1 endpoints, using safe junction at (495, 385)
  [stub_filter] ep=(494, 381)  arm_len=683.0px
  [stub_filter] ep=(538, 389)  arm_len=43.1px
  [stub_filter] ep=(499, 387)  arm_len=43.1px
  [stub_filter] ep=(491, 387)  arm_len=683.0px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 1 endpoints
  [stub_filter] ep=(494, 381)  arm_len=683.0px
  [stub_filter] ep=(538, 389)  arm_len=43.1px
  [stub_filter] ep=(499, 387)  arm_len=43.1px
  [stub_filter] ep=(491, 387)  arm_len=683.0px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 1 endpoints
  [stub_filter] ep=(502, 388)  arm_len=39.7px
  [stub_f

/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255


In [36]:
diagnose_status_v7("IMG-0235")

/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)


  [centerline] original_endpoints before recovery: [(667, 457)]
[strategy1] Protected 1 endpoints, using safe junction at (407, 450)
  [stub_filter] ep=(411, 453)  arm_len=276.7px
  [stub_filter] ep=(667, 457)  arm_len=276.7px
  [stub_filter] ep=(407, 445)  arm_len=550.2px
  [stub_filter] ep=(403, 452)  arm_len=550.2px
  [stub_filter] after dedup: 2 endpoints
  [stub_filter] final: 2 endpoints
[loop_recovery_v3] Strategy 'junction_cut' succeeded → 2 endpoints

[DEBUG] Candidate ranking:
  coord=(403, 452) | thick=11.8 | min_path_dist=90.2 | tip_y=403 | overlap=0
  coord=(667, 457) | thick=9.3 | min_path_dist=0.0 | tip_y=667 | overlap=1835
[DEBUG] Thinnest endpoint wins: (667, 457) (thick=9.3, next=11.8)
[DEBUG] Thickness: chosen=9.3, next=11.8, ratio=0.79, gap=2.5
[DEBUG] Real free endpoint, gap=2.5 < 5.0 → success

=== IMG-0235 (via compute_centerline_fixed) ===
  status:    success
  tip_coord: (667, 457)


/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# UPSTREAM DIAGNOSTIC (v2) — substring matching so you don't need exact stems
# ─────────────────────────────────────────────────────────────────────────────

def diagnose_image(query, pairs):
    """
    `query` can be any substring of the catheter filename — e.g. "0069",
    "IMG-0069", or the full stem. The first matching pair is diagnosed.
    """
    matches = [(c, a) for c, a in pairs if query in os.path.basename(c)]
    if not matches:
        print(f"!! No pair matches '{query}'")
        print("   First 5 catheter filenames in pairs (for reference):")
        for c, _ in pairs[:5]:
            print(f"     {os.path.basename(c)}")
        return
    if len(matches) > 1:
        print(f"!! '{query}' matched {len(matches)} pairs. Using the first:")
        for c, _ in matches[:3]:
            print(f"     {os.path.basename(c)}")
        print()

    cpath, apath = matches[0]
    stem = os.path.splitext(os.path.basename(cpath))[0]
    print(f"=== Diagnosing {stem} ===")
    print(f"  catheter: {cpath}")
    print(f"  atrium:   {apath}\n")

    gray8 = load_grayscale_any(cpath)
    atrium_mask = load_binary_mask(apath)
    print(f"  image shape: {gray8.shape}")
    print(f"  atrium pixels: {int(np.sum(atrium_mask > 0))}\n")

    bin_img = binarize_catheter(gray8)
    mask_bool = bin_img.astype(bool)
    print(f"  [binarize] catheter mask pixels: {int(mask_bool.sum())}")

    skel_bool, dist = build_skeleton_and_dist(mask_bool)
    print(f"  [skeletonize] skeleton pixels (before pruning): {int(skel_bool.sum())}")

    eps_before, _ = detect_all_endpoints(skel_bool)
    print(f"\n  [endpoints BEFORE spur pruning] count={len(eps_before)}")
    for ep in eps_before:
        thick = 2.0 * float(dist[ep[0], ep[1]])
        print(f"    {ep}  thick≈{thick:.1f}")

    skel_bool_pruned = prune_short_spurs(
        skel_bool, dist, max_len=SPUR_MAX_LEN, dt_ratio=SPUR_DT_RATIO
    )
    print(f"\n  [after spur prune] skeleton pixels: {int(skel_bool_pruned.sum())}")
    print(f"  (SPUR_MAX_LEN={SPUR_MAX_LEN}, SPUR_DT_RATIO={SPUR_DT_RATIO})")

    eps_after_prune, pts_set = detect_all_endpoints(skel_bool_pruned)
    print(f"\n  [endpoints AFTER spur pruning] count={len(eps_after_prune)}")
    for ep in eps_after_prune:
        thick = 2.0 * float(dist[ep[0], ep[1]])
        print(f"    {ep}  thick≈{thick:.1f}")

    had_loop = has_real_loop(skel_bool_pruned)
    print(f"\n  [loop detection] has_real_loop = {had_loop}")

    junctions, _ = detect_junction_points(skel_bool_pruned)
    print(f"  [junctions detected] count={len(junctions)}")
    for j in junctions[:5]:
        print(f"    {j}")
    if len(junctions) > 5:
        print(f"    ... and {len(junctions)-5} more")

    print("\n  [running full compute_centerline_fixed]")
    bin_img2, dist2, tip_path, full_path, tip_coord, status = (
        compute_centerline_fixed(gray8, atrium_mask)
    )
    print(f"\n  status:    {status}")
    print(f"  tip_coord: {tip_coord}")
    if full_path is not None:
        print(f"  full_path length: {len(full_path)}")
        print(f"  full_path first point: {full_path[0]}")
        print(f"  full_path last point:  {full_path[-1]}")


# Run it
diagnose_image("0069", pairs)

In [37]:
# RUN ALL CASES

pairs, missing_atrium = get_image_pairs(CATHETER_FOLDER, ATRIUM_FOLDER)

print(f"Matched pairs: {len(pairs)}")
print(f"Missing atrium masks: {len(missing_atrium)}")

VISUALS_DIR = os.path.join(OUTPUT_FOLDER, "debug_visuals")
SAVE_VISUALS = True   # set False if you only want CSV and faster processing

all_rows = []

for catheter_path, atrium_path in tqdm(pairs):
    row = process_single_case_to_row(
        catheter_image_path=catheter_path,
        atrium_mask_path=atrium_path,
        save_visuals=SAVE_VISUALS,
        visuals_dir=VISUALS_DIR
    )
    all_rows.append(row)

df = pd.DataFrame(all_rows)
print(df.shape)
df.head()

Matched pairs: 220
Missing atrium masks: 0


  0%|          | 0/220 [00:00<?, ?it/s]

gray8: (960, 960)
atrium: (960, 960)


/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(np.uint8) * 255



[DEBUG] Candidate ranking:
  coord=(397, 424) | thick=13.8 | min_path_dist=11.0 | tip_y=397 | overlap=0
  coord=(490, 293) | thick=26.9 | min_path_dist=82.6 | tip_y=490 | overlap=0
[DEBUG] Thinnest endpoint wins: (397, 424) (thick=13.8, next=26.9)


...

100%|█████████▉| 219/220 [01:18<00:00,  3.07it/s]/tmp/ipykernel_2502516/3821978093.py:45: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  bin_bool = binary_closing(bin_bool, disk(4))
/tmp/ipykernel_2502516/3821978093.py:46: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  bin_bool = remove_small_objects(bin_bool, 50)
/tmp/ipykernel_2502516/2530521301.py:41: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  out = binary_closing(out.astype(bool), disk(1)).astype(n

gray8: (960, 960)
atrium: (960, 960)

[DEBUG] Candidate ranking:
  coord=(418, 699) | thick=33.4 | min_path_dist=281.7 | tip_y=418 | overlap=0
  coord=(576, 364) | thick=10.3 | min_path_dist=0.0 | tip_y=576 | overlap=631
[DEBUG] Thinnest endpoint wins: (576, 364) (thick=10.3, next=33.4)


100%|██████████| 220/220 [01:18<00:00,  2.81it/s]

(220, 66)


,image_name,catheter_path,atrium_path,status,tip_y,tip_x,tip_pixels,atrium_pixels,overlap_pixels,tip_area_inside_ratio,...,edge_dx_right_min_norm,edge_dx_right_mean_norm,region_nearest_dist_norm_h,region_nearest_dist_norm_w,tip_depth_in_atrium_px,tip_depth_in_atrium_norm,tip_y_norm_to_atrium,tip_x_norm_to_atrium,distance_from_optimal_depth,tip_to_atrium_dist_norm
0,IMG-0023-00001.png,/vsc-hard-mounts/leuven-data/375/vsc37504/Mast...,/vsc-hard-mounts/leuven-data/375/vsc37504/Mast...,success,397.0,424.0,3079,22722,0,0.000000,...,NaN,NaN,0.019685,0.036765,0.000000,0.000000,-0.043307,0.441176,-0.376640,0.043307
1,IMG-0025-00001.png,/vsc-hard-mounts/leuven-data/375/vsc37504/Mast...,/vsc-hard-mounts/leuven-data/375/vsc37504/Mast...,success,555.0,434.0,3154,18486,888,0.281547,...,0.029412,0.261438,0.000000,0.000000,54.118063,0.226435,0.242678,0.500000,-0.090656,0.000000
2,IMG-0027-00001.png,/vsc-hard-mounts/leuven-data/375/vsc37504/Mast...,/vsc-hard-mounts/leuven-data/375/vsc37504/Mast...,success,656.0,443.0,3265,23147,3265,1.000000,...,0.041667,0.074109,0.000000,0.000000,113.449224,0.476677,0.567227,0.756944,0.067227,0.000000
3,IMG-0029-00001.png,/vsc-hard-mounts/leuven-data/375/vsc37504/Mast...,/vsc-hard-mounts/leuven-data/375/vsc37504/Mast...,success,476.0,433.0,2814,16336,668,0.237385,...,-0.121951,0.020886,0.000000,0.000000,44.083576,0.221526,0.256281,0.674797,-0.077052,0.000000
4,IMG-0031-00001.png,/vsc-hard-mounts/leuven-data/375/vsc37504/Mast...,/vsc-hard-mounts/leuven-data/375/vsc37504/Mast...,success,521.0,449.0,3353,20162,2835,0.845511,...,-0.056962,0.334499,0.000000,0.000000,0.000000,0.000000,0.983333,0.689873,0.483333,0.005556


In [38]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 23 — REPLACE THE EXISTING QUICK QA CELL
# ─────────────────────────────────────────────────────────────────────────────
# After the status-assignment fix, you should now see a non-zero count of
# success_overlap_mode cases (loop catheters that were recovered by cutting
# the narrowest point). If this number is still 0, see the diagnostic at
# the bottom of this cell.

print("=" * 60)
print("STATUS DISTRIBUTION")
print("=" * 60)
print(df["status"].value_counts(dropna=False))
print()

print("=" * 60)
print("SANITY CHECK — what each status should look like:")
print("=" * 60)
print("  success              → catheter was a clean open curve")
print("  success_overlap_mode → catheter formed a loop, narrowest-point cut")
print("                         produced 2 endpoints and a tip was found")
print("  ambiguous_tip        → loop recovery failed OR thickness anomaly")
print("                         without atrium-proximity confirmation")
print()

# Modelling set: now includes success_overlap_mode by default since these
# are also correctly recovered cases. Decide explicitly which to keep.
usable_status = ["success", "success_overlap_mode", "ambiguous_tip"]
success_df    = df[df["status"].isin(usable_status)].copy()
print(f"Usable rows (success + success_overlap_mode): {len(success_df)}")
print(f"Total rows: {len(df)}")
print(f"Excluded as ambiguous_tip / errors:           "
      f"{len(df) - len(success_df)}")

# Diagnostic if success_overlap_mode is still 0 after the fix
overlap_count = int((df["status"] == "success_overlap_mode").sum())
if overlap_count == 0:
    print()
    print("[DIAGNOSTIC] success_overlap_mode count is 0.")
    print("  Either there are genuinely no loop catheters in the dataset,")
    print("  or the spur pruning is flattening loops before junction")
    print("  detection runs. To check: lower SPUR_MAX_LEN from 40 to 20")
    print("  and rerun. If overlap_count goes up, pruning was the cause.")

STATUS DISTRIBUTION
status
success                 197
ambiguous_tip            20
success_overlap_mode      3
Name: count, dtype: int64

SANITY CHECK — what each status should look like:
  success              → catheter was a clean open curve
  success_overlap_mode → catheter formed a loop, narrowest-point cut
                         produced 2 endpoints and a tip was found
  ambiguous_tip        → loop recovery failed OR thickness anomaly
                         without atrium-proximity confirmation

Usable rows (success + success_overlap_mode): 220
Total rows: 220
Excluded as ambiguous_tip / errors:           0


In [39]:
# PLOT CENTERLINE/TIP OVERLAYS — ONE PLOT PER STATUS

import os
import cv2
import matplotlib.pyplot as plt

def plot_status_overlays(status, samples_per_row=5, max_samples=10):
    subset = df[df["status"] == status].head(max_samples)

    if len(subset) == 0:
        print(f"No images found for status: {status}")
        return

    n_cols = min(samples_per_row, len(subset))
    n_rows = (len(subset) + n_cols - 1) // n_cols  # ceiling division

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(4 * n_cols, 4 * n_rows))

    # Always make axes a 2D array for consistent indexing
    if n_rows == 1 and n_cols == 1:
        axes = [[axes]]
    elif n_rows == 1:
        axes = [axes]
    elif n_cols == 1:
        axes = [[ax] for ax in axes]

    fig.suptitle(f"Status: {status}  ({len(subset)} samples)",
                 fontsize=13, fontweight="normal", y=1.01)

    for idx, (_, row) in enumerate(subset.iterrows()):
        r = idx // n_cols
        c = idx % n_cols
        ax = axes[r][c]

        base = os.path.splitext(row["image_name"])[0]
        overlay_path = os.path.join(
            VISUALS_DIR,
            f"{base}_04_centerline_overlay.png"
        )
        img = cv2.imread(overlay_path)
        if img is None:
            ax.text(0.5, 0.5, "Not found", ha="center", va="center",
                    fontsize=9, color="gray")
            ax.axis("off")
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(row["image_name"], fontsize=8)
            ax.axis("off")

    # Turn off any unused axes
    for idx in range(len(subset), n_rows * n_cols):
        r = idx // n_cols
        c = idx % n_cols
        axes[r][c].axis("off")

    plt.tight_layout()
    plt.savefig(f"overlays_{status}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: overlays_{status}.png")
    print(f"Total {status} cases: {len(df[df['status'] == status])}")


# Run each status separately
plot_status_overlays("success",              max_samples=10)
plot_status_overlays("ambiguous_tip",        max_samples=10)
plot_status_overlays("success_overlap_mode", max_samples=10)

Saved: overlays_success.png
Total success cases: 197
Saved: overlays_ambiguous_tip.png
Total ambiguous_tip cases: 20
Saved: overlays_success_overlap_mode.png
Total success_overlap_mode cases: 3


In [40]:
import shutil, os
panel_dir = os.path.join(OUTPUT_FOLDER, "panels")
if os.path.exists(panel_dir):
    shutil.rmtree(panel_dir)

# Also clear any per-status subfolders the panel cell writes into
for status_name in ["success", "success_overlap_mode", "ambiguous_tip"]:
    sub = os.path.join(OUTPUT_FOLDER, status_name)
    if os.path.exists(sub):
        shutil.rmtree(sub)

# Then rerun the panel-generation cell

In [41]:
# Print the current status of each image in your panel
panel_images = [
    "IMG-0023", "IMG-0027", "IMG-0043", "IMG-0045", "IMG-0047",
    "IMG-0055", "IMG-0075", "IMG-0088", "IMG-0094", "IMG-0102", "IMG-0117"
]
for stem in panel_images:
    matches = df[df["image_name"].str.contains(stem)]
    if len(matches):
        print(f"{stem:15} → {matches.iloc[0]['status']}")
    else:
        print(f"{stem:15} → not in df")

IMG-0023        → success
IMG-0027        → success
IMG-0043        → success
IMG-0045        → ambiguous_tip
IMG-0047        → ambiguous_tip
IMG-0055        → success
IMG-0075        → success
IMG-0088        → ambiguous_tip
IMG-0094        → success
IMG-0102        → ambiguous_tip
IMG-0117        → ambiguous_tip


In [42]:
# Grep cell 13 for the string
import inspect
src = inspect.getsource(compute_centerline_fixed)
for ln, line in enumerate(src.split("\n")):
    if "success_overlap_mode" in line:
        print(f"{ln:3}: {line}")

140:         #   Large gap -> success_overlap_mode (confident loop recovery)
145:                 status = "success_overlap_mode"
147:                       f"{THICKNESS_ABS_GAP_MIN} → success_overlap_mode")
157:                 status = "success_overlap_mode"
159:                       f"{THICKNESS_ABS_GAP_MIN_CUT_POINT} → success_overlap_mode")
167:             status = "success_overlap_mode"
169:                   f"{THICKNESS_ABS_GAP_MIN_CUT_POINT} → success_overlap_mode")


In [43]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 25 — REPLACE THE FEATURE SELECTION CELL
# ─────────────────────────────────────────────────────────────────────────────
# Adds the 7 new tip-point features from cell 15b to the ML feature list.
# Everything else is identical to before.

feature_cols = [
    "image_name",
    "status",

    "tip_y",
    "tip_x",

    "tip_pixels",
    "atrium_pixels",
    "overlap_pixels",

    "tip_area_inside_ratio",
    "atrium_coverage_ratio",
    "iou_tip_atrium",
    "tip_inside_atrium",
    "centerline_inside_ratio",

    "tip_dx",
    "tip_dy",
    "tip_angle",

    "bbox_dy_top",
    "bbox_dy_bottom",
    "bbox_dx_left",
    "bbox_dx_right",

    "bbox_dy_top_norm",
    "bbox_dy_bottom_norm",
    "bbox_dx_left_norm",
    "bbox_dx_right_norm",

    "bbox_vertical_inside",
    "bbox_horizontal_inside",
    "bbox_inside_atrium",

    "edge_dy_top_min",
    "edge_dy_top_mean",
    "edge_dy_bottom_min",
    "edge_dy_bottom_mean",
    "edge_dx_left_min",
    "edge_dx_left_mean",
    "edge_dx_right_min",
    "edge_dx_right_mean",

    "edge_dy_top_min_norm",
    "edge_dy_top_mean_norm",
    "edge_dy_bottom_min_norm",
    "edge_dy_bottom_mean_norm",
    "edge_dx_left_min_norm",
    "edge_dx_left_mean_norm",
    "edge_dx_right_min_norm",
    "edge_dx_right_mean_norm",

    "region_nearest_euclidean",
    "region_nearest_dy",
    "region_nearest_dx",
    "region_nearest_dist_norm_h",
    "region_nearest_dist_norm_w",

    "mean_tip_radius",
    "mean_prox_radius",
    "tip_length_measured_px",

    # NEW point-level features (cell 15b)
    "tip_depth_in_atrium_px",
    "tip_depth_in_atrium_norm",
    "tip_y_norm_to_atrium",
    "tip_x_norm_to_atrium",
    "distance_from_optimal_depth",
    "tip_to_atrium_dist_norm",
    "catheter_mask_solidity",
]

existing_feature_cols = [c for c in feature_cols if c in success_df.columns]
ml_df = success_df[existing_feature_cols].copy()

ML_CSV_OUTPUT_PATH = os.path.join(OUTPUT_FOLDER, "catheter_tip_features_ml_ready.csv")
ml_df.to_csv(ML_CSV_OUTPUT_PATH, index=False)

print(f"Saved ML-ready CSV to: {ML_CSV_OUTPUT_PATH}")
print(f"Total feature columns saved: {len(existing_feature_cols)}")
print("\nColumns in ml_df:")
print(ml_df.columns.tolist())

ml_df.head()

Saved ML-ready CSV to: /vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis/predicted_masks/catheter_tip_features_ml_ready.csv
Total feature columns saved: 56

Columns in ml_df:
['image_name', 'status', 'tip_y', 'tip_x', 'tip_pixels', 'atrium_pixels', 'overlap_pixels', 'tip_area_inside_ratio', 'atrium_coverage_ratio', 'iou_tip_atrium', 'tip_inside_atrium', 'centerline_inside_ratio', 'tip_dx', 'tip_dy', 'tip_angle', 'bbox_dy_top', 'bbox_dy_bottom', 'bbox_dx_left', 'bbox_dx_right', 'bbox_dy_top_norm', 'bbox_dy_bottom_norm', 'bbox_dx_left_norm', 'bbox_dx_right_norm', 'bbox_vertical_inside', 'bbox_horizontal_inside', 'bbox_inside_atrium', 'edge_dy_top_min', 'edge_dy_top_mean', 'edge_dy_bottom_min', 'edge_dy_bottom_mean', 'edge_dx_left_min', 'edge_dx_left_mean', 'edge_dx_right_min', 'edge_dx_right_mean', 'edge_dy_top_min_norm', 'edge_dy_top_mean_norm', 'edge_dy_bottom_min_norm', 'edge_dy_bottom_mean_norm', 'edge_dx_left_min_norm', 'edge_dx_left_mean_norm', 'edge_dx_right_min_norm', 'edge

,image_name,status,tip_y,tip_x,tip_pixels,atrium_pixels,overlap_pixels,tip_area_inside_ratio,atrium_coverage_ratio,iou_tip_atrium,...,region_nearest_dist_norm_w,mean_tip_radius,mean_prox_radius,tip_length_measured_px,tip_depth_in_atrium_px,tip_depth_in_atrium_norm,tip_y_norm_to_atrium,tip_x_norm_to_atrium,distance_from_optimal_depth,tip_to_atrium_dist_norm
0,IMG-0023-00001.png,success,397.0,424.0,3079,22722,0,0.000000,0.000000,0.000000,...,0.036765,6.885749,13.269213,200.000000,0.000000,0.000000,-0.043307,0.441176,-0.376640,0.043307
1,IMG-0025-00001.png,success,555.0,434.0,3154,18486,888,0.281547,0.048036,0.042791,...,0.000000,7.000000,12.469801,200.000000,54.118063,0.226435,0.242678,0.500000,-0.090656,0.000000
2,IMG-0027-00001.png,success,656.0,443.0,3265,23147,3265,1.000000,0.141055,0.141055,...,0.000000,15.287564,8.235258,113.449224,113.449224,0.476677,0.567227,0.756944,0.067227,0.000000
3,IMG-0029-00001.png,success,476.0,433.0,2814,16336,668,0.237385,0.040891,0.036143,...,0.000000,6.664022,13.194482,200.000000,44.083576,0.221526,0.256281,0.674797,-0.077052,0.000000
4,IMG-0031-00001.png,success,521.0,449.0,3353,20162,2835,0.845511,0.140611,0.137089,...,0.000000,7.694564,16.233706,200.000000,0.000000,0.000000,0.983333,0.689873,0.483333,0.005556


In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_validate,
    learning_curve, cross_val_predict
)
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [45]:
import os
import sys
import pandas as pd

# Make sure Python knows where your custom libraries are (including openpyxl)
lib_dir = os.path.join(os.environ.get('VSC_DATA'), 'thesis_libs')
if lib_dir not in sys.path:
    sys.path.insert(0, lib_dir)

# --- LOAD DATA ---
FEATURE_CSV_PATH = ML_CSV_OUTPUT_PATH   # defined in your earlier cells

# FIX: Use the exact absolute path for the Excel file
LABEL_XLSX_PATH = "/vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis/atriumandcatheter.xlsx"

features_df = pd.read_csv(FEATURE_CSV_PATH)
print("Features shape:", features_df.shape)

labels_df = pd.read_excel(LABEL_XLSX_PATH)
print("Labels shape:", labels_df.shape)
print("Label columns:", labels_df.columns.tolist())

# Prepare join keys
features_df["ap_id"] = (
    features_df["image_name"]
    .astype(str)
    .str.replace(r"\.tif$|\.tiff$|\.png$|\.jpg$|\.jpeg$", "", regex=True)
    .str.replace(r"\s*-\s*PRIMARY$", "", regex=True)
    .str.replace(r"\.dcm$", "", regex=True)
    .str.strip()
)
labels_df["ap_id"] = labels_df["ap_id"].astype(str).str.strip()

# Merge
merged_df = features_df.merge(labels_df, on="ap_id", how="inner")
print("Merged shape:", merged_df.shape)
print("Missing labels:", merged_df["tip (1-ok, 0-no)"].isna().sum())

# Include all three valid statuses
usable_status = ["success", "success_overlap_mode", "ambiguous_tip"]
model_df = merged_df[merged_df["status"].isin(usable_status)].copy()
print("\nUsable rows:", model_df.shape[0])
print(model_df["status"].value_counts(dropna=False))
print(model_df["tip (1-ok, 0-no)"].value_counts(dropna=False))

Features shape: (220, 56)
Labels shape: (277, 3)
Label columns: ['ap_id', 'arch (1-ok, 0-no)', 'tip (1-ok, 0-no)']
Merged shape: (220, 59)
Missing labels: 0

Usable rows: 220
status
success                 197
ambiguous_tip            20
success_overlap_mode      3
Name: count, dtype: int64
tip (1-ok, 0-no)
0    119
1    101
Name: count, dtype: int64


In [46]:
import seaborn as sns
import matplotlib.pyplot as plt

# Candidate point-based features — check for redundancy before selecting
CANDIDATE_FEATURES = [
    "distance_from_optimal_depth",   # NEW — replaces tip_in_upper_third
    "tip_y_norm_to_atrium",          # may correlate with distance_from_optimal_depth
    "tip_x_norm_to_atrium",          # horizontal alignment
    "tip_depth_in_atrium_norm",      # arc-length depth
    "tip_to_atrium_dist_norm",       # NEW — replaces region_nearest_dist_norm_h
    "mean_tip_radius",               # catheter thickness
    "tip_dy",                        # direction tip is pointing
]

available = [f for f in CANDIDATE_FEATURES if f in model_df.columns]
corr_df = model_df[available].corr(method="pearson")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_df, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title(
    "Pearson Correlation — Candidate Point-Based Features\n"
    "(pairs with |r| > 0.75 are candidates for removal)",
    fontsize=12
)
plt.tight_layout()
plt.savefig("correlation_matrix_candidate.png", dpi=150)
plt.show()
print("Saved: correlation_matrix_candidate.png")

print("\n=== Pairs with |r| > 0.75 → keep only one from each pair ===")
found = False
for i, f1 in enumerate(available):
    for f2 in available[i + 1:]:
        r = corr_df.loc[f1, f2]
        if abs(r) > 0.75:
            print(f"  {f1:<42}  ↔  {f2:<42}  r = {r:+.3f}")
            found = True
if not found:
    print("  None found above 0.75.")

print("\n=== Point-biserial correlation with target label ===")
y_corr = model_df["tip (1-ok, 0-no)"].astype(float)
for f in available:
    r = model_df[f].corr(y_corr)
    print(f"  {f:<42}  r_with_target = {r:+.3f}")




Saved: correlation_matrix_candidate.png

=== Pairs with |r| > 0.75 → keep only one from each pair ===
  distance_from_optimal_depth                 ↔  tip_y_norm_to_atrium                        r = +0.982
  distance_from_optimal_depth                 ↔  tip_depth_in_atrium_norm                    r = +0.772
  distance_from_optimal_depth                 ↔  tip_to_atrium_dist_norm                     r = -0.767
  tip_y_norm_to_atrium                        ↔  tip_depth_in_atrium_norm                    r = +0.838

=== Point-biserial correlation with target label ===
  distance_from_optimal_depth                 r_with_target = +0.334
  tip_y_norm_to_atrium                        r_with_target = +0.393
  tip_x_norm_to_atrium                        r_with_target = -0.006
  tip_depth_in_atrium_norm                    r_with_target = +0.497
  tip_to_atrium_dist_norm                     r_with_target = -0.063
  mean_tip_radius                             r_with_target = +0.138
  tip_dy      

In [47]:
# See what the Excel ap_ids look like for similar images
print(labels_df[labels_df["ap_id"].str.contains("0069|0259|5114", na=False)])

              ap_id  arch (1-ok, 0-no)  tip (1-ok, 0-no)
34   IMG-0069-00001                  0                 1
130  IMG-0259-00001                  0                 1
266  IMG-5114-00001                  1                 1


In [48]:
# Check which images didn't match
unmatched = features_df[~features_df["ap_id"].isin(labels_df["ap_id"])]
print("Unmatched feature rows:", len(unmatched))
print(unmatched[["image_name", "ap_id", "status"]])

Unmatched feature rows: 0
Empty DataFrame
Columns: [image_name, ap_id, status]
Index: []


In [49]:
# Check how many success cases have a tip label of 0 (wrong tip)
success_wrong = merged_df[
    (merged_df["status"] == "success") &
    (merged_df["tip (1-ok, 0-no)"] == 0)
]
print(f"Success cases with wrong tip label: {len(success_wrong)}")
print(success_wrong[["image_name", "tip (1-ok, 0-no)"]].to_string())

Success cases with wrong tip label: 111
             image_name  tip (1-ok, 0-no)
0    IMG-0023-00001.png                 0
1    IMG-0025-00001.png                 0
3    IMG-0029-00001.png                 0
4    IMG-0031-00001.png                 0
5    IMG-0033-00001.png                 0
6    IMG-0035-00001.png                 0
8    IMG-0039-00001.png                 0
14   IMG-0051-00001.png                 0
18   IMG-0059-00001.png                 0
19   IMG-0061-00001.png                 0
21   IMG-0065-00001.png                 0
22   IMG-0067-00001.png                 0
24   IMG-0071-00001.png                 0
25   IMG-0073-00001.png                 0
28   IMG-0079-00001.png                 0
29   IMG-0081-00001.png                 0
31   IMG-0084-00001.png                 0
32   IMG-0086-00001.png                 0
34   IMG-0090-00001.png                 0
35   IMG-0092-00001.png                 0
36   IMG-0094-00001.png                 0
37   IMG-0098-00001.png             

In [50]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

SCORING = {
    "accuracy":  "accuracy",
    "precision": "precision",
    "recall":    "recall",
    "f1":        "f1",
    "roc_auc":   "roc_auc",
}

# 5 pure tip-POINT features
# All derived from the single (tip_y, tip_x) coordinate — no tube mask used.
#
# Removed from previous list:
#   tip_area_inside_ratio     tube-mask overlap ratio
#   bbox_dy_bottom_norm       tube-mask bounding-box distance
#   edge_dx_left_mean_norm    tube-mask edge distance
#   region_nearest_dist_norm_h  → replaced by tip_to_atrium_dist_norm
#   tip_in_upper_third        binary → replaced by distance_from_optimal_depth
#
# Dropped as correlated (confirmed by Change 3 heatmap):
#   tip_y_norm_to_atrium      redundant with distance_from_optimal_depth
#
SELECTED_FEATURES = [
    "tip_depth_in_atrium_norm",     # 1 — strongest predictor (r=0.490)
    "mean_tip_radius",              # 2 — independent signal  (r=0.157)
]
# 5 features × 220 samples = 44 samples/feature (healthy ratio)

missing = [f for f in SELECTED_FEATURES if f not in model_df.columns]
if missing:
    raise KeyError(
        f"Missing features: {missing}\n"
        f"Run batch processing first to regenerate the CSV with new columns.\n"
        f"Available: {sorted(model_df.columns.tolist())}"
    )

feature_cols = SELECTED_FEATURES
X_clean = model_df[feature_cols].copy()
y_clean = model_df["tip (1-ok, 0-no)"].astype(int).copy()

print(f"Features used        : {len(feature_cols)}")
print(f"Samples              : {X_clean.shape[0]}")
print(f"Samples per feature  : {X_clean.shape[0] / len(feature_cols):.1f}")
print(f"Class balance — 0 (malpositioned): {(y_clean==0).sum()}  "
      f"|  1 (correct): {(y_clean==1).sum()}")
print("\nFeature list:")
for i, f in enumerate(feature_cols, 1):
    print(f"  {i}. {f}")

for label_col in ["tip (1-ok, 0-no)", "arch (1-ok, 0-no)"]:
    assert label_col not in X_clean.columns, f"LEAK: {label_col} in features!"
print("\n✓ No label leakage detected.")

print("\nFinal pairwise correlations (|r| > 0.75 only):")
corr_final = X_clean.corr().abs()
high = [(corr_final.columns[i], corr_final.columns[j], corr_final.iloc[i, j])
        for i in range(len(corr_final.columns))
        for j in range(i + 1, len(corr_final.columns))
        if corr_final.iloc[i, j] > 0.75]
if high:
    for a, b, r in high:
        print(f"  {a}  ↔  {b}  |r| = {r:.3f}  → consider dropping one")
else:
    print("  None — feature set is well-decorrelated. ✓")

Features used        : 2
Samples              : 220
Samples per feature  : 110.0
Class balance — 0 (malpositioned): 119  |  1 (correct): 101

Feature list:
  1. tip_depth_in_atrium_norm
  2. mean_tip_radius

✓ No label leakage detected.

Final pairwise correlations (|r| > 0.75 only):
  None — feature set is well-decorrelated. ✓


In [51]:
# LOGISTIC REGRESSION  (with hyperparameter tuning)

lr_base = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("model",   LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42,
    )),
])

lr_param_grid = {
    "model__C":       [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
    "model__penalty": ["l1", "l2"],
    "model__solver":  ["liblinear"],
}

lr_grid = GridSearchCV(
    estimator=lr_base,
    param_grid=lr_param_grid,
    cv=CV,
    scoring="roc_auc",
    n_jobs=1,
    refit=True,
    verbose=0,
)
lr_grid.fit(X_clean, y_clean)

print("=== Logistic Regression — best params ===")
for k, v in lr_grid.best_params_.items():
    print(f"  {k}: {v}")
print(f"  Inner CV ROC-AUC: {lr_grid.best_score_:.4f}")

lr_pipeline = lr_grid.best_estimator_

lr_results = cross_validate(
    lr_pipeline, X_clean, y_clean,
    cv=CV, scoring=SCORING,
    return_train_score=True,
)

print("\n=== Logistic Regression — outer CV ===")
print(f"{'metric':<12} {'train':>8} {'val':>8} {'gap':>8} {'±std':>8}")
print("-" * 46)
for m in SCORING:
    tr = lr_results[f"train_{m}"].mean()
    va = lr_results[f"test_{m}"].mean()
    sd = lr_results[f"test_{m}"].std()
    print(f"{m:<12} {tr:>8.4f} {va:>8.4f} {tr-va:>+8.4f} {sd:>8.4f}")


=== Logistic Regression — best params ===
  model__C: 0.001
  model__penalty: l2
  model__solver: liblinear
  Inner CV ROC-AUC: 0.8266

=== Logistic Regression — outer CV ===
metric          train      val      gap     ±std
----------------------------------------------
accuracy       0.8341   0.8136  +0.0205   0.0633
precision      0.8209   0.8067  +0.0142   0.0742
recall         0.8169   0.7819  +0.0349   0.0754
f1             0.8188   0.7936  +0.0253   0.0723
roc_auc        0.8441   0.8266  +0.0175   0.0763


In [67]:
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)
from sklearn.model_selection import cross_val_predict, StratifiedKFold
import numpy as np

# Make sure X, y, skf are defined
FINAL_FEATURES = ["tip_depth_in_atrium_norm", "mean_tip_radius"]
X = model_df[FINAL_FEATURES].values
y = model_df["tip (1-ok, 0-no)"].values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("✓ Imports and variables ready")

✓ Imports and variables ready


In [68]:
# ── LR OOF metrics + specificity ─────────────────────────────
lr_oof_probs = cross_val_predict(lr_pipeline, X, y, cv=skf, method='predict_proba')[:,1]
lr_oof_preds = cross_val_predict(lr_pipeline, X, y, cv=skf)

lr_oof_auc  = roc_auc_score(y, lr_oof_probs)
lr_oof_acc  = accuracy_score(y, lr_oof_preds)
lr_oof_prec, lr_oof_rec, lr_oof_f1, _ = precision_recall_fscore_support(
    y, lr_oof_preds, average='binary', zero_division=0)

tn, fp, fn, tp = confusion_matrix(y, lr_oof_preds).ravel()
lr_oof_spec = tn / (tn + fp)

# SE across folds
lr_sens_folds, lr_spec_folds = [], []
for _, val_idx in skf.split(X, y):
    yv = y[val_idx]
    yp = lr_pipeline.predict(X[val_idx])
    tn_f, fp_f, fn_f, tp_f = confusion_matrix(yv, yp).ravel()
    lr_sens_folds.append(tp_f / (tp_f + fn_f))
    lr_spec_folds.append(tn_f / (tn_f + fp_f))

print("=== Logistic Regression ===")
print(f"  AUC         : {lr_oof_auc:.3f}")
print(f"  Accuracy    : {lr_oof_acc:.3f}")
print(f"  Sensitivity : {lr_oof_rec:.3f}  SE={np.std(lr_sens_folds)/np.sqrt(5):.3f}")
print(f"  Specificity : {lr_oof_spec:.3f}  SE={np.std(lr_spec_folds)/np.sqrt(5):.3f}")

=== Logistic Regression ===
  AUC         : 0.833
  Accuracy    : 0.814
  Sensitivity : 0.782  SE=0.033
  Specificity : 0.840  SE=0.025


In [52]:
import scipy.stats as stats

# ── Collect coefficients fold-by-fold ────────────────────────────────────────
best_C       = lr_grid.best_params_["model__C"]
best_penalty = lr_grid.best_params_["model__penalty"]

fold_coefs = []
for train_idx, val_idx in CV.split(X_clean, y_clean):
    X_tr = X_clean.iloc[train_idx]
    y_tr = y_clean.iloc[train_idx]
    fold_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
        ("model",   LogisticRegression(
            C=best_C, penalty=best_penalty, solver="liblinear",
            class_weight="balanced", max_iter=2000, random_state=42,
        )),
    ])
    fold_pipe.fit(X_tr, y_tr)
    fold_coefs.append(fold_pipe.named_steps["model"].coef_[0])

fold_coefs = np.array(fold_coefs)          # (5, n_features)
coef_mean  = fold_coefs.mean(axis=0)
coef_std   = fold_coefs.std(axis=0, ddof=1)

# 95% CI  (t-distribution, df = 4 for 5 folds)
t_crit = stats.t.ppf(0.975, df=len(fold_coefs) - 1)
ci_lo  = coef_mean - t_crit * coef_std / np.sqrt(len(fold_coefs))
ci_hi  = coef_mean + t_crit * coef_std / np.sqrt(len(fold_coefs))

coef_df = pd.DataFrame({
    "Feature":    SELECTED_FEATURES,
    "Coef":       coef_mean.round(4),
    "Std":        coef_std.round(4),
    "CI_lo":      ci_lo.round(4),
    "CI_hi":      ci_hi.round(4),
    "Interpretation": [
        "↑ malposition" if c > 0 else "↑ correct placement"
        for c in coef_mean
    ],
}).sort_values("Coef", key=abs, ascending=False)

# ── Print table ───────────────────────────────────────────────────────────────
print("=" * 75)
print(f"LOGISTIC REGRESSION — SIGNED COEFFICIENTS "
      f"(C={best_C}, penalty={best_penalty})")
print("Standardised — mean ± std across 5 CV folds")
print("=" * 75)
print(f"\n  {'Feature':<35} {'Coef':>8} {'±Std':>7}  "
      f"{'95% CI':>22}  Interpretation")
print("  " + "─" * 85)
for _, row in coef_df.iterrows():
    ci_str = f"[{row['CI_lo']:+.4f}, {row['CI_hi']:+.4f}]"
    print(f"  {row['Feature']:<35} {row['Coef']:>+8.4f} "
          f"{row['Std']:>7.4f}  {ci_str:>22}  {row['Interpretation']}")

print("\n  Interpretation guide:")
print("  POSITIVE coef  →  higher feature value = more likely MALPOSITIONED")
print("  NEGATIVE coef  →  higher feature value = more likely CORRECT")
print("  Expected signs:")
print("    distance_from_optimal_depth → POSITIVE  (farther from zone = worse)")
print("    tip_to_atrium_dist_norm     → POSITIVE  (farther from atrium = worse)")
print("    tip_dy                      → NEGATIVE  (tip pointing down = better)")

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 0.75 * len(SELECTED_FEATURES) + 2))
colors = ["#d62728" if c > 0 else "#1f77b4" for c in coef_df["Coef"]]
ax.barh(coef_df["Feature"], coef_df["Coef"],
        xerr=coef_df["Std"], color=colors,
        capsize=5, alpha=0.85, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.9, linestyle="--")
ax.set_xlabel(
    "Standardised Coefficient\n"
    "Red = higher value → more likely malpositioned  |  "
    "Blue = higher value → more likely correct",
    fontsize=10
)
ax.set_title(
    f"Logistic Regression — Signed Coefficients\n"
    f"C={best_C}, {best_penalty} — mean ± std over 5 folds",
    fontsize=11
)
for i, (_, row) in enumerate(coef_df.iterrows()):
    offset = 6 if row["Coef"] >= 0 else -6
    ha     = "left" if row["Coef"] >= 0 else "right"
    ax.annotate(
        f"{row['Coef']:+.3f}",
        xy=(row["Coef"], i),
        xytext=(offset, 0), textcoords="offset points",
        va="center", ha=ha, fontsize=9,
    )
plt.tight_layout()
plt.savefig("lr_signed_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: lr_signed_coefficients.png")


LOGISTIC REGRESSION — SIGNED COEFFICIENTS (C=0.001, penalty=l2)
Standardised — mean ± std across 5 CV folds

  Feature                                 Coef    ±Std                  95% CI  Interpretation
  ─────────────────────────────────────────────────────────────────────────────────────
  tip_depth_in_atrium_norm             +0.0421  0.0041      [+0.0369, +0.0472]  ↑ malposition
  mean_tip_radius                      +0.0117  0.0033      [+0.0076, +0.0158]  ↑ malposition

  Interpretation guide:
  POSITIVE coef  →  higher feature value = more likely MALPOSITIONED
  NEGATIVE coef  →  higher feature value = more likely CORRECT
  Expected signs:
    distance_from_optimal_depth → POSITIVE  (farther from zone = worse)
    tip_to_atrium_dist_norm     → POSITIVE  (farther from atrium = worse)
    tip_dy                      → NEGATIVE  (tip pointing down = better)
Saved: lr_signed_coefficients.png


In [53]:
# RANDOM FOREST  (with hyperparameter tuning)

rf_base = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

rf_param_grid = {
    "model__max_depth":         [3, 4, 5],
    "model__min_samples_leaf":  [4, 6, 8],
    "model__min_samples_split": [4, 8],
}
rf_grid = GridSearchCV(
    estimator=rf_base,
    param_grid=rf_param_grid,
    cv=CV,
    scoring="roc_auc",
    n_jobs=1,
    refit=True,
    verbose=0,
)
rf_grid.fit(X_clean, y_clean)

print("=== Random Forest — best params (from grid) ===")
for k, v in rf_grid.best_params_.items():
    print(f"  {k}: {v}")
print(f"  Inner CV ROC-AUC: {rf_grid.best_score_:.4f}")

rf_pipeline = rf_grid.best_estimator_


# ── Outer CV evaluation ──────────────────────────────────────────────────────
rf_results = cross_validate(
    rf_pipeline, X_clean, y_clean,
    cv=CV, scoring=SCORING,
    return_train_score=True,
)

print("\n=== Random Forest — outer CV ===")
print(f"{'metric':<12} {'train':>8} {'val':>8} {'gap':>8} {'±std':>8}")
print("-" * 46)
for m in SCORING:
    tr = rf_results[f"train_{m}"].mean()
    va = rf_results[f"test_{m}"].mean()
    sd = rf_results[f"test_{m}"].std()
    print(f"{m:<12} {tr:>8.4f} {va:>8.4f} {tr-va:>+8.4f} {sd:>8.4f}")

=== Random Forest — best params (from grid) ===
  model__max_depth: 5
  model__min_samples_leaf: 4
  model__min_samples_split: 4
  Inner CV ROC-AUC: 0.9026

=== Random Forest — outer CV ===
metric          train      val      gap     ±std
----------------------------------------------
accuracy       0.9023   0.7955  +0.1068   0.0643
precision      0.8908   0.7843  +0.1065   0.1039
recall         0.8985   0.7924  +0.1061   0.0723
f1             0.8942   0.7827  +0.1115   0.0610
roc_auc        0.9703   0.9026  +0.0677   0.0354


In [69]:
# ── RF OOF metrics + specificity ─────────────────────────────
rf_oof_probs = cross_val_predict(rf_pipeline, X, y, cv=skf, method='predict_proba')[:,1]
rf_oof_preds = cross_val_predict(rf_pipeline, X, y, cv=skf)

rf_oof_auc  = roc_auc_score(y, rf_oof_probs)
rf_oof_acc  = accuracy_score(y, rf_oof_preds)
rf_oof_prec, rf_oof_rec, rf_oof_f1, _ = precision_recall_fscore_support(
    y, rf_oof_preds, average='binary', zero_division=0)

tn, fp, fn, tp = confusion_matrix(y, rf_oof_preds).ravel()
rf_oof_spec = tn / (tn + fp)

rf_sens_folds, rf_spec_folds = [], []
for _, val_idx in skf.split(X, y):
    yv = y[val_idx]
    yp = rf_pipeline.predict(X[val_idx])
    tn_f, fp_f, fn_f, tp_f = confusion_matrix(yv, yp).ravel()
    rf_sens_folds.append(tp_f / (tp_f + fn_f))
    rf_spec_folds.append(tn_f / (tn_f + fp_f))

print("=== Random Forest ===")
print(f"  AUC         : {rf_oof_auc:.3f}")
print(f"  Accuracy    : {rf_oof_acc:.3f}")
print(f"  Sensitivity : {rf_oof_rec:.3f}  SE={np.std(rf_sens_folds)/np.sqrt(5):.3f}")
print(f"  Specificity : {rf_oof_spec:.3f}  SE={np.std(rf_spec_folds)/np.sqrt(5):.3f}")

=== Random Forest ===
  AUC         : 0.900
  Accuracy    : 0.795
  Sensitivity : 0.792  SE=0.021
  Specificity : 0.798  SE=0.015


In [54]:
# XGBOOST  (with hyperparameter tuning)

neg = int((y_clean == 0).sum())
pos = int((y_clean == 1).sum())
spw = neg / pos
print(f"scale_pos_weight = {spw:.3f}  (neg={neg}, pos={pos})")

xgb_base = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   XGBClassifier(
        n_estimators=300,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=spw,
        random_state=42,
        n_jobs=1,
        verbosity=0,
    )),
])

xgb_param_grid = {
    "model__max_depth":         [2, 3],            # remove 4, 5
    "model__learning_rate":     [0.02, 0.03, 0.05],
    "model__n_estimators":      [50, 100, 200],    # add early stopping range
    "model__min_child_weight":  [3, 5, 10],        # was [1, 3]; push higher
    "model__reg_lambda":        [2.0, 5.0, 10.0],  # was [1.0, 2.0]; push higher
    "model__gamma":             [0.1, 0.5, 1.0],   # was [0.0, 0.1]; push higher
    "model__subsample":         [0.6, 0.7, 0.8],
    "model__colsample_bytree":  [0.6, 0.8],
}

xgb_grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=xgb_param_grid,
    cv=CV,
    scoring="roc_auc",
    n_jobs=1,
    refit=True,
    verbose=0,
)
xgb_grid.fit(X_clean, y_clean)

print("\n=== XGBoost — best params ===")
for k, v in xgb_grid.best_params_.items():
    print(f"  {k}: {v}")
print(f"  Inner CV ROC-AUC: {xgb_grid.best_score_:.4f}")

xgb_pipeline = xgb_grid.best_estimator_

xgb_results = cross_validate(
    xgb_pipeline, X_clean, y_clean,
    cv=CV, scoring=SCORING,
    return_train_score=True,
)

print("\n=== XGBoost — outer CV ===")
print(f"{'metric':<12} {'train':>8} {'val':>8} {'gap':>8} {'±std':>8}")
print("-" * 46)
for m in SCORING:
    tr = xgb_results[f"train_{m}"].mean()
    va = xgb_results[f"test_{m}"].mean()
    sd = xgb_results[f"test_{m}"].std()
    print(f"{m:<12} {tr:>8.4f} {va:>8.4f} {tr-va:>+8.4f} {sd:>8.4f}")


scale_pos_weight = 1.178  (neg=119, pos=101)

=== XGBoost — best params ===
  model__colsample_bytree: 0.6
  model__gamma: 1.0
  model__learning_rate: 0.05
  model__max_depth: 2
  model__min_child_weight: 3
  model__n_estimators: 100
  model__reg_lambda: 10.0
  model__subsample: 0.8
  Inner CV ROC-AUC: 0.9051

=== XGBoost — outer CV ===
metric          train      val      gap     ±std
----------------------------------------------
accuracy       0.8511   0.8227  +0.0284   0.0485
precision      0.8650   0.8311  +0.0340   0.0586
recall         0.8020   0.7724  +0.0296   0.0673
f1             0.8311   0.7996  +0.0315   0.0567
roc_auc        0.9308   0.9051  +0.0258   0.0443


In [70]:
# ── XGB OOF metrics + specificity ────────────────────────────
xgb_oof_probs = cross_val_predict(xgb_pipeline, X, y, cv=skf, method='predict_proba')[:,1]
xgb_oof_preds = cross_val_predict(xgb_pipeline, X, y, cv=skf)

xgb_oof_auc  = roc_auc_score(y, xgb_oof_probs)
xgb_oof_acc  = accuracy_score(y, xgb_oof_preds)
xgb_oof_prec, xgb_oof_rec, xgb_oof_f1, _ = precision_recall_fscore_support(
    y, xgb_oof_preds, average='binary', zero_division=0)

tn, fp, fn, tp = confusion_matrix(y, xgb_oof_preds).ravel()
xgb_oof_spec = tn / (tn + fp)

xgb_sens_folds, xgb_spec_folds = [], []
for _, val_idx in skf.split(X, y):
    yv = y[val_idx]
    yp = xgb_pipeline.predict(X[val_idx])
    tn_f, fp_f, fn_f, tp_f = confusion_matrix(yv, yp).ravel()
    xgb_sens_folds.append(tp_f / (tp_f + fn_f))
    xgb_spec_folds.append(tn_f / (tn_f + fp_f))

print("=== XGBoost ===")
print(f"  AUC         : {xgb_oof_auc:.3f}")
print(f"  Accuracy    : {xgb_oof_acc:.3f}")
print(f"  Sensitivity : {xgb_oof_rec:.3f}  SE={np.std(xgb_sens_folds)/np.sqrt(5):.3f}")
print(f"  Specificity : {xgb_oof_spec:.3f}  SE={np.std(xgb_spec_folds)/np.sqrt(5):.3f}")

=== XGBoost ===
  AUC         : 0.881
  Accuracy    : 0.823
  Sensitivity : 0.772  SE=0.042
  Specificity : 0.866  SE=0.018


In [71]:
# MODEL COMPARISON TABLE

rows = []
for name, res in [
    ("Logistic Regression", lr_results),
    ("Random Forest",       rf_results),
    ("XGBoost",             xgb_results),
]:
    row = {"Model": name}
    for m in SCORING:
        va = res[f"test_{m}"]
        row[f"{m}_mean"] = round(va.mean(), 4)
        row[f"{m}_std"]  = round(va.std(),  4)
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index("Model")
print("\n=== Model comparison (validation, 5-fold CV) ===")
print(comparison_df.to_string())



=== Model comparison (validation, 5-fold CV) ===
                     accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std  roc_auc_mean  roc_auc_std
Model                                                                                                                                               
Logistic Regression         0.8136        0.0633          0.8067         0.0742       0.7819      0.0754   0.7936  0.0723        0.8266       0.0763
Random Forest               0.7955        0.0643          0.7843         0.1039       0.7924      0.0723   0.7827  0.0610        0.9026       0.0354
XGBoost                     0.8227        0.0485          0.8311         0.0586       0.7724      0.0673   0.7996  0.0567        0.9051       0.0443


In [72]:
# LEARNING CURVES

def plot_learning_curve(pipeline, X, y, cv, title, ax):
    train_sizes, train_scores, val_scores = learning_curve(
        pipeline, X, y,
        cv=cv,
        scoring="accuracy",
        train_sizes=np.linspace(0.1, 1.0, 10),
        n_jobs=1,
    )
    tr_mean = train_scores.mean(axis=1)
    tr_std  = train_scores.std(axis=1)
    va_mean = val_scores.mean(axis=1)
    va_std  = val_scores.std(axis=1)

    ax.plot(train_sizes, tr_mean, color="#185FA5", lw=2, label="Train")
    ax.fill_between(train_sizes, tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.12, color="#185FA5")
    ax.plot(train_sizes, va_mean, color="#D85A30", lw=2, label="Validation")
    ax.fill_between(train_sizes, va_mean - va_std, va_mean + va_std,
                    alpha=0.12, color="#D85A30")

    final_gap = tr_mean[-1] - va_mean[-1]
    ax.set_title(f"{title}\n(gap at full data: {final_gap:+.3f})", fontsize=10)
    ax.set_xlabel("Training samples")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0.70, 1.02)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)


fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
plot_learning_curve(lr_pipeline,  X_clean, y_clean, CV, "Logistic Regression", axes[0])
plot_learning_curve(rf_pipeline,  X_clean, y_clean, CV, "Random Forest",       axes[1])
plot_learning_curve(xgb_pipeline, X_clean, y_clean, CV, "XGBoost",             axes[2])

plt.suptitle("Learning curves — all models", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("learning_curves_all.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: learning_curves_all.png")


Saved: learning_curves_all.png


In [73]:
# FEATURE IMPORTANCE
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
TOP_N = 15    # ← add this line back

# Logistic Regression — SIGNED coefficients (actual values, not absolute)
lr_model  = lr_pipeline.named_steps["model"]
lr_coefs  = lr_model.coef_[0]                              # signed, not abs
lr_imp_df = pd.DataFrame({
    "feature":    X_clean.columns,
    "importance": lr_coefs,                                # signed
}).sort_values("importance", ascending=False).head(TOP_N).iloc[::-1]

colors_lr = ["#d62728" if v > 0 else "#1f77b4"
             for v in lr_imp_df["importance"]]
axes[0].barh(lr_imp_df["feature"], lr_imp_df["importance"],
             color=colors_lr, alpha=0.8)
axes[0].axvline(0, color="black", lw=0.8, linestyle="--")
axes[0].set_title("Logistic Regression\nSigned coefficient\n"
                  "(red=malposition, blue=correct)", fontsize=10)
axes[0].set_xlabel("Coefficient (signed)")
axes[0].grid(True, axis="x", alpha=0.3)

# Random Forest — Gini importance
rf_model  = rf_pipeline.named_steps["model"]
rf_imp_df = pd.DataFrame({
    "feature":    X_clean.columns,
    "importance": rf_model.feature_importances_,
}).sort_values("importance", ascending=False).head(TOP_N).iloc[::-1]

axes[1].barh(rf_imp_df["feature"], rf_imp_df["importance"],
             color="#1D9E75", alpha=0.8)
axes[1].set_title("Random Forest\nGini importance (top 15)", fontsize=10)
axes[1].set_xlabel("Importance")
axes[1].grid(True, axis="x", alpha=0.3)

# XGBoost — gain importance
xgb_model  = xgb_pipeline.named_steps["model"]
xgb_imp_df = pd.DataFrame({
    "feature":    X_clean.columns,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False).head(TOP_N).iloc[::-1]

axes[2].barh(xgb_imp_df["feature"], xgb_imp_df["importance"],
             color="#7F77DD", alpha=0.8)
axes[2].set_title("XGBoost\nFeature importance (top 15)", fontsize=10)
axes[2].set_xlabel("Importance")
axes[2].grid(True, axis="x", alpha=0.3)

plt.suptitle("Feature importance — all models", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("feature_importance_all.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: feature_importance_all.png")


Saved: feature_importance_all.png


In [74]:
# CELL 10 — INDIVIDUAL CONFUSION MATRICES  (cross-validated predictions)

y_pred_lr  = cross_val_predict(lr_pipeline,  X_clean, y_clean, cv=CV)
y_pred_rf  = cross_val_predict(rf_pipeline,  X_clean, y_clean, cv=CV)
y_pred_xgb = cross_val_predict(xgb_pipeline, X_clean, y_clean, cv=CV)

cm_lr  = confusion_matrix(y_clean, y_pred_lr)
cm_rf  = confusion_matrix(y_clean, y_pred_rf)
cm_xgb = confusion_matrix(y_clean, y_pred_xgb)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, cm, title, color in [
    (axes[0], cm_lr,  "Logistic Regression", "Blues"),
    (axes[1], cm_rf,  "Random Forest",       "Greens"),
    (axes[2], cm_xgb, "XGBoost",             "Purples"),
]:
    errors = cm[0, 1] + cm[1, 0]
    acc    = (cm[0, 0] + cm[1, 1]) / cm.sum()
    disp   = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Wrong tip (0)", "Correct tip (1)"]
    )
    disp.plot(ax=ax, cmap=color, colorbar=False)
    ax.set_title(f"{title}\nCV accuracy: {acc:.3f}  |  errors: {errors}", fontsize=10)

plt.suptitle("Confusion matrices — cross-validated predictions", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrices_individual.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrices_individual.png")


Saved: confusion_matrices_individual.png


**Classifier Error Analysis**

In [75]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL — CLASSIFIER ERROR ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
import pandas as pd

# Re-run LR with best params to collect per-image predictions
# Use the best estimator from the grid search — guaranteed to match reported metrics
print("Using best LR params:", lr_grid.best_params_)
y_pred_cv = cross_val_predict(lr_grid.best_estimator_, X_clean, y_clean, cv=CV)
# Build error dataframe
df_errors = model_df[["image_name", "status"] + SELECTED_FEATURES].copy()
df_errors["y_true"] = y_clean.values
df_errors["y_pred"] = y_pred_cv
df_errors["error_type"] = "correct"
df_errors.loc[(df_errors.y_true==1) & (df_errors.y_pred==0), "error_type"] = "false_negative"
df_errors.loc[(df_errors.y_true==0) & (df_errors.y_pred==1), "error_type"] = "false_positive"

fn = df_errors[df_errors.error_type == "false_negative"]
fp = df_errors[df_errors.error_type == "false_positive"]
correct = df_errors[df_errors.error_type == "correct"]

print(f"False negatives (correct placement missed by model): {len(fn)}")
print(f"False positives (wrong placement not caught):        {len(fp)}")
print(f"Correct predictions:                                 {len(correct)}")

print("\n=== Feature means by error type ===")
comparison = pd.DataFrame({
    "correct":        correct[SELECTED_FEATURES].mean(),
    "false_negative": fn[SELECTED_FEATURES].mean(),
    "false_positive": fp[SELECTED_FEATURES].mean(),
}).round(3)
print(comparison)

print("\n=== False negative image names and status ===")
print(fn[["image_name", "status"] + SELECTED_FEATURES].to_string())

print("\n=== False positive image names and status ===")
print(fp[["image_name", "status"] + SELECTED_FEATURES].to_string())

Using best LR params: {'model__C': 0.001, 'model__penalty': 'l2', 'model__solver': 'liblinear'}
False negatives (correct placement missed by model): 22
False positives (wrong placement not caught):        19
Correct predictions:                                 179

=== Feature means by error type ===
                          correct  false_negative  false_positive
tip_depth_in_atrium_norm    0.258           0.149           0.509
mean_tip_radius             6.667           7.031           7.073

=== False negative image names and status ===
                           image_name                status  tip_depth_in_atrium_norm  mean_tip_radius
9                  IMG-0041-00001.png               success                  0.248208         7.178538
11                 IMG-0045-00001.png         ambiguous_tip                  0.000000         6.669388
33                 IMG-0088-00001.png         ambiguous_tip                  0.000000         7.852809
41                 IMG-0106-00001.png    

In [76]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

_CV      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
_SCORING = {"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"}

feature_sets = {
    "6 features — full point set": [
        "distance_from_optimal_depth",
        "tip_x_norm_to_atrium",
        "tip_depth_in_atrium_norm",
        "tip_to_atrium_dist_norm",
        "mean_tip_radius",
        "tip_dy",
    ],
    "5 features — drop tip_dy": [
        "distance_from_optimal_depth",
        "tip_x_norm_to_atrium",
        "tip_depth_in_atrium_norm",
        "tip_to_atrium_dist_norm",
        "mean_tip_radius",
    ],
    "4 features — drop mean_tip_radius": [
        "distance_from_optimal_depth",
        "tip_x_norm_to_atrium",
        "tip_depth_in_atrium_norm",
        "tip_to_atrium_dist_norm",
        "tip_dy",
    ],
    "3 features — depth + distance + radius": [   # ← fixed: was duplicate key
        "distance_from_optimal_depth",
        "tip_depth_in_atrium_norm",
        "tip_to_atrium_dist_norm",
    ],
    "3 features — depth + radius + optimal": [    # ← fixed: was duplicate key
        "distance_from_optimal_depth",
        "tip_depth_in_atrium_norm",
        "mean_tip_radius",
    ],
    "2 features — depth + radius (current best)": [
        "tip_depth_in_atrium_norm",
        "mean_tip_radius",
    ],
}

neg = int((y_clean == 0).sum())
pos = int((y_clean == 1).sum())
spw = neg / pos

rows = []
for label, feats in feature_sets.items():
    avail = [f for f in feats if f in model_df.columns]
    if not avail:
        print(f"  Skipping '{label}' — features not yet in model_df")
        continue
    X = model_df[avail].copy()
    y = y_clean.copy()

    for model_name, pipeline in [
        ("LR", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
            ("m",   LogisticRegression(
                C=1.0, penalty="l2", solver="liblinear",
                class_weight="balanced", max_iter=2000, random_state=42,
            )),
        ])),
        ("RF", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("m",   RandomForestClassifier(
                n_estimators=300, max_depth=4, min_samples_leaf=4,
                class_weight="balanced", random_state=42, n_jobs=-1,
            )),
        ])),
        ("XGB", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("m",   XGBClassifier(
                n_estimators=100, max_depth=2, learning_rate=0.05,
                scale_pos_weight=spw, random_state=42,
                n_jobs=-1, verbosity=0,
            )),
        ])),
    ]:
        res = cross_validate(
            pipeline, X, y, cv=_CV,
            scoring=_SCORING, return_train_score=False,
        )
        rows.append({
            "Feature set":  label,
            "Model":        model_name,
            "N features":   len(avail),
            "Accuracy":     round(res["test_accuracy"].mean(), 4),
            "F1":           round(res["test_f1"].mean(), 4),
            "ROC-AUC":      round(res["test_roc_auc"].mean(), 4),
        })

ablation_df = pd.DataFrame(rows)
print(ablation_df.to_string(index=False))

                               Feature set Model  N features  Accuracy     F1  ROC-AUC
               6 features — full point set    LR           6    0.8000 0.7822   0.8182
               6 features — full point set    RF           6    0.8273 0.8074   0.9147
               6 features — full point set   XGB           6    0.8227 0.8053   0.8922
                  5 features — drop tip_dy    LR           5    0.8000 0.7779   0.8278
                  5 features — drop tip_dy    RF           5    0.8227 0.8001   0.9072
                  5 features — drop tip_dy   XGB           5    0.8000 0.7753   0.8880
         4 features — drop mean_tip_radius    LR           5    0.8182 0.8026   0.8137
         4 features — drop mean_tip_radius    RF           5    0.8273 0.8039   0.8930
         4 features — drop mean_tip_radius   XGB           5    0.8182 0.7942   0.8679
    3 features — depth + distance + radius    LR           3    0.8364 0.8178   0.8262
    3 features — depth + distance + radius 

In [77]:
# ═════════════════════════════════════════════════════════════════════════════
# EFFICIENTNET-B7 TIP-DOT CLASSIFICATION  —  v3 IMPROVED
# Changes vs v2:
#   1. SIGMA increased 20 → 35  (larger dot, easier for model to see)
#   2. Fine-tune head_only  (23M → 2.5M trainable, less overfitting)
#   3. DROPOUT increased 0.5 → 0.6
#   4. NUM_EPOCHS increased 30 → 50, PATIENCE increased 10 → 15
#   5. Per-fold metrics now include Precision and Recall
#   6. Optimal threshold search added after training
#   7. Full metrics table in results cell
# ═════════════════════════════════════════════════════════════════════════════


# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Install and imports  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────

# !pip install timm albumentations --quiet

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    precision_score, recall_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU — Runtime → Change runtime type → GPU")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Configuration
# ─────────────────────────────────────────────────────────────────────────────

IMG_SIZE   = 600
SIGMA      = 35      # FIX: increased 20→35, larger dot easier for model to see
BATCH_SIZE = 8
NUM_EPOCHS = 50      # FIX: increased 30→50, head-only trains slower
PATIENCE   = 15      # FIX: increased 10→15, more patience with head-only
LR         = 1e-4
N_FOLDS    = 5
DROPOUT    = 0.6     # FIX: increased 0.5→0.6, head-only needs more dropout
FINE_TUNE  = "head_only"  # FIX: was "last_block", now head only
SEED       = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_SAVE_DIR = os.path.join(OUTPUT_FOLDER, "efficientnet_tip_dot_v3")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
print(f"Model save dir : {MODEL_SAVE_DIR}")

assert "tip_y" in model_df.columns, "tip_y missing — re-run batch pipeline"
assert "tip_x" in model_df.columns, "tip_x missing — re-run batch pipeline"
n_null = model_df[["tip_y", "tip_x"]].isna().any(axis=1).sum()
print(f"tip_y / tip_x non-null : {len(model_df) - n_null} / {len(model_df)}")
print(f"Null tip coords        : {n_null}")
print(f"\nKey settings:")
print(f"  IMG_SIZE   = {IMG_SIZE}")
print(f"  SIGMA      = {SIGMA}  (was 20)")
print(f"  NUM_EPOCHS = {NUM_EPOCHS}  (was 30)")
print(f"  PATIENCE   = {PATIENCE}  (was 10)")
print(f"  DROPOUT    = {DROPOUT}  (was 0.5)")
print(f"  FINE_TUNE  = {FINE_TUNE}  (was last_block)")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Dot image + Dataset + sanity check  (unchanged structure)
# ─────────────────────────────────────────────────────────────────────────────

def make_tip_dot_image(original_shape, tip_coord, target_size, sigma=35):
    H_orig, W_orig = original_shape
    if tip_coord is None:
        return np.zeros((target_size, target_size), dtype=np.uint8)
    if np.isnan(float(tip_coord[0])) or np.isnan(float(tip_coord[1])):
        return np.zeros((target_size, target_size), dtype=np.uint8)
    ty_scaled    = float(tip_coord[0]) * target_size / H_orig
    tx_scaled    = float(tip_coord[1]) * target_size / W_orig
    sigma_scaled = sigma * target_size / H_orig
    y_grid, x_grid = np.mgrid[0:target_size,
                               0:target_size].astype(np.float32)
    canvas = np.exp(
        -((y_grid - ty_scaled)**2 + (x_grid - tx_scaled)**2)
        / (2.0 * sigma_scaled**2)
    )
    return (canvas * 255.0).astype(np.uint8)


class TipDotDataset(Dataset):
    """
    3-channel input:
        Channel 0 : Gaussian dot at tip coordinate
        Channel 1 : Atrium mask
        Channel 2 : Zeros
    """
    def __init__(self, df, labels, atrium_dir,
                 img_size=600, sigma=35, transform=None):
        self.df         = df.reset_index(drop=True)
        self.labels     = torch.tensor(labels, dtype=torch.float32)
        self.atrium_dir = atrium_dir
        self.img_size   = img_size
        self.sigma      = sigma
        self.transform  = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        H = W = 960

        # Channel 0: Gaussian dot
        dot = make_tip_dot_image(
            (H, W), (row["tip_y"], row["tip_x"]),
            self.img_size, self.sigma
        )

        # Channel 1: Atrium mask
        atrium_raw = cv2.imread(
            os.path.join(self.atrium_dir, row["image_name"]),
            cv2.IMREAD_GRAYSCALE
        )
        if atrium_raw is None:
            for ext in [".tif", ".tiff", ".jpg"]:
                alt = os.path.join(
                    self.atrium_dir,
                    os.path.splitext(row["image_name"])[0] + ext
                )
                atrium_raw = cv2.imread(alt, cv2.IMREAD_GRAYSCALE)
                if atrium_raw is not None:
                    break
        atrium_resized = (
            cv2.resize(
                (atrium_raw > 0).astype(np.uint8) * 255,
                (self.img_size, self.img_size),
                interpolation=cv2.INTER_NEAREST
            )
            if atrium_raw is not None
            else np.zeros((self.img_size, self.img_size), dtype=np.uint8)
        )

        # Channel 2: Zeros
        blank = np.zeros((self.img_size, self.img_size), dtype=np.uint8)

        feature_hwc = np.stack(
            [dot, atrium_resized, blank], axis=-1
        ).astype(np.float32)

        if self.transform:
            tensor = self.transform(image=feature_hwc)["image"]
        else:
            tensor = torch.from_numpy(feature_hwc.transpose(2, 0, 1))

        return tensor, self.labels[idx]


# Sanity check
print("Checking tip dot generation for 3 sample images...")
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, (_, row) in enumerate(model_df.head(3).iterrows()):
    dot = make_tip_dot_image(
        (960, 960), (row["tip_y"], row["tip_x"]), IMG_SIZE, SIGMA
    )
    atrium_raw = cv2.imread(
        os.path.join(ATRIUM_FOLDER, row["image_name"]),
        cv2.IMREAD_GRAYSCALE
    )
    atrium_vis = (
        cv2.resize(
            (atrium_raw > 0).astype(np.uint8) * 255,
            (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST
        ) if atrium_raw is not None
        else np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    )
    axes[0, i].imshow(dot, cmap="hot")
    axes[0, i].set_title(
        f"{row['image_name']}\n"
        f"tip=({row['tip_y']:.0f},{row['tip_x']:.0f})  "
        f"sigma={SIGMA}px\n"
        f"label={int(row['tip (1-ok, 0-no)'])}  "
        f"status={row['status']}",
        fontsize=7
    )
    axes[0, i].axis("off")
    axes[1, i].imshow(atrium_vis, cmap="gray")
    axes[1, i].set_title("Atrium mask", fontsize=8)
    axes[1, i].axis("off")

plt.suptitle(
    f"EfficientNet inputs — Gaussian dot sigma={SIGMA}px (top) "
    f"+ Atrium mask (bottom)",
    fontsize=11
)
plt.tight_layout()
plt.show()
print("✓ Dot images look correct — ready for training")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Model definition and training utilities
# ─────────────────────────────────────────────────────────────────────────────

def build_efficientnet_b7(dropout=0.6, fine_tune="head_only", device=None):
    """
    EfficientNet-B7 with configurable fine-tuning strategy.

    fine_tune options:
        "head_only"  — only conv_head + bn2 + classifier trainable
                       ~2.5M / 63M params (4%)
                       BEST for 176 training samples per fold
        "last_block" — blocks.6 + head trainable
                       ~23M / 63M params (36%)
                       Use if head_only underfits

    FIX vs v2:
        Default changed to head_only — dramatically reduces overfitting
        with 220 samples. The pretrained ImageNet features already encode
        useful representations; we only need to adapt the final mapping.
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = timm.create_model(
        "tf_efficientnet_b7",
        pretrained=True,
        in_chans=3,
        num_classes=1,
    )

    # Freeze all parameters first
    for param in model.parameters():
        param.requires_grad = False

    if fine_tune == "head_only":
        # FIX: only train the final conv + classifier
        # ~2.5M trainable params — much better for 220 samples
        for name, param in model.named_parameters():
            if any(layer in name for layer in [
                "conv_head", "bn2", "classifier"
            ]):
                param.requires_grad = True

    elif fine_tune == "last_block":
        # Original v2 setting — use if head_only underfits
        for name, param in model.named_parameters():
            if any(layer in name for layer in [
                "blocks.6", "conv_head", "bn2", "classifier"
            ]):
                param.requires_grad = True

    # Replace classifier with higher dropout
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 1),
    )

    trainable = sum(p.numel() for p in model.parameters()
                    if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"EfficientNet-B7 [{fine_tune}]: "
          f"{trainable:,} / {total:,} params trainable "
          f"({100*trainable/total:.1f}%)")

    return model.to(device)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs   = imgs.to(device)
        labels = labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        probs = torch.sigmoid(model(imgs.to(device)).squeeze(1))
        all_probs.extend(probs.cpu().numpy().tolist())
        all_labels.extend(labels.numpy().tolist())
    y_true = np.array(all_labels)
    y_prob = np.array(all_probs)
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "auc":       roc_auc_score(y_true, y_prob),
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "probs":     y_prob,
        "preds":     y_pred,
        "labels":    y_true,
    }


train_transform = A.Compose([
    A.HorizontalFlip(p=0.3),
    A.Rotate(limit=15, p=0.3, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2(),
])
val_transform = A.Compose([
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ToTensorV2(),
])
print("Model and transform definitions ready ✓")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — 5-Fold Cross-Validation Training
# ─────────────────────────────────────────────────────────────────────────────

labels_array  = model_df["tip (1-ok, 0-no)"].astype(int).values
skf           = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                random_state=SEED)
fold_results  = []
all_val_probs = np.zeros(len(model_df))
all_val_preds = np.zeros(len(model_df))

neg = int((labels_array == 0).sum())
pos = int((labels_array == 1).sum())
pos_weight = torch.tensor([neg / pos], dtype=torch.float32).to(device)
print(f"Class balance — 0: {neg}, 1: {pos}, "
      f"pos_weight: {neg/pos:.3f}")

print("\n" + "=" * 65)
print("TRAINING EFFICIENTNET-B7 TIP-DOT  (v3 — improved)")
print(f"IMG_SIZE={IMG_SIZE}, SIGMA={SIGMA}, "
      f"MAX_EPOCHS={NUM_EPOCHS}, PATIENCE={PATIENCE}, "
      f"DROPOUT={DROPOUT}, FINE_TUNE={FINE_TUNE}")
print("=" * 65)

for fold, (train_idx, val_idx) in enumerate(
        skf.split(model_df, labels_array), 1):

    print(f"\n── Fold {fold}/{N_FOLDS} "
          f"────────────────────────────────")

    train_df = model_df.iloc[train_idx]
    val_df   = model_df.iloc[val_idx]
    y_train  = labels_array[train_idx]
    y_val    = labels_array[val_idx]

    train_loader = DataLoader(
        TipDotDataset(train_df, y_train, ATRIUM_FOLDER,
                      img_size=IMG_SIZE, sigma=SIGMA,
                      transform=train_transform),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        TipDotDataset(val_df, y_val, ATRIUM_FOLDER,
                      img_size=IMG_SIZE, sigma=SIGMA,
                      transform=val_transform),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True,
    )

    model     = build_efficientnet_b7(
        dropout=DROPOUT, fine_tune=FINE_TUNE, device=device
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=1e-4,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-6,
    )

    best_auc         = 0.0
    best_epoch       = 0
    patience_counter = 0
    best_path        = os.path.join(
        MODEL_SAVE_DIR, f"fold{fold}_best.pt"
    )

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss  = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_metrics = evaluate(model, val_loader, device)
        scheduler.step()

        if val_metrics["auc"] > best_auc:
            best_auc         = val_metrics["auc"]
            best_epoch       = epoch
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_counter += 1

        if epoch % 5 == 0 or epoch == 1:
            print(
                f"  Epoch {epoch:3d}/{NUM_EPOCHS}  "
                f"loss={train_loss:.4f}  "
                f"val_auc={val_metrics['auc']:.4f}  "
                f"val_acc={val_metrics['accuracy']:.4f}  "
                f"patience={patience_counter}/{PATIENCE}  "
                f"(best={best_auc:.4f} @ ep{best_epoch})"
            )

        if patience_counter >= PATIENCE:
            print(f"  → Early stopping at epoch {epoch}")
            break

    # Load best weights
    model.load_state_dict(
        torch.load(best_path, map_location=device)
    )
    final_metrics = evaluate(model, val_loader, device)

    all_val_probs[val_idx] = final_metrics["probs"]
    all_val_preds[val_idx] = final_metrics["preds"]

    fold_results.append({
        "Fold":      fold,
        "AUC":       round(final_metrics["auc"],       4),
        "Accuracy":  round(final_metrics["accuracy"],  4),
        "Precision": round(final_metrics["precision"], 4),
        "Recall":    round(final_metrics["recall"],    4),
        "F1":        round(final_metrics["f1"],        4),
        "Best ep":   best_epoch,
        "Stopped":   epoch,
    })
    print(
        f"  ✓ Fold {fold}  "
        f"AUC={final_metrics['auc']:.4f}  "
        f"Acc={final_metrics['accuracy']:.4f}  "
        f"Prec={final_metrics['precision']:.4f}  "
        f"Rec={final_metrics['recall']:.4f}  "
        f"F1={final_metrics['f1']:.4f}  "
        f"(best ep{best_epoch}, stopped ep{epoch})"
    )

    del model, optimizer, scheduler
    torch.cuda.empty_cache()

print("\n" + "=" * 65)
print("TRAINING COMPLETE")
print("=" * 65)


# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Results, optimal threshold, comparison, error analysis
# ─────────────────────────────────────────────────────────────────────────────

results_df = pd.DataFrame(fold_results)

# ── Per-fold table ────────────────────────────────────────────────
print("=== EfficientNet-B7 Tip-Dot (v3) — Per-fold Results ===")
print(results_df.to_string(index=False))
print(f"\n  Mean  "
      f"AUC={results_df['AUC'].mean():.4f}  "
      f"Acc={results_df['Accuracy'].mean():.4f}  "
      f"Prec={results_df['Precision'].mean():.4f}  "
      f"Rec={results_df['Recall'].mean():.4f}  "
      f"F1={results_df['F1'].mean():.4f}")
print(f"  ±std  "
      f"AUC={results_df['AUC'].std():.4f}  "
      f"Acc={results_df['Accuracy'].std():.4f}  "
      f"Prec={results_df['Precision'].std():.4f}  "
      f"Rec={results_df['Recall'].std():.4f}  "
      f"F1={results_df['F1'].std():.4f}")

# ── OOF metrics at default threshold 0.5 ─────────────────────────
oof_auc  = roc_auc_score(labels_array, all_val_probs)
oof_acc  = accuracy_score(labels_array, all_val_preds)
oof_prec = precision_score(labels_array, all_val_preds, zero_division=0)
oof_rec  = recall_score(labels_array, all_val_preds, zero_division=0)
oof_f1   = f1_score(labels_array, all_val_preds, zero_division=0)

print(f"\n=== Overall OOF (threshold=0.50) ===")
print(f"  AUC       : {oof_auc:.4f}")
print(f"  Accuracy  : {oof_acc:.4f}")
print(f"  Precision : {oof_prec:.4f}")
print(f"  Recall    : {oof_rec:.4f}")
print(f"  F1        : {oof_f1:.4f}")

# ── FIX: Optimal threshold search ────────────────────────────────
# Finds the threshold that maximises F1 on OOF predictions
print(f"\n=== Optimal threshold search ===")
skf_thresh = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=SEED
)
optimal_thresholds = []

for fold, (_, val_idx) in enumerate(
        skf_thresh.split(model_df, labels_array), 1):

    y_true_f = labels_array[val_idx]
    y_prob_f = all_val_probs[val_idx]

    _, _, thresholds = roc_curve(y_true_f, y_prob_f)
    f1s = [
        f1_score(
            y_true_f,
            (y_prob_f >= t).astype(int),
            zero_division=0
        )
        for t in thresholds
    ]
    best_t  = float(thresholds[np.argmax(f1s)])
    best_f1 = float(max(f1s))
    default_f1 = f1_score(
        y_true_f,
        (y_prob_f >= 0.5).astype(int),
        zero_division=0
    )
    optimal_thresholds.append(best_t)
    print(f"  Fold {fold}: "
          f"default t=0.50 → F1={default_f1:.3f}  |  "
          f"optimal t={best_t:.3f} → F1={best_f1:.3f}")

mean_thresh      = float(np.mean(optimal_thresholds))
y_pred_opt       = (all_val_probs >= mean_thresh).astype(int)
oof_acc_opt      = accuracy_score(labels_array, y_pred_opt)
oof_prec_opt     = precision_score(
    labels_array, y_pred_opt, zero_division=0
)
oof_rec_opt      = recall_score(
    labels_array, y_pred_opt, zero_division=0
)
oof_f1_opt       = f1_score(
    labels_array, y_pred_opt, zero_division=0
)

print(f"\n=== Overall OOF (threshold={mean_thresh:.3f} optimal) ===")
print(f"  AUC       : {oof_auc:.4f}  (unchanged)")
print(f"  Accuracy  : {oof_acc_opt:.4f}  "
      f"(was {oof_acc:.4f})")
print(f"  Precision : {oof_prec_opt:.4f}  "
      f"(was {oof_prec:.4f})")
print(f"  Recall    : {oof_rec_opt:.4f}  "
      f"(was {oof_rec:.4f})")
print(f"  F1        : {oof_f1_opt:.4f}  "
      f"(was {oof_f1:.4f})")



Device : cuda
GPU    : NVIDIA B200
VRAM   : 191.5 GB
Model save dir : /vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis/predicted_masks/efficientnet_tip_dot_v3
tip_y / tip_x non-null : 220 / 220
Null tip coords        : 0

Key settings:
  IMG_SIZE   = 600
  SIGMA      = 35  (was 20)
  NUM_EPOCHS = 50  (was 30)
  PATIENCE   = 15  (was 10)
  DROPOUT    = 0.6  (was 0.5)
  FINE_TUNE  = head_only  (was last_block)
Checking tip dot generation for 3 sample images...
✓ Dot images look correct — ready for training
Model and transform definitions ready ✓
Class balance — 0: 119, 1: 101, pos_weight: 1.178

TRAINING EFFICIENTNET-B7 TIP-DOT  (v3 — improved)
IMG_SIZE=600, SIGMA=35, MAX_EPOCHS=50, PATIENCE=15, DROPOUT=0.6, FINE_TUNE=head_only

── Fold 1/5 ────────────────────────────────
EfficientNet-B7 [head_only]: 1,786,497 / 63,789,521 params trainable (2.8%)
  Epoch   1/50  loss=0.7601  val_auc=0.8562  val_acc=0.7727  patience=0/15  (best=0.8562 @ ep1)
  Epoch   5/50  loss=0.6696  val_auc=0.

EfficientNet-B7 [head_only]: 1,786,497 / 63,789,521 params trainable (2.8%)
  Epoch   1/50  loss=0.7411  val_auc=0.7479  val_acc=0.6364  patience=0/15  (best=0.7479 @ ep1)
  Epoch   5/50  loss=0.6325  val_auc=0.9104  val_acc=0.8409  patience=2/15  (best=0.9500 @ ep3)
  Epoch  10/50  loss=0.5536  val_auc=0.9333  val_acc=0.8636  patience=7/15  (best=0.9500 @ ep3)
  Epoch  15/50  loss=0.5193  val_auc=0.9250  val_acc=0.8182  patience=12/15  (best=0.9500 @ ep3)
  → Early stopping at epoch 18
  ✓ Fold 2  AUC=0.9500  Acc=0.8864  Prec=0.8261  Rec=0.9500  F1=0.8837  (best ep3, stopped ep18)

── Fold 3/5 ────────────────────────────────
EfficientNet-B7 [head_only]: 1,786,497 / 63,789,521 params trainable (2.8%)
  Epoch   1/50  loss=0.7429  val_auc=0.5521  val_acc=0.5455  patience=0/15  (best=0.5521 @ ep1)
  Epoch   5/50  loss=0.6357  val_auc=0.8229  val_acc=0.7500  patience=2/15  (best=0.8625 @ ep3)
  Epoch  10/50  loss=0.5041  val_auc=0.8250  val_acc=0.7955  patience=7/15  (best=0.8625 @ ep3)
 

NameError: name 'barche_oof_auc' is not defined

In [78]:
# ── Full comparison table — NO hardcoded values ──────────────────────
print(f"\n=== FULL COMPARISON — ALL APPROACHES ===")

# These variables must already exist in your notebook from earlier cells:
# lr_oof_auc, lr_oof_acc, lr_oof_prec, lr_oof_rec, lr_oof_f1
# rf_oof_auc, rf_oof_acc, rf_oof_prec, rf_oof_rec, rf_oof_f1
# xgb_oof_auc, xgb_oof_acc, xgb_oof_prec, xgb_oof_rec, xgb_oof_f1
# oof_auc, oof_acc, oof_prec, oof_rec, oof_f1  (tip-dot, t=0.50)
# oof_acc_opt, oof_prec_opt, oof_rec_opt, oof_f1_opt  (tip-dot, optimal t)

comparison = pd.DataFrame([
    {"Method": "LR  (2 tip-point)",
     "AUC": round(lr_oof_auc, 3),
     "Acc": round(lr_oof_acc, 3),
     "Prec": round(lr_oof_prec, 3),
     "Rec": round(lr_oof_rec, 3),
     "F1": round(lr_oof_f1, 3)},

    {"Method": "RF  (2 tip-point)",
     "AUC": round(rf_oof_auc, 3),
     "Acc": round(rf_oof_acc, 3),
     "Prec": round(rf_oof_prec, 3),
     "Rec": round(rf_oof_rec, 3),
     "F1": round(rf_oof_f1, 3)},

    {"Method": "XGB (2 tip-point)",
     "AUC": round(xgb_oof_auc, 3),
     "Acc": round(xgb_oof_acc, 3),
     "Prec": round(xgb_oof_prec, 3),
     "Rec": round(xgb_oof_rec, 3),
     "F1": round(xgb_oof_f1, 3)},

    {"Method": f"EffNet-B7 tip-dot v3 (t=0.50)",
     "AUC": round(oof_auc, 3),
     "Acc": round(oof_acc, 3),
     "Prec": round(oof_prec, 3),
     "Rec": round(oof_rec, 3),
     "F1": round(oof_f1, 3)},

    {"Method": f"EffNet-B7 tip-dot v3 (t={mean_thresh:.2f} optimal)",
     "AUC": round(oof_auc, 3),
     "Acc": round(oof_acc_opt, 3),
     "Prec": round(oof_prec_opt, 3),
     "Rec": round(oof_rec_opt, 3),
     "F1": round(oof_f1_opt, 3)},
])

print(comparison.to_string(index=False))

# ── Confusion matrix (optimal threshold) ─────────────────────────
cm  = confusion_matrix(labels_array, y_pred_opt)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Malpositioned (0)", "Correct (1)"]
).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(
    f"EfficientNet-B7 Tip-Dot (v3)\n"
    f"OOF  AUC={oof_auc:.3f}  "
    f"Acc={oof_acc_opt:.3f}  "
    f"F1={oof_f1_opt:.3f}  "
    f"(threshold={mean_thresh:.2f})",
    fontsize=10
)
plt.tight_layout()
plt.savefig("efficientnet_tipdot_v3_confusion.png", dpi=150)
plt.show()
print("Saved: efficientnet_tipdot_v3_confusion.png")

# ── Save OOF predictions ──────────────────────────────────────────
oof_df = model_df[["image_name", "status",
                    "tip (1-ok, 0-no)"]].copy().reset_index(drop=True)
oof_df["eff_prob"]      = all_val_probs.round(4)
oof_df["eff_pred_050"]  = all_val_preds.astype(int)
oof_df["eff_pred_opt"]  = y_pred_opt.astype(int)
oof_df["opt_threshold"] = round(mean_thresh, 3)

oof_save_path = os.path.join(MODEL_SAVE_DIR, "oof_predictions_v3.csv")
oof_df.to_csv(oof_save_path, index=False)
print(f"Saved OOF predictions: {oof_save_path}")

# ── Classification report (optimal threshold) ────────────────────
print("\n=== Classification Report (optimal threshold) ===")
print(classification_report(
    labels_array, y_pred_opt,
    target_names=["Malpositioned (0)", "Correct (1)"]
))

# ── Error analysis by status ──────────────────────────────────────
print("=== Error analysis by status ===")
oof_df["status"] = model_df["status"].values
for s in ["success", "success_overlap_mode", "ambiguous_tip"]:
    sub = oof_df[oof_df["status"] == s]
    if len(sub) == 0:
        continue
    sub_auc = (
        roc_auc_score(
            sub["tip (1-ok, 0-no)"], sub["eff_prob"]
        )
        if len(sub["tip (1-ok, 0-no)"].unique()) > 1
        else float("nan")
    )
    sub_acc = accuracy_score(
        sub["tip (1-ok, 0-no)"], sub["eff_pred_opt"]
    )
    sub_f1 = f1_score(
        sub["tip (1-ok, 0-no)"], sub["eff_pred_opt"],
        zero_division=0
    )
    print(f"  {s:<25}  n={len(sub):3d}  "
          f"Acc={sub_acc:.3f}  "
          f"AUC={sub_auc:.3f}  "
          f"F1={sub_f1:.3f}")


=== FULL COMPARISON — ALL APPROACHES ===
                               Method   AUC   Acc  Prec   Rec    F1
                    LR  (2 tip-point) 0.833 0.814 0.806 0.782 0.794
                    RF  (2 tip-point) 0.900 0.795 0.769 0.792 0.780
                    XGB (2 tip-point) 0.881 0.823 0.830 0.772 0.800
        EffNet-B7 tip-dot v3 (t=0.50) 0.892 0.855 0.848 0.832 0.840
EffNet-B7 tip-dot v3 (t=0.58 optimal) 0.892 0.818 0.907 0.673 0.773
Saved: efficientnet_tipdot_v3_confusion.png
Saved OOF predictions: /vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis/predicted_masks/efficientnet_tip_dot_v3/oof_predictions_v3.csv

=== Classification Report (optimal threshold) ===
                   precision    recall  f1-score   support

Malpositioned (0)       0.77      0.94      0.85       119
      Correct (1)       0.91      0.67      0.77       101

         accuracy                           0.82       220
        macro avg       0.84      0.81      0.81       220
     weighted av

In [79]:
# ── Complete metrics breakdown for EfficientNet-B7 tip-dot ───────
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, confusion_matrix
)
import numpy as np

print("=" * 70)
print("EFFICIENTNET-B7 TIP-DOT — COMPLETE METRICS PER FOLD")
print("=" * 70)
print(f"\n{'Fold':<6} {'AUC':>7} {'Acc':>7} {'Prec':>7} "
      f"{'Recall':>8} {'F1':>7} {'Specificity':>12}")
print("  " + "─" * 62)

fold_metrics_full = []
skf_eval = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for fold, (_, val_idx) in enumerate(
        skf_eval.split(model_df, labels_array), 1):

    y_true_fold = labels_array[val_idx]
    y_prob_fold = all_val_probs[val_idx]
    y_pred_fold = (y_prob_fold >= 0.5).astype(int)

    auc  = roc_auc_score(y_true_fold, y_prob_fold)
    acc  = accuracy_score(y_true_fold, y_pred_fold)
    prec = precision_score(y_true_fold, y_pred_fold, zero_division=0)
    rec  = recall_score(y_true_fold, y_pred_fold, zero_division=0)
    f1   = f1_score(y_true_fold, y_pred_fold, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true_fold, y_pred_fold).ravel()
    spec = tn / (tn + fp)

    fold_metrics_full.append({
        "Fold": fold, "AUC": auc, "Accuracy": acc,
        "Precision": prec, "Recall": rec, "F1": f1,
        "Specificity": spec
    })

    print(f"  {fold:<4} {auc:>7.4f} {acc:>7.4f} {prec:>7.4f} "
          f"{rec:>8.4f} {f1:>7.4f} {spec:>12.4f}")

# ── Summary rows ──────────────────────────────────────────────
fm = pd.DataFrame(fold_metrics_full)
print("  " + "─" * 62)
print(f"  {'Mean':<4} {fm['AUC'].mean():>7.4f} "
      f"{fm['Accuracy'].mean():>7.4f} "
      f"{fm['Precision'].mean():>7.4f} "
      f"{fm['Recall'].mean():>8.4f} "
      f"{fm['F1'].mean():>7.4f} "
      f"{fm['Specificity'].mean():>12.4f}")
print(f"  {'±std':<4} {fm['AUC'].std():>7.4f} "
      f"{fm['Accuracy'].std():>7.4f} "
      f"{fm['Precision'].std():>7.4f} "
      f"{fm['Recall'].std():>8.4f} "
      f"{fm['F1'].std():>7.4f} "
      f"{fm['Specificity'].std():>12.4f}")

# ── SE for sensitivity and specificity ────────────────────────
sens_se = fm['Recall'].std() / np.sqrt(5)
spec_se = fm['Specificity'].std() / np.sqrt(5)
print(f"\n  Sensitivity SE : {sens_se:.4f}")
print(f"  Specificity SE : {spec_se:.4f}")

# ── Overall OOF metrics ───────────────────────────────────────
print("\n" + "=" * 70)
print("OVERALL OUT-OF-FOLD METRICS (all 220 samples combined)")
print("=" * 70)
y_pred_all = (all_val_probs >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(labels_array, y_pred_all).ravel()

oof_auc_val  = roc_auc_score(labels_array, all_val_probs)
oof_acc_val  = accuracy_score(labels_array, y_pred_all)
oof_prec_val = precision_score(labels_array, y_pred_all)
oof_rec_val  = recall_score(labels_array, y_pred_all)
oof_f1_val   = f1_score(labels_array, y_pred_all)
oof_spec_val = tn / (tn + fp)

print(f"  AUC         : {oof_auc_val:.4f}")
print(f"  Accuracy    : {oof_acc_val:.4f}")
print(f"  Precision   : {oof_prec_val:.4f}")
print(f"  Sensitivity : {oof_rec_val:.4f}  ±SE {sens_se:.4f}")
print(f"  Specificity : {oof_spec_val:.4f}  ±SE {spec_se:.4f}")
print(f"  F1          : {oof_f1_val:.4f}")

# ── Full comparison table — nothing hardcoded ─────────────────
print("\n" + "=" * 70)
print("FULL COMPARISON — ALL APPROACHES")
print("=" * 70)
full_comparison = pd.DataFrame([
    {"Method": "LR  (2 tip-point)",
     "AUC": round(lr_oof_auc, 3),
     "Acc": round(lr_oof_acc, 3),
     "Prec": round(lr_oof_prec, 3),
     "Sens": round(lr_oof_rec, 3),
     "Spec": round(lr_oof_spec, 3),
     "F1":   round(lr_oof_f1, 3)},

    {"Method": "RF  (2 tip-point)",
     "AUC": round(rf_oof_auc, 3),
     "Acc": round(rf_oof_acc, 3),
     "Prec": round(rf_oof_prec, 3),
     "Sens": round(rf_oof_rec, 3),
     "Spec": round(rf_oof_spec, 3),
     "F1":   round(rf_oof_f1, 3)},

    {"Method": "XGB (2 tip-point)",
     "AUC": round(xgb_oof_auc, 3),
     "Acc": round(xgb_oof_acc, 3),
     "Prec": round(xgb_oof_prec, 3),
     "Sens": round(xgb_oof_rec, 3),
     "Spec": round(xgb_oof_spec, 3),
     "F1":   round(xgb_oof_f1, 3)},

    {"Method": "EfficientNet tip-dot",
     "AUC":  round(oof_auc_val, 3),
     "Acc":  round(oof_acc_val, 3),
     "Prec": round(oof_prec_val, 3),
     "Sens": round(oof_rec_val, 3),
     "Spec": round(oof_spec_val, 3),
     "F1":   round(oof_f1_val, 3)},
])
print(full_comparison.to_string(index=False))

EFFICIENTNET-B7 TIP-DOT — COMPLETE METRICS PER FOLD

Fold       AUC     Acc    Prec   Recall      F1  Specificity
  ──────────────────────────────────────────────────────────────
  1     0.9833  0.9545  0.9500   0.9500  0.9500       0.9583
  2     0.9500  0.8864  0.8261   0.9500  0.8837       0.8333
  3     0.8625  0.8182  0.8750   0.7000  0.7778       0.9167
  4     0.8688  0.7955  0.7895   0.7500  0.7692       0.8333
  5     0.8923  0.8182  0.8095   0.8095  0.8095       0.8261
  ──────────────────────────────────────────────────────────────
  Mean  0.9114  0.8545  0.8500   0.8319  0.8381       0.8736
  ±std  0.0530  0.0655  0.0642   0.1146  0.0771       0.0603

  Sensitivity SE : 0.0512
  Specificity SE : 0.0270

OVERALL OUT-OF-FOLD METRICS (all 220 samples combined)
  AUC         : 0.8916
  Accuracy    : 0.8545
  Precision   : 0.8485
  Sensitivity : 0.8317  ±SE 0.0512
  Specificity : 0.8739  ±SE 0.0270
  F1          : 0.8400

FULL COMPARISON — ALL APPROACHES
              Method   A

In [66]:
from sklearn.metrics import roc_curve

print("=== Optimal threshold per fold ===")
skf_thresh = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for fold, (_, val_idx) in enumerate(
        skf_thresh.split(model_df, labels_array), 1):

    y_true_fold = labels_array[val_idx]
    y_prob_fold = all_val_probs[val_idx]

    # Find threshold that maximises F1
    fpr, tpr, thresholds = roc_curve(y_true_fold, y_prob_fold)
    f1_scores = []
    for t in thresholds:
        pred = (y_prob_fold >= t).astype(int)
        f1_scores.append(f1_score(y_true_fold, pred, zero_division=0))

    best_t   = thresholds[np.argmax(f1_scores)]
    best_f1  = max(f1_scores)
    pred_opt = (y_prob_fold >= best_t).astype(int)

    print(f"  Fold {fold}: "
          f"default threshold=0.50 → F1={f1_score(y_true_fold,(y_prob_fold>=0.5).astype(int)):.3f}  |  "
          f"optimal threshold={best_t:.3f} → F1={best_f1:.3f}  "
          f"Acc={accuracy_score(y_true_fold, pred_opt):.3f}")

# Apply optimal threshold globally
fpr, tpr, thresholds = roc_curve(labels_array, all_val_probs)
f1_scores = [f1_score(labels_array, (all_val_probs>=t).astype(int),
             zero_division=0) for t in thresholds]
best_t_global = thresholds[np.argmax(f1_scores)]
pred_opt_global = (all_val_probs >= best_t_global).astype(int)

print(f"\n=== With optimal threshold ({best_t_global:.3f}) ===")
print(f"  AUC       : {roc_auc_score(labels_array, all_val_probs):.4f}")
print(f"  Accuracy  : {accuracy_score(labels_array, pred_opt_global):.4f}")
print(f"  Precision : {precision_score(labels_array, pred_opt_global):.4f}")
print(f"  Recall    : {recall_score(labels_array, pred_opt_global):.4f}")
print(f"  F1        : {f1_score(labels_array, pred_opt_global):.4f}")

=== Optimal threshold per fold ===
  Fold 1: default threshold=0.50 → F1=0.976  |  optimal threshold=0.553 → F1=0.976  Acc=0.977
  Fold 2: default threshold=0.50 → F1=0.878  |  optimal threshold=0.581 → F1=0.900  Acc=0.909
  Fold 3: default threshold=0.50 → F1=0.718  |  optimal threshold=0.804 → F1=0.800  Acc=0.841
  Fold 4: default threshold=0.50 → F1=0.737  |  optimal threshold=0.145 → F1=0.800  Acc=0.795
  Fold 5: default threshold=0.50 → F1=0.800  |  optimal threshold=0.680 → F1=0.821  Acc=0.841

=== With optimal threshold (0.516) ===
  AUC       : 0.8666
  Accuracy  : 0.8455
  Precision : 0.8454
  Recall    : 0.8119
  F1        : 0.8283


In [81]:
# Reload OOF predictions instead of retraining
import pandas as pd
import numpy as np

oof_df = pd.read_csv(
    "/vsc-hard-mounts/leuven-data/375/vsc37504/Master Thesis/"
    "predicted_masks/efficientnet_tip_dot_v3/"
    "oof_predictions_v3.csv"
)

all_val_probs  = oof_df["eff_prob"].values
labels_array   = oof_df["tip (1-ok, 0-no)"].values

print(f"Loaded {len(oof_df)} OOF predictions")
print(f"OOF AUC = {roc_auc_score(labels_array, all_val_probs):.4f}")
# Should print 0.8851 — confirming you have the right run

Loaded 220 OOF predictions
OOF AUC = 0.8915


**Feature Removal Experiment**